In [ ]:
"""import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TORCH_CUDA_ARCH_LIST'] = '9.0'

import torch
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False

print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.version.cuda}")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
"""

'import os\nos.environ[\'CUDA_LAUNCH_BLOCKING\'] = \'1\'\nos.environ[\'TORCH_CUDA_ARCH_LIST\'] = \'9.0\'\n\nimport torch\ntorch.backends.cudnn.benchmark = False\ntorch.backends.cuda.matmul.allow_tf32 = False\n\nprint(f"✅ PyTorch: {torch.__version__}")\nprint(f"✅ CUDA: {torch.version.cuda}")\nprint(f"✅ GPU: {torch.cuda.get_device_name(0)}")\n'

In [ ]:
!df -h /content
!du -h -d 2 /content | sort -h | tail -n 20

Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   87G  149G  37% /
4.0K	/content/drive/.shortcut-targets-by-id
8.0K	/content/.config/configurations
12K	/content/drive/.Trash-0
92K	/content/.config/logs
148K	/content/.config
1.2M	/content/drive/.Encrypted
55M	/content/sample_data
11G	/content/drive
11G	/content/drive/MyDrive
38G	/content/data
38G	/content/data/BraTS
48G	/content


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

# اول PyTorch 2.6 با CUDA 12.6 و ماژول‌های وابسته
!python -m pip install --upgrade --extra-index-url https://download.pytorch.org/whl/cu126 \
  torch==2.6.0+cu126 torchvision==0.21.0+cu126 torchaudio==2.6.0+cu126

# بعد MONAI پایدار و بقیه ابزارها
#!python -m pip install monai==1.5.0 nibabel pydicom tqdm scikit-image
!python -m pip install monai nibabel pydicom tqdm scikit-image

!pip install --upgrade monai
"""

# پاک‌سازی کامل
!pip uninstall torch torchvision torchaudio -y
!pip cache purge

# نصب PyTorch 2.5.1 (پایدار برای Blackwell)
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# نصب MONAI و ابزارها
!pip install monai==1.5.0 nibabel pydicom tqdm scikit-image

print("✅ نصب تکمیل شد. حالا Runtime را restart کن:")
print("Runtime → Restart runtime")

!pip uninstall torch torchvision torchaudio -y
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu126

print("✅ PyTorch Nightly نصب شد. Runtime را restart کن:")
print("Runtime → Restart runtime")
"""

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu126


'\n\n# پاک\u200cسازی کامل\n!pip uninstall torch torchvision torchaudio -y\n!pip cache purge\n\n# نصب PyTorch 2.5.1 (پایدار برای Blackwell)\n!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121\n\n# نصب MONAI و ابزارها\n!pip install monai==1.5.0 nibabel pydicom tqdm scikit-image\n\nprint("✅ نصب تکمیل شد. حالا Runtime را restart کن:")\nprint("Runtime → Restart runtime")\n\n!pip uninstall torch torchvision torchaudio -y\n!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu126\n\nprint("✅ PyTorch Nightly نصب شد. Runtime را restart کن:")\nprint("Runtime → Restart runtime")\n'

In [ ]:
!apt -y install unrar
!pip -q install rarfile
!pip install importnb

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [ ]:
'''
# not bad befor being confused
from dataclasses import dataclass
from typing import Tuple, List, Optional
import os
import torch
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass, field

@dataclass
class CFG:
    # =========================================================
    # 1. تنظیمات اصلی
    # =========================================================
    root_path: str = "/content/drive/MyDrive/Colab Notebooks/testi"
    save_dir: str  = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead_v2"
    seed: int = 42

    do_viz: bool = False
    do_plot: bool = True
    log_step_freq: int = 10

    deep_supervision: bool = True
    ds_weights: List[float] = field(default_factory=lambda: [0.5, 0.25])

    # =========================================================
    # 2. مسیرها و دیتالودر (بهینه‌سازی برای سرعت بالا)
    # =========================================================
    dataset_path: str = "/content/data/BraTS"

    cache_backend: str = "ram"
    # با ۸۰ گیگ VRAM و رم بالای سیستم در A100، بخشی از دیتا را کش می‌کنیم
    cache_rate: float = 0.5
    cache_dir: str = "/content/cache_pre"

    kfold: int = 5
    fold: int = 0

    # ✅ تغییر استراتژی: افزایش Batch Size واقعی به جای Accumulation بیش از حد
    # در A100، بچ‌سایز 4 برای رزولوشن 160 به راحتی جا می‌شود و سرعت را 2 برابر می‌کند.
    batch_size: int = 2
    num_workers: int = 8          # افزایش برای تغذیه سریع GPU
    accum_iter: int = 3          # بچ‌سایز موثر همچنان 16 (4 * 4) باقی می‌ماند

    pin_memory: bool = True
    prefetch_factor: int = 2      # افزایش برای لود پیش‌دستانه دیتا

    # =========================================================
    # 3. تنظیمات سایز و نمونه‌برداری
    # =========================================================
    patch_size: Tuple[int,int,int] = (160, 160, 160)
    roi_size: Tuple[int,int,int]   = (160, 160, 160)

    num_samples: int = 3          # برداشت 2 پچ از هر حجم؛ تنوع داده را در هر بچ بالا می‌برد
    min_frac: float = 0.3
    pos_ratio: float = 0.7
    pos_dilate_kernel: int = 3

    # =========================================================
    # 4. معماری مدل
    # =========================================================
    num_classes: int = 4
    base_ch: int = 64
    num_heads: int = 8
    deep_supervision: bool = True

    # دریفت دراپ‌اوت برای جلوگیری از اوورفیت روی 1250 تصویر
    dropout: float = 0.15         # کمی افزایش برای دیتای بزرگ
    attn_dropout: float = 0.10
    proj_dropout: float = 0.10
    ffn_dropout: float = 0.15
    dec_dropout: float = 0.10

    #ds_weights: Optional[List[float]] = None

    # =========================================================
    # 5. تنظیمات آموزش (Advanced AdamW)
    # =========================================================
    epochs: int = 100
    warmup_epochs: int = 10
    lr: float = 1e-3              # ریت 1e-3 گاهی باعث انفجار گرادیان می‌شود؛ 8e-4 با بچ 16 ایمن‌تر است
    weight_decay: float = 3e-5    # مقدار استاندارد SOTA
    clip_grad_norm: float = 1.0

    use_amp: bool = True
    amp: bool = True

    lambda_dice: float = 1.0
    lambda_ce: float = 1.0
    ds_weight_start: float = 0.5

    # =========================================================
    # 6. تنظیمات پیشرفته و اینفرنس (SOTA Precision)
    # =========================================================
    ema_alpha: float = 0.995      # افزایش پایداری Teacher در دیتای زیاد
    kd_temp: float = 3.0
    kd_max_weight: float = 0.5

    sw_batch_size: int = 2        # ولیدیشن سریع‌تر روی A100
    sw_roi_size: Tuple[int,int,int] = (160, 160, 160)
    sw_overlap: float = 0.5      # ✅ افزایش به 0.7 برای دقت حداکثری در مرزهای تومور
    hd_every: int = 5

    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    channels_last: bool = True

# =========================================================
# مقداردهی نهایی
# =========================================================
cfg = CFG()
#if cfg.deep_supervision and (cfg.ds_weights is None):
#    cfg.ds_weights = [0.50, 0.25]

os.makedirs(cfg.save_dir, exist_ok=True)
print(f"🚀 Gemini SOTA Config Loaded for A100!") و هر ایپاک داره 20 دقه برای کا دیتاست 1250 تصویری طول میکشه. ایا این نرماله یا نیازه کار بکنم؟

'''

'\n# not bad befor being confused\nfrom dataclasses import dataclass\nfrom typing import Tuple, List, Optional\nimport os\nimport torch\nimport torch.nn.functional as F\nimport numpy as np\nfrom dataclasses import dataclass, field\n\n@dataclass\nclass CFG:\n\xa0 \xa0 # =========================================================\n\xa0 \xa0 # 1. تنظیمات اصلی\n\xa0 \xa0 # =========================================================\n\xa0 \xa0 root_path: str = "/content/drive/MyDrive/Colab Notebooks/testi"\n\xa0 \xa0 save_dir: str \xa0= "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead_v2"\n\xa0 \xa0 seed: int = 42\n\n\xa0 \xa0 do_viz: bool = False\n\xa0 \xa0 do_plot: bool = True\n\xa0 \xa0 log_step_freq: int = 10\n\n\xa0 \xa0 deep_supervision: bool = True\n\xa0 \xa0 ds_weights: List[float] = field(default_factory=lambda: [0.5, 0.25])\n\n\xa0 \xa0 # =========================================================\n\xa0 \xa0 # 2. مسیرها و دیتالودر (بهینه\u200cسازی برای س

In [ ]:
# A100 80G
'''
# Gemini Ultra-SOTA Hyperparameters for BraTS2021
from dataclasses import dataclass
from typing import Tuple, List, Optional
import os
import torch
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass, field

@dataclass
class CFG:
    # =========================================================
    # 1. تنظیمات اصلی
    # =========================================================
    root_path: str = "/content/drive/MyDrive/Colab Notebooks/testi"
    save_dir: str  = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead_v2"
    seed: int = 42

    do_viz: bool = False
    do_plot: bool = True
    log_step_freq: int = 10

    deep_supervision: bool = True
    ds_weights: List[float] = field(default_factory=lambda: [0.5, 0.25])

    # =========================================================
    # 2. مسیرها و دیتالودر (بهینه‌سازی برای سرعت بالا)
    # =========================================================
    dataset_path: str = "/content/data/BraTS"

    cache_backend: str = "ram"
    # با ۸۰ گیگ VRAM و رم بالای سیستم در A100، بخشی از دیتا را کش می‌کنیم
    cache_rate: float = 0.8
    #cache_rate: float = 0.8

    cache_dir: str = "/content/cache_pre"

    kfold: int = 5
    fold: int = 0

    # ✅ تغییر استراتژی: افزایش Batch Size واقعی به جای Accumulation بیش از حد
    # در A100، بچ‌سایز 4 برای رزولوشن 160 به راحتی جا می‌شود و سرعت را 2 برابر می‌کند.
    batch_size: int = 2
    num_workers: int = 10          # افزایش برای تغذیه سریع GPU
    accum_iter: int = 4          # بچ‌سایز موثر همچنان 16 (4 * 4) باقی می‌ماند

    pin_memory: bool = True
    prefetch_factor: int = 2      # افزایش برای لود پیش‌دستانه دیتا

    val_sample_limit: int = 100   # اضافه کنید: فقط ۴۰ مورد از ۱۲۵۰ تا برای ولیدیشن سریع
    # =========================================================
    # 3. تنظیمات سایز و نمونه‌برداری
    # =========================================================
    patch_size: Tuple[int,int,int] = (160, 160, 160)
    roi_size: Tuple[int,int,int]   = (160, 160, 160)

    num_samples: int = 3          # برداشت 2 پچ از هر حجم؛ تنوع داده را در هر بچ بالا می‌برد
    min_frac: float = 0.3
    pos_ratio: float = 0.7
    pos_dilate_kernel: int = 3

    # =========================================================
    # 4. معماری مدل
    # =========================================================
    num_classes: int = 4
    base_ch: int = 64
    num_heads: int = 8
    deep_supervision: bool = True

    # دریفت دراپ‌اوت برای جلوگیری از اوورفیت روی 1250 تصویر
    dropout: float = 0.15         # کمی افزایش برای دیتای بزرگ
    attn_dropout: float = 0.10
    proj_dropout: float = 0.10
    ffn_dropout: float = 0.15
    dec_dropout: float = 0.10

    #ds_weights: Optional[List[float]] = None

    # =========================================================
    # 5. تنظیمات آموزش (Advanced AdamW)
    # =========================================================
    epochs: int = 150
    warmup_epochs: int = 10
    lr: float = 2e-5
    #lr: float = 1e-3              # ریت 1e-3 گاهی باعث انفجار گرادیان می‌شود؛ 8e-4 با بچ 16 ایمن‌تر است
    weight_decay: float = 3e-5    # مقدار استاندارد SOTA
    clip_grad_norm: float = 1.0

    use_amp: bool =
    amp: bool = True

    lambda_dice: float = 1.0
    lambda_ce: float = 1.0
    ds_weight_start: float = 0.5

    # =========================================================
    # 6. تنظیمات پیشرفته و اینفرنس (SOTA Precision)
    # =========================================================
    ema_alpha: float = 0.995      # افزایش پایداری Teacher در دیتای زیاد
    kd_temp: float = 3.0
    kd_max_weight: float = 0.5

    sw_batch_size: int = 2        # ولیدیشن سریع‌تر روی A100
    sw_roi_size: Tuple[int,int,int] = (160, 160, 160)
    sw_overlap: float = 0.7      # ✅ افزایش به 0.7 برای دقت حداکثری در مرزهای تومور
    hd_every: int = 3

    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    channels_last: bool = True

# =========================================================
# مقداردهی نهایی
# =========================================================
cfg = CFG()
#if cfg.deep_supervision and (cfg.ds_weights is None):
#    cfg.ds_weights = [0.50, 0.25]

os.makedirs(cfg.save_dir, exist_ok=True)
print(f"🚀 Gemini SOTA Config Loaded for A100!")
'''

'\n# Gemini Ultra-SOTA Hyperparameters for BraTS2021\nfrom dataclasses import dataclass\nfrom typing import Tuple, List, Optional\nimport os\nimport torch\nimport torch.nn.functional as F\nimport numpy as np\nfrom dataclasses import dataclass, field\n\n@dataclass\nclass CFG:\n    # =========================================================\n    # 1. تنظیمات اصلی\n    # =========================================================\n    root_path: str = "/content/drive/MyDrive/Colab Notebooks/testi"\n    save_dir: str  = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead_v2"\n    seed: int = 42\n\n    do_viz: bool = False\n    do_plot: bool = True\n    log_step_freq: int = 10\n\n    deep_supervision: bool = True\n    ds_weights: List[float] = field(default_factory=lambda: [0.5, 0.25])\n\n    # =========================================================\n    # 2. مسیرها و دیتالودر (بهینه\u200cسازی برای سرعت بالا)\n    # =============================================

In [ ]:
"""# befor adding channels and loss

# Gemini Ultra-SOTA Hyperparameters for BraTS2021
from dataclasses import dataclass
from typing import Tuple, List, Optional
import os
import torch
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass, field

@dataclass
class CFG:
    # =========================================================
    # 1. تنظیمات اصلی
    # =========================================================
    root_path: str = "/content/drive/MyDrive/Colab Notebooks/testi"
    save_dir: str  = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead_v2"
    seed: int = 42

    do_viz: bool = False
    do_plot: bool = True
    log_step_freq: int = 10

    deep_supervision: bool = True
    ds_weights: List[float] = field(default_factory=lambda: [0.5, 0.25])

    # =========================================================
    # 2. مسیرها و دیتالودر (بهینه‌سازی برای سرعت بالا)
    # =========================================================
    dataset_path: str = "/content/data/BraTS"

    cache_backend: str = "ram"
    # با ۸۰ گیگ VRAM و رم بالای سیستم در A100، بخشی از دیتا را کش می‌کنیم
    cache_rate: float = 0.8
    #cache_rate: float = 0.8

    cache_dir: str = "/content/cache_pre"

    kfold: int = 5
    fold: int = 0

    # ✅ تغییر استراتژی: افزایش Batch Size واقعی به جای Accumulation بیش از حد
    # در A100، بچ‌سایز 4 برای رزولوشن 160 به راحتی جا می‌شود و سرعت را 2 برابر می‌کند.
    batch_size: int = 2
    num_workers: int = 10          # افزایش برای تغذیه سریع GPU
    accum_iter: int = 4          # بچ‌سایز موثر همچنان 16 (4 * 4) باقی می‌ماند

    pin_memory: bool = True
    prefetch_factor: int = 2      # افزایش برای لود پیش‌دستانه دیتا

    val_sample_limit: int = 100   # اضافه کنید: فقط ۴۰ مورد از ۱۲۵۰ تا برای ولیدیشن سریع
    # =========================================================
    # 3. تنظیمات سایز و نمونه‌برداری
    # =========================================================
    patch_size: Tuple[int,int,int] = (160, 160, 160)
    roi_size: Tuple[int,int,int]   = (160, 160, 160)

    num_samples: int = 2          # برداشت 2 پچ از هر حجم؛ تنوع داده را در هر بچ بالا می‌برد
    min_frac: float = 0.3
    pos_ratio: float = 0.7
    pos_dilate_kernel: int = 3

    # =========================================================
    # 4. معماری مدل
    # =========================================================
    num_classes: int = 4
    base_ch: int = 64
    num_heads: int = 8
    deep_supervision: bool = True

    # دریفت دراپ‌اوت برای جلوگیری از اوورفیت روی 1250 تصویر
    dropout: float = 0.20         # کمی افزایش برای دیتای بزرگ
    attn_dropout: float = 0.15
    proj_dropout: float = 0.15
    ffn_dropout: float = 0.20
    dec_dropout: float = 0.25

    #ds_weights: Optional[List[float]] = None

    # =========================================================
    # 5. تنظیمات آموزش (Advanced AdamW)
    # =========================================================
    epochs: int = 150
    warmup_epochs: int = 10
    lr: float = 1e-5
    #lr: float = 1e-3              # ریت 1e-3 گاهی باعث انفجار گرادیان می‌شود؛ 8e-4 با بچ 16 ایمن‌تر است
    weight_decay: float = 3e-5    # مقدار استاندارد SOTA
    clip_grad_norm: float = 1.0

    use_amp: bool = True

    lambda_dice: float = 1.0
    lambda_ce: float = 1.0
    ds_weight_start: float = 0.5

    # =========================================================
    # 6. تنظیمات پیشرفته و اینفرنس (SOTA Precision)
    # =========================================================
    ema_alpha: float = 0.995      # افزایش پایداری Teacher در دیتای زیاد
    kd_temp: float = 3.0
    kd_max_weight: float = 0.5

    sw_batch_size: int = 2        # ولیدیشن سریع‌تر روی A100
    sw_roi_size: Tuple[int,int,int] = (160, 160, 160)
    sw_overlap: float = 0.7      # ✅ افزایش به 0.7 برای دقت حداکثری در مرزهای تومور
    hd_every: int = 3

    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    channels_last: bool = True

# =========================================================
# مقداردهی نهایی
# =========================================================
cfg = CFG()
#if cfg.deep_supervision and (cfg.ds_weights is None):
#    cfg.ds_weights = [0.50, 0.25]

os.makedirs(cfg.save_dir, exist_ok=True)
print(f"🚀 Gemini SOTA Config Loaded for A100!")
"""

'# befor adding channels and loss\n\n# Gemini Ultra-SOTA Hyperparameters for BraTS2021\nfrom dataclasses import dataclass\nfrom typing import Tuple, List, Optional\nimport os\nimport torch\nimport torch.nn.functional as F\nimport numpy as np\nfrom dataclasses import dataclass, field\n\n@dataclass\nclass CFG:\n    # =========================================================\n    # 1. تنظیمات اصلی\n    # =========================================================\n    root_path: str = "/content/drive/MyDrive/Colab Notebooks/testi"\n    save_dir: str  = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead_v2"\n    seed: int = 42\n\n    do_viz: bool = False\n    do_plot: bool = True\n    log_step_freq: int = 10\n\n    deep_supervision: bool = True\n    ds_weights: List[float] = field(default_factory=lambda: [0.5, 0.25])\n\n    # =========================================================\n    # 2. مسیرها و دیتالودر (بهینه\u200cسازی برای سرعت بالا)\n    # ===========

In [ ]:

# Gemini Ultra-SOTA Hyperparameters for BraTS2021
from dataclasses import dataclass
from typing import Tuple, List, Optional
import os
import torch
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass, field

@dataclass
class CFG:
    # =========================================================
    # 1. تنظیمات اصلی
    # =========================================================
    root_path: str = "/content/drive/MyDrive/Colab Notebooks/testi"
    save_dir: str  = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead_v2"
    seed: int = 42

    do_viz: bool = False
    do_plot: bool = True
    log_step_freq: int = 10

    deep_supervision: bool = True
    ds_weights: List[float] = field(default_factory=lambda: [0.5, 0.25])

    # =========================================================
    # 2. مسیرها و دیتالودر (بهینه‌سازی برای سرعت بالا)
    # =========================================================
    dataset_path: str = "/content/data/BraTS"

    cache_backend: str = "ram"
    # با ۸۰ گیگ VRAM و رم بالای سیستم در A100، بخشی از دیتا را کش می‌کنیم
    cache_rate: float = 0.8
    #cache_rate: float = 0.8

    cache_dir: str = "/content/cache_pre"

    kfold: int = 5
    fold: int = 0

    # ✅ تغییر استراتژی: افزایش Batch Size واقعی به جای Accumulation بیش از حد
    # در A100، بچ‌سایز 4 برای رزولوشن 160 به راحتی جا می‌شود و سرعت را 2 برابر می‌کند.
    batch_size: int = 1
    num_workers: int = 10          # افزایش برای تغذیه سریع GPU
    accum_iter: int = 4          # بچ‌سایز موثر همچنان 16 (4 * 4) باقی می‌ماند

    pin_memory: bool = True
    prefetch_factor: int = 2      # افزایش برای لود پیش‌دستانه دیتا

    val_sample_limit: int = 100   # اضافه کنید: فقط ۴۰ مورد از ۱۲۵۰ تا برای ولیدیشن سریع
    # =========================================================
    # 3. تنظیمات سایز و نمونه‌برداری
    # =========================================================
    patch_size: Tuple[int,int,int] = (160, 160, 160)
    roi_size: Tuple[int,int,int]   = (160, 160, 160)

    num_samples: int = 3          # برداشت 2 پچ از هر حجم؛ تنوع داده را در هر بچ بالا می‌برد
    min_frac: float = 0.3
    pos_ratio: float = 0.7
    pos_dilate_kernel: int = 3

    # =========================================================
    # 4. معماری مدل
    # =========================================================
    num_classes: int = 4
    base_ch: int = 64
    num_heads: int = 8
    deep_supervision: bool = True

    # دریفت دراپ‌اوت برای جلوگیری از اوورفیت روی 1250 تصویر
    dropout: float = 0.20         # کمی افزایش برای دیتای بزرگ
    attn_dropout: float = 0.15
    proj_dropout: float = 0.15
    ffn_dropout: float = 0.20
    dec_dropout: float = 0.25

    #ds_weights: Optional[List[float]] = None

    # =========================================================
    # 5. تنظیمات آموزش (Advanced AdamW)
    # =========================================================
    epochs: int = 150
    warmup_epochs: int = 10
    lr: float = 1e-5
    #lr: float = 1e-3              # ریت 1e-3 گاهی باعث انفجار گرادیان می‌شود؛ 8e-4 با بچ 16 ایمن‌تر است
    weight_decay: float = 3e-5    # مقدار استاندارد SOTA
    clip_grad_norm: float = 1.0

    use_amp: bool = True

    lambda_dice: float = 1.0
    lambda_ce: float = 1.0
    ds_weight_start: float = 0.5

    # =========================================================
    # 6. تنظیمات پیشرفته و اینفرنس (SOTA Precision)
    # =========================================================
    ema_alpha: float = 0.995      # افزایش پایداری Teacher در دیتای زیاد
    kd_temp: float = 3.0
    kd_max_weight: float = 0.5

    sw_batch_size: int = 2        # ولیدیشن سریع‌تر روی A100
    sw_roi_size: Tuple[int,int,int] = (160, 160, 160)
    sw_overlap: float = 0.7      # ✅ افزایش به 0.7 برای دقت حداکثری در مرزهای تومور
    hd_every: int = 3

    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    channels_last: bool = True

# =========================================================
# مقداردهی نهایی
# =========================================================
cfg = CFG()
#if cfg.deep_supervision and (cfg.ds_weights is None):
#    cfg.ds_weights = [0.50, 0.25]

os.makedirs(cfg.save_dir, exist_ok=True)
print(f"🚀 Gemini SOTA Config Loaded for A100!")

🚀 Gemini SOTA Config Loaded for A100!


In [ ]:
'''
#backup

# Gemini Ultra-SOTA Hyperparameters for BraTS2021
from dataclasses import dataclass
from typing import Tuple, List, Optional
import os
import torch
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass, field

@dataclass
class CFG:
    # =========================================================
    # 1. تنظیمات اصلی
    # =========================================================
    root_path: str = "/content/drive/MyDrive/Colab Notebooks/testi"
    save_dir: str  = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead_v2"
    seed: int = 42

    do_viz: bool = False
    do_plot: bool = True
    log_step_freq: int = 10

    deep_supervision: bool = True
    ds_weights: List[float] = field(default_factory=lambda: [0.5, 0.25])

    # =========================================================
    # 2. مسیرها و دیتالودر (بهینه‌سازی برای سرعت بالا)
    # =========================================================
    dataset_path: str = "/content/data/BraTS"

    cache_backend: str = "ram"
    # با ۸۰ گیگ VRAM و رم بالای سیستم در A100، بخشی از دیتا را کش می‌کنیم
    cache_rate: float = 0.8
    cache_dir: str = "/content/cache_pre"

    kfold: int = 5
    fold: int = 0

    # ✅ تغییر استراتژی: افزایش Batch Size واقعی به جای Accumulation بیش از حد
    # در A100، بچ‌سایز 4 برای رزولوشن 160 به راحتی جا می‌شود و سرعت را 2 برابر می‌کند.
    batch_size: int = 2
    num_workers: int = 10          # افزایش برای تغذیه سریع GPU
    accum_iter: int = 4          # بچ‌سایز موثر همچنان 16 (4 * 4) باقی می‌ماند

    pin_memory: bool = True
    prefetch_factor: int = 2      # افزایش برای لود پیش‌دستانه دیتا

    val_sample_limit: int = 251   # اضافه کنید: فقط ۴۰ مورد از ۱۲۵۰ تا برای ولیدیشن سریع
    # =========================================================
    # 3. تنظیمات سایز و نمونه‌برداری
    # =========================================================
    patch_size: Tuple[int,int,int] = (160, 160, 160)
    roi_size: Tuple[int,int,int]   = (160, 160, 160)

    num_samples: int = 3          # برداشت 2 پچ از هر حجم؛ تنوع داده را در هر بچ بالا می‌برد
    min_frac: float = 0.3
    pos_ratio: float = 0.7
    pos_dilate_kernel: int = 3

    # =========================================================
    # 4. معماری مدل
    # =========================================================
    num_classes: int = 4
    base_ch: int = 64
    num_heads: int = 8
    deep_supervision: bool = True

    # دریفت دراپ‌اوت برای جلوگیری از اوورفیت روی 1250 تصویر
    dropout: float = 0.15         # کمی افزایش برای دیتای بزرگ
    attn_dropout: float = 0.10
    proj_dropout: float = 0.10
    ffn_dropout: float = 0.15
    dec_dropout: float = 0.10

    #ds_weights: Optional[List[float]] = None

    # =========================================================
    # 5. تنظیمات آموزش (Advanced AdamW)
    # =========================================================
    epochs: int = 130
    warmup_epochs: int = 10
    lr: float = 3e-4
    #lr: float = 1e-3              # ریت 1e-3 گاهی باعث انفجار گرادیان می‌شود؛ 8e-4 با بچ 16 ایمن‌تر است
    weight_decay: float = 3e-5    # مقدار استاندارد SOTA
    clip_grad_norm: float = 1.0

    use_amp: bool = True
    amp: bool = True

    lambda_dice: float = 1.0
    lambda_ce: float = 1.0
    ds_weight_start: float = 0.5

    # =========================================================
    # 6. تنظیمات پیشرفته و اینفرنس (SOTA Precision)
    # =========================================================
    ema_alpha: float = 0.995      # افزایش پایداری Teacher در دیتای زیاد
    kd_temp: float = 3.0
    kd_max_weight: float = 0.5

    sw_batch_size: int = 2        # ولیدیشن سریع‌تر روی A100
    sw_roi_size: Tuple[int,int,int] = (160, 160, 160)
    sw_overlap: float = 0.5      # ✅ افزایش به 0.7 برای دقت حداکثری در مرزهای تومور
    hd_every: int = 1

    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    channels_last: bool = True

# =========================================================
# مقداردهی نهایی
# =========================================================
cfg = CFG()
#if cfg.deep_supervision and (cfg.ds_weights is None):
#    cfg.ds_weights = [0.50, 0.25]

os.makedirs(cfg.save_dir, exist_ok=True)
print(f"🚀 Gemini SOTA Config Loaded for A100!")

'''

'\n#backup\n\n# Gemini Ultra-SOTA Hyperparameters for BraTS2021\nfrom dataclasses import dataclass\nfrom typing import Tuple, List, Optional\nimport os\nimport torch\nimport torch.nn.functional as F\nimport numpy as np\nfrom dataclasses import dataclass, field\n\n@dataclass\nclass CFG:\n    # =========================================================\n    # 1. تنظیمات اصلی\n    # =========================================================\n    root_path: str = "/content/drive/MyDrive/Colab Notebooks/testi"\n    save_dir: str  = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead_v2"\n    seed: int = 42\n\n    do_viz: bool = False\n    do_plot: bool = True\n    log_step_freq: int = 10\n\n    deep_supervision: bool = True\n    ds_weights: List[float] = field(default_factory=lambda: [0.5, 0.25])\n\n    # =========================================================\n    # 2. مسیرها و دیتالودر (بهینه\u200cسازی برای سرعت بالا)\n    # ==================================

In [ ]:

'''
# jemini best hyper parameters - UPDATED FOR SOTA 1250 CASES
from dataclasses import dataclass
from typing import Tuple, List, Optional
import os
import torch
import torch.nn.functional as F
import numpy as np

@dataclass
class CFG:
    # =========================================================
    # 1. تنظیمات اصلی
    # =========================================================
    root_path: str = "/content/drive/MyDrive/Colab Notebooks/testi"
    # نام پوشه را برای ایجاد تمایز با تست‌های قبلی کمی تغییر دادیم
    save_dir: str  = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead"
    seed: int = 42

    do_viz: bool = False
    do_plot: bool = True
    log_step_freq: int = 10

    # =========================================================
    # 2. مسیرها و دیتالودر (بهینه شده برای 1250 تصویر)
    # =========================================================
    dataset_path: str = "/content/data/BraTS"

    cache_backend: str = "disk"
    cache_rate: float = 0.0       # برای 1250 دیتا RAM پر نشود (در صورت داشتن RAM بالای 80GB روی 0.5 بگذارید)
    cache_dir: str = "/content/cache_pre"

    kfold: int = 5
    fold: int = 0

    # ✅ تنظیمات بهینه برای A100 (تعادل بین سرعت و پایداری)
    batch_size: int = 2
    num_workers: int = 4
    accum_iter: int = 8           # ✅ قدرت معادل بچ 16 (2 * 8) برای همگرایی بهتر در 1250 دیتا

    pin_memory: bool = True
    prefetch_factor: int = 2

    # =========================================================
    # 3. تنظیمات سایز و نمونه‌برداری (پنجره 160)
    # =========================================================
    patch_size: Tuple[int,int,int] = (160, 160, 160)
    roi_size: Tuple[int,int,int]   = (160, 160, 160)

    num_samples: int = 1
    min_frac: float = 0.3
    pos_ratio: float = 0.5
    pos_dilate_kernel: int = 3

    # =========================================================
    # 4. معماری مدل
    # =========================================================
    num_classes: int = 4
    base_ch: int = 64
    num_heads: int = 8
    deep_supervision: bool = True

    dropout: float = 0.10
    attn_dropout: float = 0.10
    proj_dropout: float = 0.05
    ffn_dropout: float = 0.10
    dec_dropout: float = 0.10

    ds_weights: Optional[List[float]] = None

    # =========================================================
    # 5. تنظیمات آموزش (طبق مقاله MAT و SOTA AdamW)
    # =========================================================
    epochs: int = 100
    warmup_epochs: int = 10       # ✅ افزایش به 10 طبق مقاله MAT برای پایداری در 1250 دیتا
    lr: float = 1e-3              # ✅ افزایش به 1e-3؛ چون بچ‌سایز موثر (16) بالاست، این ریت طبق مقاله MAT بهینه است
    weight_decay: float = 1e-5
    clip_grad_norm: float = 1.0

    use_amp: bool = True
    amp: bool = True

    lambda_dice: float = 1.0
    lambda_ce: float = 1.0
    ds_weight_start: float = 0.5

    # =========================================================
    # 6. تنظیمات پیشرفته و اینفرنس
    # =========================================================
    ema_alpha: float = 0.99
    kd_temp: float = 3.0
    kd_max_weight: float = 0.5

    sw_batch_size: int = 2
    sw_roi_size: Tuple[int,int,int] = (160, 160, 160)
    sw_overlap: float = 0.60      # افزایش اورلپ برای دقت بالاتر در تست (SOTA)
    hd_every: int = 5

    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    channels_last: bool = True

# =========================================================
# مقداردهی نهایی
# =========================================================
cfg = CFG()

if cfg.deep_supervision and (cfg.ds_weights is None):
    cfg.ds_weights = [0.50, 0.25]

os.makedirs(cfg.save_dir, exist_ok=True)

print(f"🚀 SOTA MAT-Based Config for 1250 Cases Loaded!")
print(f"   Window Size: {cfg.roi_size} | Effective Batch: {cfg.batch_size * cfg.accum_iter}")
print(f"   Warmup: {cfg.warmup_epochs} epochs | Max LR: {cfg.lr}")

'''

'\n# jemini best hyper parameters - UPDATED FOR SOTA 1250 CASES\nfrom dataclasses import dataclass\nfrom typing import Tuple, List, Optional\nimport os\nimport torch\nimport torch.nn.functional as F\nimport numpy as np\n\n@dataclass\nclass CFG:\n    # =========================================================\n    # 1. تنظیمات اصلی\n    # =========================================================\n    root_path: str = "/content/drive/MyDrive/Colab Notebooks/testi"\n    # نام پوشه را برای ایجاد تمایز با تست\u200cهای قبلی کمی تغییر دادیم\n    save_dir: str  = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead"\n    seed: int = 42\n\n    do_viz: bool = False\n    do_plot: bool = True\n    log_step_freq: int = 10\n\n    # =========================================================\n    # 2. مسیرها و دیتالودر (بهینه شده برای 1250 تصویر)\n    # =========================================================\n    dataset_path: str = "/content/data/BraTS"\n\n    cache_back

In [ ]:
# Download dataset from kaggle

In [ ]:

#!kaggle datasets list -s brats-2021

In [ ]:
'''
# پاک کردن پوشه قبلی برای شروع تمیز
!rm -rf /content/data/BraTS
!mkdir -p /content/data/BraTS

# دانلود مجدد با نمایش پیشرفت واقعی
!kaggle datasets download -d dschettler8845/brats-2021-task1 -p /content/data/BraTS
'''

'\n# پاک کردن پوشه قبلی برای شروع تمیز\n!rm -rf /content/data/BraTS\n!mkdir -p /content/data/BraTS\n\n# دانلود مجدد با نمایش پیشرفت واقعی\n!kaggle datasets download -d dschettler8845/brats-2021-task1 -p /content/data/BraTS\n'

In [ ]:


from google.colab import drive
import os

# ۱. اتصال به درایو
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# ۲. تنظیمات کگل
!mkdir -p ~/.kaggle
!cp "/content/drive/MyDrive/kaggle.json" ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# ۳. دانلود با مدیریت خطا
# استفاده از --quiet برای جلوگیری از پر شدن لاگ و سرعت بیشتر
print("🚀 Downloading dataset (this may take a while)...")
!kaggle datasets download -d dschettler8845/brats-2021-task1 -p /content/data/BraTS --quiet

# ۴. چک کردن وجود فایل قبل از آنزیپ
zip_path = "/content/data/BraTS/brats-2021-task1.zip"
if os.path.exists(zip_path):
    print('✅ Dataset downloaded successfully. Unzipping...')
    !unzip -q {zip_path} -d /content/data/BraTS
    # حذف فایل زیپ برای آزاد کردن فضا بعد از استخراج
    # !rm {zip_path}
    print('✨ All done!')
else:
    print('❌ Error: The zip file was not found. Check your Kaggle connection or disk space.')


🚀 Downloading dataset (this may take a while)...
Dataset URL: https://www.kaggle.com/datasets/dschettler8845/brats-2021-task1
License(s): copyright-authors
✅ Dataset downloaded successfully. Unzipping...
replace /content/data/BraTS/BraTS2021_00495.tar? [y]es, [n]o, [A]ll, [N]one, [r]ename: N
✨ All done!


In [ ]:
'''
from google.colab import drive
import os

# 1. اتصال به گوگل درایو
drive.mount('/content/drive')

# 2. کپی کردن کلید از درایو به کولب
!mkdir -p ~/.kaggle
!cp "/content/drive/MyDrive/kaggle.json" ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. دانلود دیتاست
print("🚀 Downloading dataset...")
!kaggle datasets download -d dschettler8845/brats-2021-task1 -p /content/data/BraTS
print(' DataSet is downloaded')
!unzip -q /content/data/BraTS/brats-2021-task1.zip -d /content/data/BraTS
'''

'\nfrom google.colab import drive\nimport os\n\n# 1. اتصال به گوگل درایو\ndrive.mount(\'/content/drive\')\n\n# 2. کپی کردن کلید از درایو به کولب\n!mkdir -p ~/.kaggle\n!cp "/content/drive/MyDrive/kaggle.json" ~/.kaggle/\n!chmod 600 ~/.kaggle/kaggle.json\n\n# 3. دانلود دیتاست\nprint("🚀 Downloading dataset...")\n!kaggle datasets download -d dschettler8845/brats-2021-task1 -p /content/data/BraTS\nprint(\' DataSet is downloaded\')\n!unzip -q /content/data/BraTS/brats-2021-task1.zip -d /content/data/BraTS\n'

In [ ]:
'''
import os
print("📂 محتویات پوشه دیتاست:")
print(os.listdir("/content/data/BraTS")[:5])
'''

'\nimport os\nprint("📂 محتویات پوشه دیتاست:")\nprint(os.listdir("/content/data/BraTS")[:5])\n'

In [ ]:
# importing Data Processing

In [ ]:


def setup_data_pipeline(cfg,
    *,
    root,
    notebook="preprocess_pipeline.ipynb",
    zip_path,
    extract_to,
    train_cache,
    val_cache,
    force_reextract=False,):

  path_to_nb = os.path.join(root, notebook)

  # اجرای نوت‌بوک؛ تمام توابع داخل آن حالا در دسترس هستند
  %run "$path_to_nb"


  dataset_root = prepare_dataset_from_archive(
        cfg,
        archive_path=zip_path,
        extract_dir=extract_to,
        ensure_cache_dirs=(train_cache, val_cache),
        force_reextract=force_reextract,
  )
  print("[dataset_root]", dataset_root)

  print('data is extracted ')

  cfg.dataset_path = dataset_root

  train_loader, val_loader = build_loaders(cfg, cfg.dataset_path )

  b = next(iter(train_loader))

  print("image:", b["image"].dtype, tuple(b["image"].shape))  # (N,4,128,128,128)
  print("label:", b["label"].dtype, tuple(b["label"].shape))  # (N,128,128,128)
  print(f"train_loader size = {len(train_loader)}")
  print(f"val_loader size   = {len(val_loader)}")


  return  dataset_root, train_loader, val_loader, b


In [ ]:

import torch
import torch.nn.functional as F
import shutil

cfg = CFG()

device = torch.device(cfg.device if getattr(cfg, "device", "auto") != "auto"
                          else ("cuda" if torch.cuda.is_available() else "cpu"))

ZIP_PATH    = "/content/data/BraTS/BraTS2021_Training_Data.tar"
EXTRACT_TO  = "/content/data/BraTS/dataset"
train_cache = "/content/cache_pre_train"
val_cache   = "/content/cache_pre_val"

# Clear existing cache directories to prevent I/O errors
if os.path.exists(train_cache): shutil.rmtree(train_cache)
if os.path.exists(val_cache): shutil.rmtree(val_cache)

dataset_root, train_loader, val_loader, batch = setup_data_pipeline(
    cfg,
    root=cfg.root_path,
    notebook="preprocess_pipeline.ipynb",
    zip_path=ZIP_PATH,
    extract_to=EXTRACT_TO,
    train_cache=train_cache,
    val_cache=val_cache,
    force_reextract=False, # Force re-extraction and cache regeneration
)


cfg.dataset_path = dataset_root   # مثلا "/content/data/BraTS/dataset"

print("dataset_root:", dataset_root)

print('**********     Data is processed       **********')


NameError: name 'setup_data_pipeline' is not defined

In [ ]:
'''
import torch
import torch.nn.functional as F
import shutil

cfg = CFG()

device = torch.device(cfg.device if getattr(cfg, "device", "auto") != "auto"
                          else ("cuda" if torch.cuda.is_available() else "cpu"))

ZIP_PATH    = "/content/drive/MyDrive/brats2021_5.zip"
EXTRACT_TO  = "/content/data/BraTS/dataset"
train_cache = "/content/cache_pre_train"
val_cache   = "/content/cache_pre_val"

# Clear existing cache directories to prevent I/O errors
if os.path.exists(train_cache): shutil.rmtree(train_cache)
if os.path.exists(val_cache): shutil.rmtree(val_cache)

dataset_root, train_loader, val_loader, batch = setup_data_pipeline(
    cfg,
    root=cfg.root_path,
    notebook="preprocess_pipeline.ipynb",
    zip_path=ZIP_PATH,
    extract_to=EXTRACT_TO,
    train_cache=train_cache,
    val_cache=val_cache,
    force_reextract=False, # Force re-extraction and cache regeneration
)


cfg.dataset_path = dataset_root   # مثلا "/content/data/BraTS/dataset"

print("dataset_root:", dataset_root)

print('**********     Data is processed       **********')
'''

'\nimport torch\nimport torch.nn.functional as F\nimport shutil\n\ncfg = CFG()\n\ndevice = torch.device(cfg.device if getattr(cfg, "device", "auto") != "auto"\n                          else ("cuda" if torch.cuda.is_available() else "cpu"))\n\nZIP_PATH    = "/content/drive/MyDrive/brats2021_5.zip"\nEXTRACT_TO  = "/content/data/BraTS/dataset"\ntrain_cache = "/content/cache_pre_train"\nval_cache   = "/content/cache_pre_val"\n\n# Clear existing cache directories to prevent I/O errors\nif os.path.exists(train_cache): shutil.rmtree(train_cache)\nif os.path.exists(val_cache): shutil.rmtree(val_cache)\n\ndataset_root, train_loader, val_loader, batch = setup_data_pipeline(\n    cfg,\n    root=cfg.root_path,\n    notebook="preprocess_pipeline.ipynb",\n    zip_path=ZIP_PATH,\n    extract_to=EXTRACT_TO,\n    train_cache=train_cache,\n    val_cache=val_cache,\n    force_reextract=False, # Force re-extraction and cache regeneration\n)\n\n\ncfg.dataset_path = dataset_root   # مثلا "/content/data/Bra

In [ ]:
# importing Axial Attention Model

In [ ]:

# ===== simple_model_loader.py =====
from importnb import Notebook
import importlib, inspect, sys, os
from pathlib import Path

def build_model(
    cfg,
    module: str,                 # نام فایل نوت‌بوک بدون پسوند یا با ".ipynb" مثل: "MultiSeg_Model2" یا "MultiSeg_Model2.ipynb"
    class_name: str,             # نام کلاس داخل نوت‌بوک
    root: str | None = None,     # پوشه‌ی نوت‌بوک‌ها؛ پیش‌فرض از cfg.root_path
    fresh: bool = True,          # هر بار از نو import شود
    **extra_kwargs               # هرچی دستی بدی، اگر در signature بود ست می‌شود
):
    # 1) مسیر
    root = root or getattr(cfg, "model_py_path", getattr(cfg, "root_path", "."))
    root = os.path.abspath(root)
    if root not in sys.path:
        sys.path.append(root)

    # 2) نام ماژول (بدون .ipynb)
    module = Path(module).stem  # "x.ipynb" -> "x"

    # 3) تازه‌سازی
    if fresh and module in sys.modules:
        del sys.modules[module]

    # 4) import نوت‌بوک مثل ماژول
    with Notebook():
        mod = importlib.import_module(module)

    if not hasattr(mod, class_name):
        raise AttributeError(f"کلاس '{class_name}' داخل '{module}.ipynb' پیدا نشد.")

    Model = getattr(mod, class_name)

    # 5) ساخت kwargs خیلی ساده و خودکار از cfg
    sig_params = inspect.signature(Model).parameters
    pick = {}

    def get_any(*names, default=None):
        for n in names:
            if hasattr(cfg, n):
                return getattr(cfg, n)
        return default

    # نگاشت رایج + پارامترهای جدید dropout
    candidates = {
        # عمومی
        "in_channels":       get_any("in_channels", "in_ch", default=None),
        "num_classes":       get_any("num_classes", "out_ch", default=None),
        "base_ch":           get_any("base_ch", default=None),
        "num_heads":         get_any("num_heads", "heads", default=None),
        "up_method":         get_any("up_method", default=None),
        "deep_supervision":  get_any("deep_supervision", default=None),
        "img_size":          get_any("imgshape", "patch_size", default=None),
        "patch_size":        get_any("patch_size", default=None),

        # مخصوص مدل تو (اگر در امضای کلاس وجود داشته باشند پاس می‌شوند)
        "use_gating":        get_any("use_gating", default=None),
        "bottleneck_heads":  get_any("bottleneck_heads", default=None),
        "max_len":           get_any("max_len", default=None),
        "negative_slope":    get_any("negative_slope", default=None),
        "tcfc_depthwise":    get_any("tcfc_depthwise", default=None),
        "norm_type":         get_any("norm_type", default=None),

        # --- پارامترهای dropout (جدید) ---
        "dropout":           get_any("dropout", default=None),
        "attn_dropout":      get_any("attn_dropout", default=None),
        "proj_dropout":      get_any("proj_dropout", default=None),
        "ffn_dropout":       get_any("ffn_dropout", default=None),
        "dec_dropout":       get_any("dec_dropout", default=None),
    }

    # هرچی در امضای مدل هست و مقدارش None نیست را بردار
    for k, v in candidates.items():
        if k in sig_params and v is not None:
            pick[k] = v

    # هرچی دستی دادی (extra_kwargs) اگر در امضا باشد، override کند
    for k, v in extra_kwargs.items():
        if k in sig_params:
            pick[k] = v

    # 6) بساز و تمام
    model = Model(**pick)
    print(f"[model] {module}.{class_name} kwargs={pick}")
    return model


In [ ]:
# فرض: cfg.root_path = مسیر پوشه‌ی نوت‌بوک‌ها
#from simple_model_loader import build_model

model1 = build_model(
    cfg,
    module="MultiSeg_Model2.ipynb",
    class_name="MultiModalSegNet",
    root=cfg.root_path,   # یا نذار، خودش از cfg برمی‌داره
    fresh=True            # هر بار تغییرات نوت‌بوک اعمال شود
).to(cfg.device)


[model] MultiSeg_Model2.MultiModalSegNet kwargs={'num_classes': 4, 'base_ch': 64, 'num_heads': 8, 'deep_supervision': True, 'dropout': 0.2, 'attn_dropout': 0.15, 'proj_dropout': 0.15, 'ffn_dropout': 0.2, 'dec_dropout': 0.25}


In [ ]:
#test

In [ ]:
'''
b = next(iter(train_loader))
print("train_loader has kd_mask:", "kd_mask" in b)
print("image:", tuple(b["image"].shape), b["image"].dtype)  # (bs*num_samples, 4, 128, 128, 128), float32
print("label:", tuple(b["label"].shape), b["label"].dtype)  # (bs*num_samples, 128, 128, 128), int64

v = next(iter(val_loader))
print("val_loader has kd_mask:", "kd_mask" in v)
print("image:", tuple(v["image"].shape), v["image"].dtype)  # (bs*num_samples, 4, 128, 128, 128), float32
print("label:", tuple(v["label"].shape), v["label"].dtype)  # (bs*num_samples, 128, 128, 128), int64
'''

'\nb = next(iter(train_loader))\nprint("train_loader has kd_mask:", "kd_mask" in b)\nprint("image:", tuple(b["image"].shape), b["image"].dtype)  # (bs*num_samples, 4, 128, 128, 128), float32\nprint("label:", tuple(b["label"].shape), b["label"].dtype)  # (bs*num_samples, 128, 128, 128), int64\n\nv = next(iter(val_loader))\nprint("val_loader has kd_mask:", "kd_mask" in v)\nprint("image:", tuple(v["image"].shape), v["image"].dtype)  # (bs*num_samples, 4, 128, 128, 128), float32\nprint("label:", tuple(v["label"].shape), v["label"].dtype)  # (bs*num_samples, 128, 128, 128), int64\n'

In [ ]:
import torch
import gc
import os

def clean_gpu_memory():
    print("🧹 Cleaning GPU memory...")

    # 1. گزارش وضعیت قبل از پاکسازی
    if torch.cuda.is_available():
        print(f"📊 Initial Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
        print(f"📊 Initial Reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GB")

    # 2. حذف متغیرهای سنگین از حافظه گلوبال (اگر وجود داشته باشند)
    # این لیست را بر اساس نام متغیرهای خودت می‌توانی تغییر دهی
    heavy_vars = ['model', 'optimizer', 'scheduler', 'scaler', 'loss_fn', 'outputs', 'loss', 'logits']

    for var in heavy_vars:
        if var in globals():
            del globals()[var]
            print(f"   🗑️ Deleted global variable: {var}")

    # 3. اجرای زباله‌روب پایتون (Python Garbage Collector)
    gc.collect()

    # 4. خالی کردن کش کودا (PyTorch CUDA Cache)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    # 5. گزارش وضعیت بعد از پاکسازی
    if torch.cuda.is_available():
        print(f"✨ Final Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
        print(f"✨ Final Reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GB")

    print("✅ GPU is clean and ready!")

# اجرا
clean_gpu_memory()

🧹 Cleaning GPU memory...
📊 Initial Allocated: 1.99 GB
📊 Initial Reserved:  4.91 GB
✨ Final Allocated: 1.99 GB
✨ Final Reserved:  4.91 GB
✅ GPU is clean and ready!


In [ ]:
'''
import os
import copy
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.metrics import HausdorffDistanceMetric
#from monai.transforms import KeepLargestConnectedComponent, RandFlip, RandRotate, RandScale, RandGaussianNoise
from monai.utils import set_determinism

# =========================
# 1) CONFIGURATION
# =========================
@dataclass
class CFG:
    save_dir: str = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_SOTA"
    seed: int = 42

    batch_size: int = 4
    num_workers: int = 4
    accum_iter: int = 3

    epochs: int = 100
    lr: float = 1e-4
    weight_decay: float = 1e-5
    warmup_epochs: int = 5

    roi_size: tuple = (128, 128, 128)  # افزایش سایز ROI به 128x128x128 برای آموزش بهینه
    sw_batch_size: int = 4
    sw_overlap: float = 0.5

    hd_every: int = 5

    deep_supervision: bool = True
    ds_weights: tuple = (0.5, 0.25)

    use_amp: bool = True
    ema_alpha: float = 0.99

    patience: int = 15
    min_delta: float = 1e-4

    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = CFG()
set_determinism(cfg.seed)

# =========================
# 2) EMA TEACHER
# =========================
class EMATeacher:
    def __init__(self, model, alpha=0.99):
        self.teacher = copy.deepcopy(model).eval()
        for p in self.teacher.parameters():
            p.requires_grad = False
        self.alpha = alpha

    @torch.no_grad()
    def update(self, student):
        for t_param, s_param in zip(self.teacher.parameters(), student.parameters()):
            t_param.data.mul_(self.alpha).add_(s_param.data, alpha=1.0 - self.alpha)

# =========================
# 3) LOSS
# =========================
class HybridLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        # وزن‌ها برای کلاس‌های BraTS: [پس‌زمینه، بافت مرده، ادم، تومور]
        self.ce_weights = torch.tensor([0.10, 1.0, 1.0, 1.5], device=device)

        # استفاده از DiceCELoss برای ترکیب Dice و CrossEntropy
        self.dice_loss = DiceCELoss(
            include_background=False,
            to_onehot_y=True,         # تبدیل خودکار لیبل به One-hot برای Dice
            softmax=True,             # اعمال Softmax روی Logits
            batch=True,
            lambda_dice=1.0,
            lambda_ce=1.0,
            smooth_nr=1e-5,
            smooth_dr=1e-5
        )

        # استفاده از CrossEntropyLoss برای Cross-Entropy
        self.ce_loss = nn.CrossEntropyLoss(weight=self.ce_weights)

    def forward(self, logits, y):
        # محاسبه Loss برای Dice
        dice_loss_value = self.dice_loss(logits, y)

        # CrossEntropy برای طبقات مختلف
        ce_loss_value = self.ce_loss(logits, y.squeeze(1))  # نیاز به حذف ابعاد اضافی برای CrossEntropyLoss

        return dice_loss_value + ce_loss_value


# =========================
# 4) LOGGER
# =========================
class CSVLogger:
    def __init__(self, save_dir):
        self.path = os.path.join(save_dir, "training_log.csv")
        self.fields = [
            "epoch","train_loss","val_loss","lr",
            "Dice_WT","Dice_TC","Dice_ET","Dice_Mean",
            "Score_ETfocus",
            "HD95_raw_WT","HD95_raw_TC","HD95_raw_ET","HD95_raw_mean",
            "HD95_post_WT","HD95_post_TC","HD95_post_ET","HD95_post_mean",
        ]
        if not os.path.exists(self.path):
            with open(self.path, "w", newline="") as f:
                csv.DictWriter(f, fieldnames=self.fields).writeheader()

    def log(self, row):
        with open(self.path, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=self.fields).writerow(row)

# =========================
# 5) METRICS HELPERS
# =========================
@torch.no_grad()
def to_regions(lbl_4class):
    """
    lbl_4class: [B,H,W,D] long
    returns: [B,3,H,W,D] float (WT,TC,ET)
    """
    wt = (lbl_4class > 0).float()
    tc = ((lbl_4class == 1) | (lbl_4class == 3)).float()
    et = (lbl_4class == 3).float()
    return torch.stack([wt, tc, et], dim=1)

def train_one_epoch(model, loader, optimizer, loss_fn, scaler):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for i, batch in enumerate(loader):
        x = batch["image"].to(cfg.device)
        y = batch["label"].to(cfg.device)

        # تبدیل ابعاد به [B, 1, H, W, D] (اگر کانال نداشته باشد، اضافه می‌شود)
        if y.ndim == 4:
            y = y.unsqueeze(1)
        elif y.ndim == 5 and y.shape[1] != 1:
            y = torch.argmax(y, dim=1, keepdim=True)

        y = y.long()
        y[y == 4] = 3  # اصلاح برچسب‌ها

        with torch.amp.autocast("cuda", enabled=cfg.use_amp):
            out = model(x)  # خروجی مدل که یک دیکشنری است
            logits = out["logits"]

            # محاسبه Loss با استفاده از y به طور مستقیم (بدون تبدیل به One-hot)
            loss = loss_fn(logits, y)

            # Deep supervision (اگر فعال باشد)
            if cfg.deep_supervision and ("aux1" in out):
                loss = loss + cfg.ds_weights[0] * loss_fn(out["aux1"], y)
                loss = loss + cfg.ds_weights[1] * loss_fn(out["aux2"], y)

            loss = loss / cfg.accum_iter

        scaler.scale(loss).backward()

        # اعمال به‌روزرسانی‌های گریدینت و گام‌های آپدیت
        do_step = ((i + 1) % cfg.accum_iter == 0) or ((i + 1) == len(loader))
        if do_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            if hasattr(model, "ema_teacher"):
                model.ema_teacher.update(model)

        total_loss += loss.item()

    return (total_loss * cfg.accum_iter) / len(loader)

from monai.metrics import HausdorffDistanceMetric

from monai.transforms import KeepLargestConnectedComponent

@torch.no_grad()
def validate(model, loader, loss_fn):
    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model
    net.eval()

    # For region masks (WT/TC/ET): include_background must be False
    hd_metric_raw = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    hd_metric_post = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")

    post_filter = KeepLargestConnectedComponent(applied_labels=[1, 2, 3], is_onehot=False)

    dices = []
    val_loss = 0.0

    for batch in loader:
        x = batch["image"].to(cfg.device)
        y = batch["label"].to(cfg.device)

        if y.ndim == 5:
            y = y.squeeze(1)
        elif y.ndim == 4:
            y = y.unsqueeze(1)

        y = y.long()
        y[y == 4] = 3  # Convert labels

        # Sliding window inference (dict -> logits)
        logits = sliding_window_inference(
            x,
            cfg.roi_size,
            cfg.sw_batch_size,
            predictor=lambda z: net(z)["logits"],
            overlap=cfg.sw_overlap,
            mode="gaussian"
        )

        val_loss += loss_fn(logits, y).item()

        pred_raw = torch.argmax(logits, dim=1)  # [B,H,W,D]

        # Postprocess on GPU (instead of CPU to avoid bottlenecks)
        pred_raw_cpu = pred_raw.cpu()
        processed = []
        for p in pred_raw_cpu:
            p2 = post_filter(p.unsqueeze(0)).squeeze(0)
            processed.append(p2)
        pred_post = torch.stack(processed).to(cfg.device)

        true_lbl = y.squeeze(1)  # [B,H,W,D]

        raw_reg = to_regions(pred_raw)
        post_reg = to_regions(pred_post)
        true_reg = to_regions(true_lbl)

        # Dice per region
        inter = (raw_reg * true_reg).sum((2, 3, 4))
        union = raw_reg.sum((2, 3, 4)) + true_reg.sum((2, 3, 4))
        dice = torch.nan_to_num(2.0 * inter / (union + 1e-5), 0.0)  # [B,3]
        dices.append(dice.mean(0).cpu().numpy())

        # HD95
        hd_metric_raw(raw_reg, true_reg)
        hd_metric_post(post_reg, true_reg)

    d_cls = np.mean(dices, axis=0)  # [WT, TC, ET]

    # استخراج نتیجه Hausdorff Distance برای raw (بدون پس‌پردازش) و post (بعد از پس‌پردازش)
    hd_raw = np.nan_to_num(hd_metric_raw.aggregate().cpu().numpy(), nan=100.0, posinf=100.0, neginf=100.0)
    hd_post = np.nan_to_num(hd_metric_post.aggregate().cpu().numpy(), nan=100.0, posinf=100.0, neginf=100.0)

    # Reset metrics for the next validation iteration
    hd_metric_raw.reset()
    hd_metric_post.reset()

    return d_cls, hd_raw, hd_post, val_loss / max(1, len(loader))


# =========================
# 8) MAIN TRAINING LOOP
# =========================
def run_training(model, train_loader, val_loader):
    os.makedirs(cfg.save_dir, exist_ok=True)
    logger = CSVLogger(cfg.save_dir)

    loss_fn = HybridLoss(cfg.device).to(cfg.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    # Warmup + Cosine scheduler
    def lr_lambda(e):
        if e < cfg.warmup_epochs:
            return (e + 1) / cfg.warmup_epochs
        t = (e - cfg.warmup_epochs) / max(1, (cfg.epochs - cfg.warmup_epochs))
        return 0.5 * (1 + np.cos(np.pi * t))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)

    # EMA teacher
    model.ema_teacher = EMATeacher(model, cfg.ema_alpha)

    # Check for existing checkpoint to resume from
    checkpoint_path = os.path.join(cfg.save_dir, "LAST_CHECKPOINT.pt")
    best_model_path = os.path.join(cfg.save_dir, "BEST_EMA_MODEL.pt")

    start_epoch = 1
    best_score = -1e9

    # If last checkpoint exists, load from it
    if os.path.exists(checkpoint_path):
        print(f"🔄 Found last checkpoint! Resuming from {checkpoint_path}...")
        try:
            ckpt = torch.load(checkpoint_path, map_location=cfg.device)
            model.load_state_dict(ckpt["model_state_dict"])
            model.ema_teacher.teacher.load_state_dict(ckpt["ema_teacher_state_dict"])
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            scaler.load_state_dict(ckpt["scaler_state_dict"])
            start_epoch = ckpt["epoch"] + 1
            best_score = ckpt["best_score"]
            print(f"✅ Successfully resumed at Epoch {start_epoch}")
        except Exception as e:
            print(f"⚠️ Could not resume: {e}. Starting from scratch.")

    # If no last checkpoint, try best model
    elif os.path.exists(best_model_path):
        print(f"🔄 Found best model! Resuming from {best_model_path}...")
        model.load_state_dict(torch.load(best_model_path, map_location=cfg.device))
        print(f"✅ Successfully resumed from best model")

    print("\n" + "=" * 100)
    print(f"{'Ep':<3} | {'vLoss':<7} | {'Dice (WT/TC/ET/Mean)':<24} | {'Score':<6} | {'HD95 POST (W/T/E/M)'}")
    print("-" * 100)

    for ep in range(start_epoch, cfg.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, scaler)
        d_cls, hd_raw, hd_post, v_loss = validate(model, val_loader, loss_fn)

        curr_lr = optimizer.param_groups[0]["lr"]
        d_mean = float(np.mean(d_cls))

        # ET-focused score for saving best (helps push ET up)
        score = float(0.2 * d_cls[0] + 0.3 * d_cls[1] + 0.5 * d_cls[2])

        hr_m = float(np.mean(hd_raw))
        hp_m = float(np.mean(hd_post))

        # Safe handling of HD95 values with error prevention for missing data
        hd_post_values = [
            hd_post[0] if len(hd_post) > 0 and hd_post[0] is not None else 0.0,
            hd_post[1] if len(hd_post) > 1 and hd_post[1] is not None else 0.0,
            hd_post[2] if len(hd_post) > 2 and hd_post[2] is not None else 0.0
        ]

        print(f"{ep:02d} | {v_loss:.4f} | "
              f"{d_cls[0]:.3f}/{d_cls[1]:.3f}/{d_cls[2]:.3f}/{d_mean:.3f} | "
              f"{score:.3f} | "
              f"{hd_post_values[0]:.1f}/{hd_post_values[1]:.1f}/{hd_post_values[2]:.1f}/{hp_m:.1f}")

        # Log data to CSV
        logger.log({
            "epoch": ep,
            "train_loss": float(train_loss),
            "val_loss": float(v_loss),
            "lr": float(curr_lr),

            "Dice_WT": float(d_cls[0]),
            "Dice_TC": float(d_cls[1]),
            "Dice_ET": float(d_cls[2]),
            "Dice_Mean": d_mean,
            "Score_ETfocus": float(score),

            "HD95_raw_WT": float(hd_raw[0] if len(hd_raw) > 0 else 0.0),
            "HD95_raw_TC": float(hd_raw[1] if len(hd_raw) > 1 else 0.0),
            "HD95_raw_ET": float(hd_raw[2] if len(hd_raw) > 2 else 0.0),
            "HD95_raw_mean": hr_m,

            "HD95_post_WT": hd_post_values[0],
            "HD95_post_TC": hd_post_values[1],
            "HD95_post_ET": hd_post_values[2],
            "HD95_post_mean": hp_m,
        })

        # Save best EMA model
        if score > best_score + cfg.min_delta:
            best_score = score
            torch.save(model.ema_teacher.teacher.state_dict(), best_model_path)
            print(f"💾 New Best EMA Model Saved! (Score: {score:.4f}, MeanDice: {d_mean:.4f})")

        # Save last checkpoint
        torch.save({
            "epoch": ep,
            "best_score": best_score,
            "model_state_dict": model.state_dict(),
            "ema_teacher_state_dict": model.ema_teacher.teacher.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
        }, checkpoint_path)

        scheduler.step()

# =========================
# 9) EXECUTION
# =========================
# IMPORTANT: model1 should be your MultiModalSegNet (already created with deep supervision enabled)
model = model1.to(cfg.device)
run_training(model, train_loader, val_loader)

'''

# works well without hd_every

In [ ]:
'''import os
import copy
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.metrics import HausdorffDistanceMetric
from monai.transforms import KeepLargestConnectedComponent
from monai.utils import set_determinism

# =========================
# 1) CONFIGURATION
# =========================
@dataclass
class CFG:
    save_dir: str = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_SOTA"
    seed: int = 42
    batch_size: int = 4
    num_workers: int = 4
    accum_iter: int = 3
    epochs: int = 100
    lr: float = 1e-4
    weight_decay: float = 1e-5
    warmup_epochs: int = 5
    roi_size: tuple = (128, 128, 128)
    sw_batch_size: int = 4
    sw_overlap: float = 0.5
    hd_every: int = 5  # Every `hd_every` epochs calculate HD
    deep_supervision: bool = True
    ds_weights: tuple = (0.5, 0.25)
    use_amp: bool = True
    ema_alpha: float = 0.99
    patience: int = 15
    min_delta: float = 1e-4
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = CFG()
set_determinism(cfg.seed)

# =========================
# 2) EMA TEACHER
# =========================
class EMATeacher:
    def __init__(self, model, alpha=0.99):
        self.teacher = copy.deepcopy(model).eval()
        for p in self.teacher.parameters():
            p.requires_grad = False
        self.alpha = alpha

    @torch.no_grad()
    def update(self, student):
        for t_param, s_param in zip(self.teacher.parameters(), student.parameters()):
            t_param.data.mul_(self.alpha).add_(s_param.data, alpha=1.0 - self.alpha)

# =========================
# 3) LOSS (اصلاح شده برای رفع TypeError و خطای ابعاد)
# =========================
class HybridLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        # وزن‌ها برای کلاس‌های BraTS: [BG, NCR, ED, ET]
        weights = torch.tensor([0.10, 1.0, 1.0, 1.5], device=device)

        # DiceLoss خالص (بدون CE چون در برخی نسخه‌ها خطا می‌دهد)
        self.dice_fn = DiceCELoss(
            include_background=False,
            to_onehot_y=True,
            softmax=True,
            batch=True,
            lambda_dice=1.0,
            lambda_ce=0.0,
            smooth_nr=1e-5,
            smooth_dr=1e-5
        )

        # CrossEntropy جداگانه برای اعمال وزن‌ها
        self.ce_fn = nn.CrossEntropyLoss(weight=weights)

    def forward(self, logits, y):
        # Dice محاسبه [B, 1, H, W, D]
        dice_loss = self.dice_fn(logits, y)

        # CE محاسبه (نیاز به حذف بعد کانال لیبل دارد)
        if y.ndim == 5 and y.shape[1] == 1:
            target_ce = y.squeeze(1).long()
        else:
            target_ce = y.long()

        ce_loss = self.ce_fn(logits, target_ce)
        return dice_loss + ce_loss

# =========================
# 4) LOGGER & HELPERS
# =========================
class CSVLogger:
    def __init__(self, save_dir):
        self.path = os.path.join(save_dir, "training_log.csv")
        self.fields = [
            "epoch","train_loss","val_loss","lr",
            "Dice_WT","Dice_TC","Dice_ET","Dice_Mean",
            "Score_ETfocus","HD95_mean"
        ]
        if not os.path.exists(self.path):
            with open(self.path, "w", newline="") as f:
                csv.DictWriter(f, fieldnames=self.fields).writeheader()

    def log(self, row):
        with open(self.path, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=self.fields).writerow(row)

@torch.no_grad()
def to_regions(lbl_4class):
    # تبدیل لیبل‌های تکی به ریجن‌های BraTS برای محاسبه متریک
    wt = (lbl_4class > 0).float()
    tc = ((lbl_4class == 1) | (lbl_4class == 3)).float()
    et = (lbl_4class == 3).float()
    return torch.stack([wt, tc, et], dim=1)

# =========================
# 5) TRAIN & VALIDATE
# =========================
# =========================
# 5) TRAIN & VALIDATE
# =========================
def train_one_epoch(model, loader, optimizer, loss_fn, scaler):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for i, batch in enumerate(loader):
        x = batch["image"].to(cfg.device)
        y = batch["label"].to(cfg.device)

        if y.ndim == 4: y = y.unsqueeze(1)
        y = y.long()
        y[y == 4] = 3

        with torch.amp.autocast("cuda", enabled=cfg.use_amp):
            out = model(x)
            logits = out["logits"]
            loss = loss_fn(logits, y)
            if cfg.deep_supervision and ("aux1" in out):
                loss += cfg.ds_weights[0] * loss_fn(out["aux1"], y)
                loss += cfg.ds_weights[1] * loss_fn(out["aux2"], y)
            loss /= cfg.accum_iter

        scaler.scale(loss).backward()

        if ((i + 1) % cfg.accum_iter == 0) or ((i + 1) == len(loader)):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if hasattr(model, "ema_teacher"):
                model.ema_teacher.update(model)

        total_loss += loss.item()
    return (total_loss * cfg.accum_iter) / len(loader)

@torch.no_grad()
def validate(model, loader, loss_fn, ep):  # Adding ep parameter here
    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model
    net.eval()

    hd_metric = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    dices = []
    val_loss = 0.0

    for batch in loader:
        x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
        if y.ndim == 4: y = y.unsqueeze(1)
        y = y.long()
        y[y == 4] = 3

        logits = sliding_window_inference(
            x, cfg.roi_size, cfg.sw_batch_size,
            predictor=lambda z: net(z)["logits"], overlap=cfg.sw_overlap
        )

        val_loss += loss_fn(logits, y).item()

        pred = torch.argmax(logits, dim=1, keepdim=True)
        raw_reg = to_regions(pred.squeeze(1))
        true_reg = to_regions(y.squeeze(1))

        # Dice calculation
        inter = (raw_reg * true_reg).sum((2, 3, 4))
        union = raw_reg.sum((2, 3, 4)) + true_reg.sum((2, 3, 4))
        dice = torch.nan_to_num(2.0 * inter / (union + 1e-5), 0.0)
        dices.append(dice.mean(0).cpu().numpy())
        hd_metric(raw_reg, true_reg)

    d_cls = np.mean(dices, axis=0)
    hd_val = np.nan_to_num(hd_metric.aggregate().cpu().numpy(), nan=100.0)
    hd_metric.reset()

    return d_cls, hd_val, val_loss / len(loader)

# =========================
# 6) RUN TRAINING
# =========================
def run_training(model, train_loader, val_loader):
    os.makedirs(cfg.save_dir, exist_ok=True)
    logger = CSVLogger(cfg.save_dir)
    loss_fn = HybridLoss(cfg.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)
    model.ema_teacher = EMATeacher(model, cfg.ema_alpha)

    best_score = -1.0
    print(f"Starting Training on {cfg.device}...")

    for ep in range(1, cfg.epochs + 1):
        t_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, scaler)
        d_cls, hd_val, v_loss = validate(model, val_loader, loss_fn, ep)  # Pass ep here

        d_mean = np.mean(d_cls)
        score = 0.2*d_cls[0] + 0.3*d_cls[1] + 0.5*d_cls[2] # ET focused

        print(f"Ep {ep:02d} | Loss: {t_loss:.4f}/{v_loss:.4f} | Dice: {d_cls[0]:.3f}/{d_cls[1]:.3f}/{d_cls[2]:.3f} | Score: {score:.4f}")

        logger.log({
            "epoch": ep, "train_loss": t_loss, "val_loss": v_loss,
            "lr": optimizer.param_groups[0]["lr"], "Dice_WT": d_cls[0],
            "Dice_TC": d_cls[1], "Dice_ET": d_cls[2], "Dice_Mean": d_mean,
            "Score_ETfocus": score, "HD95_mean": np.mean(hd_val)
        })

        if score > best_score:
            best_score = score
            torch.save(model.ema_teacher.teacher.state_dict(), os.path.join(cfg.save_dir, "BEST_MODEL.pt"))
            print("💾 Best Model Saved!")

        scheduler.step()

# =========================
# 7) EXECUTION
# =========================
model = model1.to(cfg.device) # model1 باید از قبل تعریف شده باشد
run_training(model, train_loader, val_loader)


# =========================
# 7) EXECUTION
# =========================
model = model1.to(cfg.device) # model1 باید از قبل تعریف شده باشد
run_training(model, train_loader, val_loader)

'''

In [ ]:
'''import os
import copy
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.metrics import HausdorffDistanceMetric
from monai.transforms import KeepLargestConnectedComponent
from monai.utils import set_determinism

# =========================
# 1) CONFIGURATION (بهینه‌سازی شده برای جهش یادگیری)
# =========================
@dataclass
class CFG:
    save_dir: str = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_SOTA"
    seed: int = 42
    batch_size: int = 4
    num_workers: int = 4
    accum_iter: int = 1  # آپدیت سریع‌تر گریدینت روی A100
    epochs: int = 100
    lr: float = 4e-4     # نرخ یادگیری تهاجمی‌تر
    weight_decay: float = 1e-5
    warmup_epochs: int = 2 # وارم‌آپ سریع
    roi_size: tuple = (128, 128, 128)
    sw_batch_size: int = 4
    sw_overlap: float = 0.5
    hd_every: int = 5
    deep_supervision: bool = True
    ds_weights: tuple = (0.5, 0.25)
    use_amp: bool = True
    ema_alpha: float = 0.99
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = CFG()
set_determinism(cfg.seed)

# =========================
# 2) EMA TEACHER & HYBRID LOSS (با وزن‌دهی جدید)
# =========================
class EMATeacher:
    def __init__(self, model, alpha=0.99):
        self.teacher = copy.deepcopy(model).eval()
        for p in self.teacher.parameters():
            p.requires_grad = False
        self.alpha = alpha

    @torch.no_grad()
    def update(self, student):
        for t_param, s_param in zip(self.teacher.parameters(), student.parameters()):
            t_param.data.mul_(self.alpha).add_(s_param.data, alpha=1.0 - self.alpha)

class HybridLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        # تمرکز شدید روی کلاس‌های تومور (1,2,3) و جریمه کمتر برای پس‌زمینه (0)
        self.ce_weights = torch.tensor([0.05, 1.0, 1.0, 2.0], device=device)
        self.dice_loss = DiceCELoss(include_background=False, to_onehot_y=True, softmax=True, batch=True)
        self.ce_loss = nn.CrossEntropyLoss(weight=self.ce_weights)

    def forward(self, logits, y):
        return self.dice_loss(logits, y) + self.ce_loss(logits, y.squeeze(1).long())

@torch.no_grad()
def to_regions(lbl):
    wt = (lbl > 0).float()
    tc = ((lbl == 1) | (lbl == 3)).float()
    et = (lbl == 3).float()
    return torch.stack([wt, tc, et], dim=1)

# =========================
# 3) VALIDATE FUNCTION
# =========================
@torch.no_grad()
def validate(model, loader, loss_fn, epoch):
    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model
    net.eval()
    compute_hd = (epoch % cfg.hd_every == 0)
    hd_m_raw = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    hd_m_post = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    post_filter = KeepLargestConnectedComponent(applied_labels=[1, 2, 3], is_onehot=False)

    dices, val_loss = [], 0.0
    for batch in loader:
        x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
        if y.ndim == 4: y = y.unsqueeze(1)
        y = y.long(); y[y == 4] = 3
        logits = sliding_window_inference(x, cfg.roi_size, cfg.sw_batch_size, lambda z: net(z)["logits"], overlap=cfg.sw_overlap)
        val_loss += loss_fn(logits, y).item()
        pred_raw = torch.argmax(logits, dim=1, keepdim=True)
        raw_reg, true_reg = to_regions(pred_raw.squeeze(1)), to_regions(y.squeeze(1))

        inter = (raw_reg * true_reg).sum((2,3,4))
        union = raw_reg.sum((2,3,4)) + true_reg.sum((2,3,4))
        dices.append(torch.nan_to_num(2.*inter/(union+1e-5), 0.).mean(0).cpu().numpy())

        if compute_hd:
            processed = torch.stack([post_filter(p.cpu().unsqueeze(0)).squeeze(0) for p in pred_raw.squeeze(1)]).to(cfg.device)
            post_reg = to_regions(processed)
            hd_m_raw(raw_reg, true_reg); hd_m_post(post_reg, true_reg)

    d_cls = np.mean(dices, axis=0)

    def safe_agg(metric):
        val = metric.aggregate()
        if isinstance(val, torch.Tensor): val = val.cpu().numpy()
        res = np.ones(3) * 100.0
        if np.isscalar(val): res[:] = val
        elif len(val) >= 3: res = val[:3]
        elif len(val) > 0: res[:len(val)] = val
        return np.nan_to_num(res, nan=100.0)

    h_raw = safe_agg(hd_m_raw) if compute_hd else np.array([-1.0]*3)
    h_post = safe_agg(hd_m_post) if compute_hd else np.array([-1.0]*3)
    return d_cls, h_raw, h_post, val_loss / len(loader)

# =========================
# 4) MAIN TRAINING LOOP
# =========================
def run_training(model, train_loader, val_loader):
    os.makedirs(cfg.save_dir, exist_ok=True)
    loss_fn = HybridLoss(cfg.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    # Linear Warmup + Cosine Annealing
    def lr_lambda(epoch):
        if epoch < cfg.warmup_epochs:
            return (epoch + 1) / cfg.warmup_epochs
        progress = (epoch - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs)
        return 0.5 * (1.0 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)
    model.ema_teacher = EMATeacher(model, cfg.ema_alpha)
    best_score = -1.0

    print("\n" + "=" * 115)
    print(f"{'Ep':<3} | {'vLoss':<7} | {'Dice (WT/TC/ET/Mean)':<24} | {'Score':<6} | {'HD95 POST (W/T/E/M)':<20} | {'HD95 RAW (W/T/E/M)'}")
    print("-" * 115)

    for ep in range(1, cfg.epochs + 1):
        model.train()
        t_loss = 0.0
        for i, batch in enumerate(train_loader):
            x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
            if y.ndim == 4: y = y.unsqueeze(1)
            y = y.long(); y[y == 4] = 3
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                out = model(x)
                loss = loss_fn(out["logits"], y)
                if cfg.deep_supervision and "aux1" in out:
                    loss += cfg.ds_weights[0] * loss_fn(out["aux1"], y)
                    loss += cfg.ds_weights[1] * loss_fn(out["aux2"], y)
            scaler.scale(loss/cfg.accum_iter).backward()
            if (i+1) % cfg.accum_iter == 0 or (i+1) == len(train_loader):
                scaler.step(optimizer); scaler.update()
                if hasattr(model, "ema_teacher"): model.ema_teacher.update(model)
            t_loss += loss.item()

        d_cls, h_raw, h_post, v_loss = validate(model, val_loader, loss_fn, ep)
        d_mean, score = np.mean(d_cls), (0.2*d_cls[0] + 0.3*d_cls[1] + 0.5*d_cls[2])

        def fmt_hd(arr):
            if arr[0] < 0: return f"{'-/-/-/-':<20}"
            return f"{arr[0]:.1f}/{arr[1]:.1f}/{arr[2]:.1f}/{np.mean(arr):.1f}"

        print(f"{ep:02d} | {v_loss:.4f} | {d_cls[0]:.3f}/{d_cls[1]:.3f}/{d_cls[2]:.3f}/{d_mean:.3f} | {score:.3f} | {fmt_hd(h_post):<20} | {fmt_hd(h_raw)}")

        if score > best_score:
            best_score = score
            torch.save(model.ema_teacher.teacher.state_dict(), os.path.join(cfg.save_dir, "BEST_MODEL.pt"))
            print(f"💾 Best Model Saved (Score: {score:.4f}, LR: {optimizer.param_groups[0]['lr']:.2e})")

        scheduler.step()

# =========================
# 5) EXECUTION
# =========================
model = model1.to(cfg.device)
run_training(model, train_loader, val_loader)
'''
#befor adding best model

In [ ]:
# فایل ها هم اصلاح شدن
'''
import os
import copy
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.metrics import HausdorffDistanceMetric
from monai.transforms import KeepLargestConnectedComponent
from monai.utils import set_determinism

# =========================
# 1) CONFIGURATION
# =========================
@dataclass
class CFG:
    save_dir: str = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_SOTA"
    seed: int = 42
    batch_size: int = 4
    num_workers: int = 4
    accum_iter: int = 2
    epochs: int = 100
    lr: float = 5e-4
    weight_decay: float = 1e-5
    warmup_epochs: int = 2
    roi_size: tuple = (128, 128, 128)
    sw_batch_size: int = 4
    sw_overlap: float = 0.5
    hd_every: int = 5
    deep_supervision: bool = True
    ds_weights: tuple = (0.5, 0.25)
    use_amp: bool = True
    ema_alpha: float = 0.99
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = CFG()
set_determinism(cfg.seed)

# =========================
# 2) EMA TEACHER & HYBRID LOSS
# =========================
class EMATeacher:
    def __init__(self, model, alpha=0.99):
        self.teacher = copy.deepcopy(model).eval()
        for p in self.teacher.parameters():
            p.requires_grad = False
        self.alpha = alpha

    @torch.no_grad()
    def update(self, student):
        for t_param, s_param in zip(self.teacher.parameters(), student.parameters()):
            t_param.data.mul_(self.alpha).add_(s_param.data, alpha=1.0 - self.alpha)

class HybridLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.ce_weights = torch.tensor([0.1, 1.0, 1.5, 2.0], device=device)
        self.dice_loss = DiceCELoss(include_background=False, to_onehot_y=True, softmax=True, batch=True)
        self.ce_loss = nn.CrossEntropyLoss(weight=self.ce_weights)

    def forward(self, logits, y):
        return self.dice_loss(logits, y) + self.ce_loss(logits, y.squeeze(1).long())

@torch.no_grad()
def to_regions(lbl):
    wt = (lbl > 0).float()
    tc = ((lbl == 1) | (lbl == 3)).float()
    et = (lbl == 3).float()
    return torch.stack([wt, tc, et], dim=1)

# =========================
# 3) VALIDATE FUNCTION
# =========================
@torch.no_grad()
def validate(model, loader, loss_fn, epoch):
    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model
    net.eval()
    compute_hd = (epoch % cfg.hd_every == 0)
    hd_m_raw = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    hd_m_post = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    post_filter = KeepLargestConnectedComponent(applied_labels=[1, 2, 3], is_onehot=False)

    dices, val_loss = [], 0.0
    for batch in loader:
        x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
        if y.ndim == 4: y = y.unsqueeze(1)
        y = y.long(); y[y == 4] = 3
        logits = sliding_window_inference(x, cfg.roi_size, cfg.sw_batch_size, lambda z: net(z)["logits"], overlap=cfg.sw_overlap)
        val_loss += loss_fn(logits, y).item()
        pred_raw = torch.argmax(logits, dim=1, keepdim=True)
        raw_reg, true_reg = to_regions(pred_raw.squeeze(1)), to_regions(y.squeeze(1))

        inter = (raw_reg * true_reg).sum((2,3,4))
        union = raw_reg.sum((2,3,4)) + true_reg.sum((2,3,4))
        dices.append(torch.nan_to_num(2.*inter/(union+1e-5), 0.).mean(0).cpu().numpy())

        if compute_hd:
            processed = torch.stack([post_filter(p.cpu().unsqueeze(0)).squeeze(0) for p in pred_raw.squeeze(1)]).to(cfg.device)
            post_reg = to_regions(processed)
            hd_m_raw(raw_reg, true_reg); hd_m_post(post_reg, true_reg)

    d_cls = np.mean(dices, axis=0)

    def safe_agg(metric):
        val = metric.aggregate()
        if isinstance(val, torch.Tensor): val = val.cpu().numpy()
        res = np.ones(3) * 100.0
        if np.isscalar(val): res[:] = val
        elif len(val) >= 3: res = val[:3]
        elif len(val) > 0: res[:len(val)] = val
        return np.nan_to_num(res, nan=100.0)

    h_raw = safe_agg(hd_m_raw) if compute_hd else np.array([-1.0]*3)
    h_post = safe_agg(hd_m_post) if compute_hd else np.array([-1.0]*3)
    return d_cls, h_raw, h_post, val_loss / len(loader)

# =========================
# 4) MAIN TRAINING LOOP (With Custom Resume Logic)
# =========================
import json

def run_training(model, train_loader, val_loader):
    os.makedirs(cfg.save_dir, exist_ok=True)

    # --- ذخیره فایل Config در ابتدای آموزش برای مراجعات بعدی ---
    with open(os.path.join(cfg.save_dir, 'config.json'), 'w') as f:
        json.dump(vars(cfg), f, indent=4, default=lambda x: str(x))

    loss_fn = HybridLoss(cfg.device)
    loss_fn.ce_weights = torch.tensor([0.1, 1.0, 1.5, 2.0], device=cfg.device)
    loss_fn.ce_loss = nn.CrossEntropyLoss(weight=loss_fn.ce_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    def lr_lambda(epoch):
        if epoch < cfg.warmup_epochs:
            return (epoch + 1) / cfg.warmup_epochs
        progress = (epoch - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs)
        return 0.5 * (1.0 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)
    model.ema_teacher = EMATeacher(model, cfg.ema_alpha)

    last_ckpt_path = os.path.join(cfg.save_dir, "LAST_CHECKPOINT.pt")
    best_model_path = os.path.join(cfg.save_dir, "BEST_MODEL.pt")
    best_ema_path = os.path.join(cfg.save_dir, "BEST_MODEL_EMA.pt") # فایل جدید
    csv_path = os.path.join(cfg.save_dir, "training_log_detailed.csv")

    start_epoch = 1
    best_score = -1.0

    # بازیابی (بدون تغییر در منطق لودینگ قبلی)
    if os.path.exists(last_ckpt_path):
        try:
            ckpt = torch.load(last_ckpt_path, map_location=cfg.device, weights_only=False)
            if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
                model.load_state_dict(ckpt["model_state_dict"])
                model.ema_teacher.teacher.load_state_dict(ckpt["ema_teacher_state_dict"])
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])
                scaler.load_state_dict(ckpt["scaler_state_dict"])
                start_epoch = ckpt.get("epoch", 0) + 1
                best_score = ckpt.get("best_score", -1.0)
                print(f"✅ Full System Restored. Resuming at Epoch {start_epoch}")
        except Exception as e:
            print(f"⚠️ Load failed ({e}). Starting fresh.")

    print("\n" + "=" * 115)
    print(f"{'Ep':<3} | {'vLoss':<7} | {'Dice (WT/TC/ET/Mean)':<24} | {'Score':<6} | {'HD95 POST (W/T/E/M)':<20} | {'HD95 RAW (W/T/E/M)'}")
    print("-" * 115)

    for ep in range(start_epoch, cfg.epochs + 1):
        # --- TRAIN PHASE ---
        model.train()
        train_loss = 0.0
        for i, batch in enumerate(train_loader):
            x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
            if y.ndim == 4: y = y.unsqueeze(1)
            y = y.long(); y[y == 4] = 3

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                out = model(x)
                loss = loss_fn(out["logits"], y)
                if cfg.deep_supervision and "aux1" in out:
                    loss += cfg.ds_weights[0] * loss_fn(out["aux1"], y)
                    loss += cfg.ds_weights[1] * loss_fn(out["aux2"], y)

            scaler.scale(loss/cfg.accum_iter).backward()
            if (i+1) % cfg.accum_iter == 0 or (i+1) == len(train_loader):
                scaler.step(optimizer); scaler.update()
                if hasattr(model, "ema_teacher"): model.ema_teacher.update(model)
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        # --- VALIDATION PHASE ---
        d_cls, h_raw, h_post, v_loss = validate(model, val_loader, loss_fn, ep)
        d_mean, score = np.mean(d_cls), (0.2*d_cls[0] + 0.3*d_cls[1] + 0.5*d_cls[2])
        current_lr = optimizer.param_groups[0]['lr']

        def fmt_hd(arr):
            if arr[0] < 0: return f"{'-/-/-/-':<20}"
            return f"{arr[0]:.1f}/{arr[1]:.1f}/{arr[2]:.1f}/{np.mean(arr):.1f}"

        print(f"{ep:02d} | {v_loss:.4f} | {d_cls[0]:.3f}/{d_cls[1]:.3f}/{d_cls[2]:.3f}/{d_mean:.3f} | {score:.4f} | {fmt_hd(h_post):<20} | {fmt_hd(h_raw)}")

        # --- DETAILED LOGGING (CSV) ---
        log_header = not os.path.exists(csv_path)
        with open(csv_path, "a", newline="") as f:
            writer = csv.writer(f)
            if log_header:
                writer.writerow([
                    "epoch", "lr", "train_loss", "val_loss",
                    "dice_WT", "dice_TC", "dice_ET", "dice_mean", "score",
                    "hd_post_WT", "hd_post_TC", "hd_post_ET", "hd_post_mean",
                    "hd_raw_WT", "hd_raw_TC", "hd_raw_ET", "hd_raw_mean"
                ])
            writer.writerow([
                ep, f"{current_lr:.2e}", f"{avg_train_loss:.4f}", f"{v_loss:.4f}",
                f"{d_cls[0]:.4f}", f"{d_cls[1]:.4f}", f"{d_cls[2]:.4f}", f"{d_mean:.4f}", f"{score:.4f}",
                f"{h_post[0]:.2f}", f"{h_post[1]:.2f}", f"{h_post[2]:.2f}", f"{np.mean(h_post):.2f}",
                f"{h_raw[0]:.2f}", f"{h_raw[1]:.2f}", f"{h_raw[2]:.2f}", f"{np.mean(h_raw):.2f}"
            ])

        # --- SAVING (FIXED AND DETAILED) ---
        checkpoint = {
            "epoch": ep,
            "best_score": max(best_score, score),
            "model_state_dict": model.state_dict(),
            "ema_teacher_state_dict": model.ema_teacher.teacher.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
        }
        torch.save(checkpoint, last_ckpt_path)

        if score > best_score:
            best_score = score
            # ذخیره وزن‌های مدل شاگرد
            torch.save(model.state_dict(), best_model_path)
            # ذخیره وزن‌های مدل معلم (EMA) - حیاتی برای استنتاج نهایی
            torch.save(model.ema_teacher.teacher.state_dict(), best_ema_path)
            print(f"💾 Best Models Updated (Score: {score:.4f})")

        scheduler.step()
# =========================
# 5) EXECUTION
# =========================
model = model1.to(cfg.device)
run_training(model, train_loader, val_loader)
'''
'''
✅ Full System Restored. Resuming at Epoch 2

===================================================================================================================
Ep  | vLoss   | Dice (WT/TC/ET/Mean)     | Score  | HD95 POST (W/T/E/M)  | HD95 RAW (W/T/E/M)
-------------------------------------------------------------------------------------------------------------------
02 | 6.1108 | 0.042/0.016/0.010/0.023 | 0.0181 | -/-/-/-              | -/-/-/-
03 | 5.4389 | 0.042/0.016/0.012/0.023 | 0.0193 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.0193)
04 | 5.0440 | 0.042/0.016/0.013/0.023 | 0.0193 | -/-/-/-              | -/-/-/-
/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:221: FutureWarning: monai.metrics.utils get_mask_edges:always_return_as_numpy: Argument `always_return_as_numpy` has been deprecated since version 1.5.0. It will be removed in version 1.7.0. The option is removed and the return type will always be equal to the input type.
  warn_deprecated(argname, msg, warning_category)
05 | 4.8892 | 0.043/0.016/0.014/0.024 | 0.0202 | 113.0/100.0/100.0/104.3 | 113.0/100.0/100.0/104.3
💾 Best Models Updated (Score: 0.0202)
06 | 4.7257 | 0.043/0.016/0.014/0.024 | 0.0205 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.0205)
07 | 4.4369 | 0.043/0.017/0.014/0.025 | 0.0208 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.0208)
08 | 4.1998 | 0.043/0.018/0.015/0.025 | 0.0215 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.0215)
09 | 3.9595 | 0.046/0.020/0.018/0.028 | 0.0239 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.0239)
10 | 3.7708 | 0.047/0.021/0.018/0.029 | 0.0249 | 114.0/100.0/100.0/104.7 | 113.7/100.0/100.0/104.6
💾 Best Models Updated (Score: 0.0249)
11 | 3.6189 | 0.048/0.022/0.019/0.030 | 0.0256 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.0256)
12 | 3.4822 | 0.052/0.024/0.020/0.032 | 0.0279 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.0279)
13 | 3.3820 | 0.058/0.027/0.022/0.036 | 0.0306 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.0306)
14 | 3.3078 | 0.065/0.029/0.024/0.039 | 0.0338 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.0338)
15 | 3.2632 | 0.073/0.033/0.027/0.044 | 0.0378 | 116.7/100.0/100.0/105.6 | 116.6/100.0/100.0/105.5
💾 Best Models Updated (Score: 0.0378)

'''

In [ ]:
# اضافه کردن lookahead
# فایل ها هم اصلاح شدن
'''
import os
import copy
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.metrics import HausdorffDistanceMetric
from monai.transforms import KeepLargestConnectedComponent
from monai.utils import set_determinism

# =========================
# 1) CONFIGURATION
# =========================
@dataclass
class CFG:
    # مسیر ذخیره‌سازی (نام پوشه را برای نظم بیشتر کمی تغییر دادم)
    save_dir: str = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead"
    seed: int = 42

    # تنظیمات Batch و Accumulation برای 1250 دیتا
    # معادل بچ سایز موثر 16 (2 * 8) برای پایداری در گرادیان
    batch_size: int = 2
    num_workers: int = 4
    accum_iter: int = 8

    epochs: int = 100

    # نرخ یادگیری و وارم‌آپ (برای Lookahead و پنجره بزرگ، 5 اپوک وارم‌آپ ایمن‌تر است)
    lr: float = 3e-4
    weight_decay: float = 3e-5
    warmup_epochs: int = 5

    # سایز پنجره (طبق تصمیم جدید روی 160)
    roi_size: tuple = (160, 160, 160)
    patch_size: Tuple[int,int,int] = (160, 160, 160)

    # تنظیمات اینفرنس (Inference)
    sw_batch_size: int = 2
    sw_overlap: float = 0.6  # افزایش برای دقت SOTA در مرزهای تومور
    hd_every: int = 5        # محاسبه Hausdorff هر 5 اپوک (چون سنگین است)

    # تنظیمات معماری و نظارت عمیق
    deep_supervision: bool = True
    ds_weights: tuple = (0.5, 0.25)

    # پایداری عددی و معلم (EMA)
    use_amp: bool = True
    ema_alpha: float = 0.99
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = CFG()
set_determinism(cfg.seed)

# =========================
# 2) EMA TEACHER & HYBRID LOSS
# =========================
class EMATeacher:
    def __init__(self, model, alpha=0.99):
        self.teacher = copy.deepcopy(model).eval()
        for p in self.teacher.parameters():
            p.requires_grad = False
        self.alpha = alpha

    @torch.no_grad()
    def update(self, student):
        for t_param, s_param in zip(self.teacher.parameters(), student.parameters()):
            t_param.data.mul_(self.alpha).add_(s_param.data, alpha=1.0 - self.alpha)

class HybridLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.ce_weights = torch.tensor([0.2, 1.0, 1.5, 2.0], device=device)
        self.dice_loss = DiceCELoss(include_background=False, to_onehot_y=True, softmax=True, batch=True)
        self.ce_loss = nn.CrossEntropyLoss(weight=self.ce_weights)

    def forward(self, logits, y):
        return self.dice_loss(logits, y) + self.ce_loss(logits, y.squeeze(1).long())

@torch.no_grad()
def to_regions(lbl):
    wt = (lbl > 0).float()
    tc = ((lbl == 1) | (lbl == 3)).float()
    et = (lbl == 3).float()
    return torch.stack([wt, tc, et], dim=1)

# =========================
# 3) VALIDATE FUNCTION
# =========================
@torch.no_grad()
def validate(model, loader, loss_fn, epoch):
    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model
    net.eval()
    compute_hd = (epoch % cfg.hd_every == 0)
    hd_m_raw = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    hd_m_post = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    post_filter = KeepLargestConnectedComponent(applied_labels=[1, 2, 3], is_onehot=False)

    dices, val_loss = [], 0.0
    for batch in loader:
        x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
        if y.ndim == 4: y = y.unsqueeze(1)
        y = y.long(); y[y == 4] = 3
        logits = sliding_window_inference(x, cfg.roi_size, cfg.sw_batch_size, lambda z: net(z)["logits"], overlap=cfg.sw_overlap)
        val_loss += loss_fn(logits, y).item()
        pred_raw = torch.argmax(logits, dim=1, keepdim=True)
        raw_reg, true_reg = to_regions(pred_raw.squeeze(1)), to_regions(y.squeeze(1))

        inter = (raw_reg * true_reg).sum((2,3,4))
        union = raw_reg.sum((2,3,4)) + true_reg.sum((2,3,4))
        dices.append(torch.nan_to_num(2.*inter/(union+1e-5), 0.).mean(0).cpu().numpy())

        if compute_hd:
            processed = torch.stack([post_filter(p.cpu().unsqueeze(0)).squeeze(0) for p in pred_raw.squeeze(1)]).to(cfg.device)
            post_reg = to_regions(processed)
            hd_m_raw(raw_reg, true_reg); hd_m_post(post_reg, true_reg)

    d_cls = np.mean(dices, axis=0)

    def safe_agg(metric):
        val = metric.aggregate()
        if isinstance(val, torch.Tensor): val = val.cpu().numpy()
        res = np.ones(3) * 100.0
        if np.isscalar(val): res[:] = val
        elif len(val) >= 3: res = val[:3]
        elif len(val) > 0: res[:len(val)] = val
        return np.nan_to_num(res, nan=100.0)

    h_raw = safe_agg(hd_m_raw) if compute_hd else np.array([-1.0]*3)
    h_post = safe_agg(hd_m_post) if compute_hd else np.array([-1.0]*3)
    return d_cls, h_raw, h_post, val_loss / len(loader)

# =========================
# 4) MAIN TRAINING LOOP (With Custom Resume Logic)
# =========================
import json

import torch
import torch.nn as nn
import numpy as np
import os
import json
import csv

# =========================================================
# 1. کلاس Lookahead برای پایداری SOTA
# =========================================================
class Lookahead(torch.optim.Optimizer):
    def __init__(self, optimizer, k=5, alpha=0.5):
        self.optimizer = optimizer
        self.k = k
        self.alpha = alpha
        self.param_groups = self.optimizer.param_groups
        self.state = self.optimizer.state
        self.defaults = self.optimizer.defaults  # اضافه شدن برای حل خطای AttributeError
        self.slow_weights = [[p.data.clone().detach() for p in group['params']]
                             for group in self.param_groups]

    def step(self, closure=None):
        loss = self.optimizer.step(closure)
        for group, slow_weights in zip(self.param_groups, self.slow_weights):
            for p, slow_p in zip(group['params'], slow_weights):
                if p not in self.optimizer.state or 'step' not in self.optimizer.state[p]:
                    continue
                # هندل کردن تفاوت فرمت‌های مختلف ذخیره Step در PyTorch
                step = self.optimizer.state[p]['step']
                if isinstance(step, torch.Tensor):
                    step = step.item()

                if step % self.k == 0:
                    slow_p.add_(p.data - slow_p, alpha=self.alpha)
                    p.data.copy_(slow_p)
        return loss

    # اضافه کردن متد zero_grad برای هدایت مستقیم به اپتیمایزر اصلی
    def zero_grad(self, set_to_none=True):
        self.optimizer.zero_grad(set_to_none=set_to_none)

    def load_state_dict(self, state_dict):
        self.optimizer.load_state_dict(state_dict)

    def state_dict(self):
        return self.optimizer.state_dict()
# =========================================================
# 2. تابع اصلاح شده run_training
# =========================================================

def run_training(model, train_loader, val_loader):
    os.makedirs(cfg.save_dir, exist_ok=True)

    with open(os.path.join(cfg.save_dir, 'config.json'), 'w') as f:
        json.dump(vars(cfg), f, indent=4, default=lambda x: str(x))

    loss_fn = HybridLoss(cfg.device)
    loss_fn.ce_weights = torch.tensor([0.2, 1.0, 1.2, 2.0], device=cfg.device)
    loss_fn.ce_loss = nn.CrossEntropyLoss(weight=loss_fn.ce_weights)

    # --- بخش اصلاح شده: Optimizer با Lookahead و Betas جدید ---
    base_optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        betas=(0.95, 0.999), # افزایش بتا برای Momentum بیشتر
        weight_decay=cfg.weight_decay
    )
    optimizer = Lookahead(base_optimizer, k=5, alpha=0.5)
    # -------------------------------------------------------

    def lr_lambda(epoch):
        if epoch < cfg.warmup_epochs:
            return (epoch + 1) / cfg.warmup_epochs
        progress = (epoch - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs)
        return 0.5 * (1.0 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(base_optimizer, lr_lambda=lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)
    model.ema_teacher = EMATeacher(model, cfg.ema_alpha)

    last_ckpt_path = os.path.join(cfg.save_dir, "LAST_CHECKPOINT.pt")
    best_model_path = os.path.join(cfg.save_dir, "BEST_MODEL.pt")
    best_ema_path = os.path.join(cfg.save_dir, "BEST_MODEL_EMA.pt")
    csv_path = os.path.join(cfg.save_dir, "training_log_detailed.csv")

    start_epoch = 1
    best_score = -1.0

    # --- بخش اصلاح شده: Resume Logic برای هماهنگی با Lookahead ---
    if os.path.exists(last_ckpt_path):
        try:
            ckpt = torch.load(last_ckpt_path, map_location=cfg.device, weights_only=False)
            if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
                model.load_state_dict(ckpt["model_state_dict"])
                model.ema_teacher.teacher.load_state_dict(ckpt["ema_teacher_state_dict"])

                # لود کردن استیت روی اپتیمایزر اصلی (Base Optimizer)
                optimizer.optimizer.load_state_dict(ckpt["optimizer_state_dict"])

                scheduler.load_state_dict(ckpt["scheduler_state_dict"])
                scaler.load_state_dict(ckpt["scaler_state_dict"])
                start_epoch = ckpt.get("epoch", 0) + 1
                best_score = ckpt.get("best_score", -1.0)
                print(f"✅ Full System Restored. Resuming at Epoch {start_epoch}")
        except Exception as e:
            print(f"⚠️ Load failed ({e}). Starting fresh.")
    # -------------------------------------------------------

    print("\n" + "=" * 115)
    print(f"{'Ep':<3} | {'vLoss':<7} | {'Dice (WT/TC/ET/Mean)':<24} | {'Score':<6} | {'HD95 POST (W/T/E/M)':<20} | {'HD95 RAW (W/T/E/M)'}")
    print("-" * 115)

    for ep in range(start_epoch, cfg.epochs + 1):
        model.train()
        train_loss = 0.0
        for i, batch in enumerate(train_loader):
            x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
            if y.ndim == 4: y = y.unsqueeze(1)
            y = y.long(); y[y == 4] = 3

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                out = model(x)
                loss = loss_fn(out["logits"], y)
                if cfg.deep_supervision and "aux1" in out:
                    loss += cfg.ds_weights[0] * loss_fn(out["aux1"], y)
                    loss += cfg.ds_weights[1] * loss_fn(out["aux2"], y)

            scaler.scale(loss/cfg.accum_iter).backward()
            if (i+1) % cfg.accum_iter == 0 or (i+1) == len(train_loader):
                scaler.step(optimizer) # اینجا مستقیماً از wrapper استفاده می‌شود
                scaler.update()
                if hasattr(model, "ema_teacher"): model.ema_teacher.update(model)
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        d_cls, h_raw, h_post, v_loss = validate(model, val_loader, loss_fn, ep)
        d_mean, score = np.mean(d_cls), (0.2*d_cls[0] + 0.3*d_cls[1] + 0.5*d_cls[2])
        current_lr = base_optimizer.param_groups[0]['lr'] # استفاده از base_optimizer برای گرفتن LR

        def fmt_hd(arr):
            if arr[0] < 0: return f"{'-/-/-/-':<20}"
            return f"{arr[0]:.1f}/{arr[1]:.1f}/{arr[2]:.1f}/{np.mean(arr):.1f}"

        print(f"{ep:02d} | {v_loss:.4f} | {d_cls[0]:.3f}/{d_cls[1]:.3f}/{d_cls[2]:.3f}/{d_mean:.3f} | {score:.4f} | {fmt_hd(h_post):<20} | {fmt_hd(h_raw)}")

        log_header = not os.path.exists(csv_path)
        with open(csv_path, "a", newline="") as f:
            writer = csv.writer(f)
            if log_header:
                writer.writerow(["epoch", "lr", "train_loss", "val_loss", "dice_WT", "dice_TC", "dice_ET", "dice_mean", "score", "hd_post_WT", "hd_post_TC", "hd_post_ET", "hd_post_mean", "hd_raw_WT", "hd_raw_TC", "hd_raw_ET", "hd_raw_mean"])
            writer.writerow([ep, f"{current_lr:.2e}", f"{avg_train_loss:.4f}", f"{v_loss:.4f}", f"{d_cls[0]:.4f}", f"{d_cls[1]:.4f}", f"{d_cls[2]:.4f}", f"{d_mean:.4f}", f"{score:.4f}", f"{h_post[0]:.2f}", f"{h_post[1]:.2f}", f"{h_post[2]:.2f}", f"{np.mean(h_post):.2f}", f"{h_raw[0]:.2f}", f"{h_raw[1]:.2f}", f"{h_raw[2]:.2f}", f"{np.mean(h_raw):.2f}"])

        checkpoint = {
            "epoch": ep,
            "best_score": max(best_score, score),
            "model_state_dict": model.state_dict(),
            "ema_teacher_state_dict": model.ema_teacher.teacher.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(), # متد state_dict در کلاس بالا تعریف شده
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
        }
        torch.save(checkpoint, last_ckpt_path)

        if score > best_score:
            best_score = score
            torch.save(model.state_dict(), best_model_path)
            torch.save(model.ema_teacher.teacher.state_dict(), best_ema_path)
            print(f"💾 Best Models Updated (Score: {score:.4f})")

        scheduler.step()
# =========================
# 5) EXECUTION
# =========================
model = model1.to(cfg.device)
run_training(model, train_loader, val_loader)
'''

In [ ]:
# هایپر پارامتر ها طبق مقالات پایه
'''
import os
import copy
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.metrics import HausdorffDistanceMetric
from monai.transforms import KeepLargestConnectedComponent
from monai.utils import set_determinism

# =========================
# 1) CONFIGURATION
# =========================
@dataclass
class CFG:
    # مسیر ذخیره‌سازی (نام پوشه را برای نظم بیشتر کمی تغییر دادم)
    save_dir: str = "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead"
    seed: int = 42

    # تنظیمات Batch و Accumulation برای 1250 دیتا
    # معادل بچ سایز موثر 16 (2 * 8) برای پایداری در گرادیان
    batch_size: int = 2
    num_workers: int = 4
    accum_iter: int = 4

    epochs: int = 100

    # نرخ یادگیری و وارم‌آپ (برای Lookahead و پنجره بزرگ، 5 اپوک وارم‌آپ ایمن‌تر است)
    lr: float = 1e-4
    weight_decay: float = 3e-5
    warmup_epochs: int = 5

    # سایز پنجره (طبق تصمیم جدید روی 160)
    roi_size: tuple = (160, 160, 160)
    patch_size: Tuple[int,int,int] = (160, 160, 160)

    # تنظیمات اینفرنس (Inference)
    sw_batch_size: int = 2
    sw_overlap: float = 0.6  # افزایش برای دقت SOTA در مرزهای تومور
    hd_every: int = 5        # محاسبه Hausdorff هر 5 اپوک (چون سنگین است)

    # تنظیمات معماری و نظارت عمیق
    deep_supervision: bool = True
    ds_weights: tuple = (0.5, 0.25)

    # پایداری عددی و معلم (EMA)
    use_amp: bool = True
    ema_alpha: float = 0.99
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = CFG()
set_determinism(cfg.seed)

# =========================
# 2) EMA TEACHER & HYBRID LOSS
# =========================
class EMATeacher:
    def __init__(self, model, alpha=0.99):
        self.teacher = copy.deepcopy(model).eval()
        for p in self.teacher.parameters():
            p.requires_grad = False
        self.alpha = alpha

    @torch.no_grad()
    def update(self, student):
        for t_param, s_param in zip(self.teacher.parameters(), student.parameters()):
            t_param.data.mul_(self.alpha).add_(s_param.data, alpha=1.0 - self.alpha)

class HybridLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.ce_weights = torch.tensor([0.2, 1.0, 1.5, 2.0], device=device)
        self.dice_loss = DiceCELoss(include_background=False, to_onehot_y=True, softmax=True, batch=True)
        self.ce_loss = nn.CrossEntropyLoss(weight=self.ce_weights)

    def forward(self, logits, y):
        return self.dice_loss(logits, y) + self.ce_loss(logits, y.squeeze(1).long())

@torch.no_grad()
def to_regions(lbl):
    wt = (lbl > 0).float()
    tc = ((lbl == 1) | (lbl == 3)).float()
    et = (lbl == 3).float()
    return torch.stack([wt, tc, et], dim=1)

# =========================
# 3) VALIDATE FUNCTION
# =========================
@torch.no_grad()
def validate(model, loader, loss_fn, epoch):
    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model
    net.eval()
    compute_hd = (epoch % cfg.hd_every == 0)
    hd_m_raw = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    hd_m_post = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    post_filter = KeepLargestConnectedComponent(applied_labels=[1, 2, 3], is_onehot=False)

    dices, val_loss = [], 0.0
    for batch in loader:
        x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
        if y.ndim == 4: y = y.unsqueeze(1)
        y = y.long(); y[y == 4] = 3
        logits = sliding_window_inference(x, cfg.roi_size, cfg.sw_batch_size, lambda z: net(z)["logits"], overlap=cfg.sw_overlap)
        val_loss += loss_fn(logits, y).item()
        pred_raw = torch.argmax(logits, dim=1, keepdim=True)
        raw_reg, true_reg = to_regions(pred_raw.squeeze(1)), to_regions(y.squeeze(1))

        inter = (raw_reg * true_reg).sum((2,3,4))
        union = raw_reg.sum((2,3,4)) + true_reg.sum((2,3,4))
        dices.append(torch.nan_to_num(2.*inter/(union+1e-5), 0.).mean(0).cpu().numpy())

        if compute_hd:
            processed = torch.stack([post_filter(p.cpu().unsqueeze(0)).squeeze(0) for p in pred_raw.squeeze(1)]).to(cfg.device)
            post_reg = to_regions(processed)
            hd_m_raw(raw_reg, true_reg); hd_m_post(post_reg, true_reg)

    d_cls = np.mean(dices, axis=0)

    def safe_agg(metric):
        val = metric.aggregate()
        if isinstance(val, torch.Tensor): val = val.cpu().numpy()
        res = np.ones(3) * 100.0
        if np.isscalar(val): res[:] = val
        elif len(val) >= 3: res = val[:3]
        elif len(val) > 0: res[:len(val)] = val
        return np.nan_to_num(res, nan=100.0)

    h_raw = safe_agg(hd_m_raw) if compute_hd else np.array([-1.0]*3)
    h_post = safe_agg(hd_m_post) if compute_hd else np.array([-1.0]*3)
    return d_cls, h_raw, h_post, val_loss / len(loader)

# =========================
# 4) MAIN TRAINING LOOP (With Custom Resume Logic)
# =========================
import json

import torch
import torch.nn as nn
import numpy as np
import os
import json
import csv

# =========================================================
# 1. کلاس Lookahead برای پایداری SOTA
# =========================================================
class Lookahead(torch.optim.Optimizer):
    def __init__(self, optimizer, k=5, alpha=0.5):
        self.optimizer = optimizer
        self.k = k
        self.alpha = alpha
        self.param_groups = self.optimizer.param_groups
        self.state = self.optimizer.state
        self.defaults = self.optimizer.defaults

        # تعریف یک شمارشگر داخلی برای استقلال از اپتیمایزر پایه
        self.lookahead_step = 0

        self.slow_weights = [[p.data.clone().detach() for p in group['params']]
                             for group in self.param_groups]

    def step(self, closure=None):
        # انجام یک گام آپدیت توسط اپتیمایزر اصلی (AdamW)
        loss = self.optimizer.step(closure)

        # اضافه کردن به شمارشگر اختصاصی خودمان
        self.lookahead_step += 1

        # بررسی شرط همگام‌سازی وزن‌های کند و تند
        if self.lookahead_step % self.k == 0:
            for group, slow_weights in zip(self.param_groups, self.slow_weights):
                for p, slow_p in zip(group['params'], slow_weights):
                    if p.grad is None:
                        continue

                    # میانگین‌گیری وزنی بین وزن کند و تند (اصطلاحاً Interpolation)
                    slow_p.add_(p.data - slow_p, alpha=self.alpha)
                    # کپی کردن نتیجه در وزن اصلی مدل برای ادامه مسیر
                    p.data.copy_(slow_p)

        return loss

    def zero_grad(self, set_to_none=True):
        self.optimizer.zero_grad(set_to_none=set_to_none)

    def state_dict(self):
        # ذخیره وضعیت شامل شمارشگر داخلی ما
        fast_state_dict = self.optimizer.state_dict()
        fast_state_dict['lookahead_step'] = self.lookahead_step
        return fast_state_dict

    def load_state_dict(self, state_dict):
        # بازیابی شمارشگر در صورت وجود در فایل
        self.lookahead_step = state_dict.pop('lookahead_step', 0)
        self.optimizer.load_state_dict(state_dict)
# =========================================================
# 2. تابع اصلاح شده run_training
# =========================================================
def run_training(model, train_loader, val_loader):
    os.makedirs(cfg.save_dir, exist_ok=True)

    # ذخیره کانفیگ برای ارجاع در آینده
    with open(os.path.join(cfg.save_dir, 'config.json'), 'w') as f:
        json.dump(vars(cfg), f, indent=4, default=lambda x: str(x))

    # --- تنظیمات Loss بر اساس مقالات SOTA ---
    loss_fn = HybridLoss(cfg.device)
    loss_fn.ce_weights = torch.tensor([0.2, 1.0, 1.0, 2.0], device=cfg.device)
    loss_fn.ce_loss = nn.CrossEntropyLoss(weight=loss_fn.ce_weights)

    # --- Optimizer ---
    base_optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        betas=(0.9, 0.999),
        weight_decay=cfg.weight_decay
    )
    optimizer = Lookahead(base_optimizer, k=5, alpha=0.5)

    # --- Scheduler ---
    def lr_lambda(epoch):
        current_ep = epoch + 1
        if current_ep <= cfg.warmup_epochs:
            return current_ep / cfg.warmup_epochs
        progress = (current_ep - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs)
        return 0.5 * (1.0 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(base_optimizer, lr_lambda=lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)
    model.ema_teacher = EMATeacher(model, cfg.ema_alpha)

    # مسیرهای فایل
    last_ckpt_path = os.path.join(cfg.save_dir, "LAST_CHECKPOINT.pt")
    best_model_path = os.path.join(cfg.save_dir, "BEST_MODEL.pt")
    best_ema_path = os.path.join(cfg.save_dir, "BEST_MODEL_EMA.pt")
    csv_path = os.path.join(cfg.save_dir, "training_log_detailed.csv")

    start_epoch = 1
    best_score = -1.0

    # =========================================================
    # اصلاح شده: Resume Logic (اولویت با Last، در غیر این صورت Best)
    # =========================================================
    ckpt_to_load = None
    if os.path.exists(last_ckpt_path):
        ckpt_to_load = last_ckpt_path
        print(f"🔄 Found Last Checkpoint. Resuming full state...")
    elif os.path.exists(best_model_path):
        ckpt_to_load = best_model_path
        print(f"⭐ Last Checkpoint not found. Loading weights from Best Model...")

    if ckpt_to_load:
        try:
            ckpt = torch.load(ckpt_to_load, map_location=cfg.device, weights_only=False)

            if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
                # اگر فایل چک‌پوینت کامل باشد (معمولاً LAST_CHECKPOINT)
                model.load_state_dict(ckpt["model_state_dict"])
                model.ema_teacher.teacher.load_state_dict(ckpt["ema_teacher_state_dict"])
                optimizer.optimizer.load_state_dict(ckpt["optimizer_state_dict"])
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])
                scaler.load_state_dict(ckpt["scaler_state_dict"])
                start_epoch = ckpt.get("epoch", 0) + 1
                best_score = ckpt.get("best_score", -1.0)
            else:
                # اگر فایل فقط وزن‌های مدل باشد (معمولاً BEST_MODEL)
                model.load_state_dict(ckpt)
                if os.path.exists(best_ema_path):
                    ema_weights = torch.load(best_ema_path, map_location=cfg.device)
                    model.ema_teacher.teacher.load_state_dict(ema_weights)
                    print("✅ EMA Teacher weights loaded from BEST_MODEL_EMA.")
                else:
                    model.ema_teacher.teacher.load_state_dict(model.state_dict())

            print(f"✅ Recovery Successful. Starting from Epoch {start_epoch}")
        except Exception as e:
            print(f"⚠️ Load failed ({e}). Starting fresh.")

    # =========================================================
    # ادامه کد بدون تغییر
    # =========================================================
    print("\n" + "=" * 115)
    print(f"{'Ep':<3} | {'vLoss':<7} | {'Dice (WT/TC/ET/Mean)':<24} | {'Score':<6} | {'HD95 POST (W/T/E/M)':<20} | {'HD95 RAW (W/T/E/M)'}")
    print("-" * 115)

    for ep in range(start_epoch, cfg.epochs + 1):
        model.train()
        train_loss = 0.0

        for i, batch in enumerate(train_loader):
            x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
            if y.ndim == 4: y = y.unsqueeze(1)
            y = y.long(); y[y == 4] = 3

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                out = model(x)
                loss = loss_fn(out["logits"], y)
                if cfg.deep_supervision and "aux1" in out:
                    loss += cfg.ds_weights[0] * loss_fn(out["aux1"], y)
                    loss += cfg.ds_weights[1] * loss_fn(out["aux2"], y)

            scaler.scale(loss/cfg.accum_iter).backward()

            if (i+1) % cfg.accum_iter == 0 or (i+1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                if hasattr(model, "ema_teacher"):
                    model.ema_teacher.update(model)

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        # Validation
        d_cls, h_raw, h_post, v_loss = validate(model, val_loader, loss_fn, ep)
        d_mean = np.mean(d_cls)
        score = (0.2 * d_cls[0] + 0.3 * d_cls[1] + 0.5 * d_cls[2])
        current_lr = base_optimizer.param_groups[0]['lr']

        def fmt_hd(arr):
            if arr[0] < 0: return f"{'-/-/-/-':<20}"
            return f"{arr[0]:.1f}/{arr[1]:.1f}/{arr[2]:.1f}/{np.mean(arr):.1f}"

        print(f"{ep:02d} | {v_loss:.4f} | {d_cls[0]:.3f}/{d_cls[1]:.3f}/{d_cls[2]:.3f}/{d_mean:.3f} | {score:.4f} | {fmt_hd(h_post):<20} | {fmt_hd(h_raw)}")

        # Logging
        log_header = not os.path.exists(csv_path)
        with open(csv_path, "a", newline="") as f:
            writer = csv.writer(f)
            if log_header:
                writer.writerow(["epoch", "lr", "train_loss", "val_loss", "dice_WT", "dice_TC", "dice_ET", "dice_mean", "score", "hd_post_WT", "hd_post_TC", "hd_post_ET", "hd_post_mean", "hd_raw_WT", "hd_raw_TC", "hd_raw_ET", "hd_raw_mean"])
            writer.writerow([ep, f"{current_lr:.2e}", f"{avg_train_loss:.4f}", f"{v_loss:.4f}", f"{d_cls[0]:.4f}", f"{d_cls[1]:.4f}", f"{d_cls[2]:.4f}", f"{d_mean:.4f}", f"{score:.4f}", f"{h_post[0]:.2f}", f"{h_post[1]:.2f}", f"{h_post[2]:.2f}", f"{np.mean(h_post):.2f}", f"{h_raw[0]:.2f}", f"{h_raw[1]:.2f}", f"{h_raw[2]:.2f}", f"{np.mean(h_raw):.2f}"])

        # Saving Checkpoints
        checkpoint = {
            "epoch": ep,
            "best_score": max(best_score, score),
            "model_state_dict": model.state_dict(),
            "ema_teacher_state_dict": model.ema_teacher.teacher.state_dict(),
            "optimizer_state_dict": optimizer.optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
        }
        torch.save(checkpoint, last_ckpt_path)

        if score > best_score:
            best_score = score
            torch.save(model.state_dict(), best_model_path)
            torch.save(model.ema_teacher.teacher.state_dict(), best_ema_path)
            print(f"💾 Best Models Updated (Score: {score:.4f})")

        scheduler.step()
# =========================
# 5) EXECUTION
# =========================
model = model1.to(cfg.device)
#run_training(model, train_loader, val_loader)
'''

# مشکل در لود کردن بهترین مدل


In [ ]:
import os
import torch
import gc

# آزاد سازی کش‌های قبلی
gc.collect()
torch.cuda.empty_cache()

# ترفند طلایی برای پچ‌های بزرگ 160x160x160
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128,expandable_segments:True"

print("✅ GPU Memory ready for heavy 160x160 patches.")

In [ ]:
'''

# هایپر پارامتر ها طبق مقالات پایه
import os
import copy
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Tuple # اضافه شده برای رفع خطای احتمالی تایپ
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.metrics import HausdorffDistanceMetric
from monai.transforms import KeepLargestConnectedComponent
from monai.utils import set_determinism

# =========================
# 1) CONFIGURATION
# =========================


#cfg = CFG()
set_determinism(cfg.seed)

# =========================
# 2) EMA TEACHER & HYBRID LOSS
# =========================
class EMATeacher:
    def __init__(self, model, alpha=0.99):
        self.teacher = copy.deepcopy(model).eval()
        for p in self.teacher.parameters():
            p.requires_grad = False
        self.alpha = alpha

    @torch.no_grad()
    def update(self, student):
        for t_param, s_param in zip(self.teacher.parameters(), student.parameters()):
            t_param.data.mul_(self.alpha).add_(s_param.data, alpha=1.0 - self.alpha)

class HybridLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        #self.ce_weights = torch.tensor([0.2, 1.0, 1.5, 2.0], device=device)
        #self.ce_weights = torch.tensor([0.5, 2.0, 1.5, 4.0], device=device)

        self.dice_loss = DiceCELoss(include_background=False, to_onehot_y=True, softmax=True, batch=True)
        self.ce_loss = nn.CrossEntropyLoss(weight=self.ce_weights)

    def forward(self, logits, y):
        return self.dice_loss(logits, y) + self.ce_loss(logits, y.squeeze(1).long())

@torch.no_grad()
def to_regions(lbl):
    wt = (lbl > 0).float()
    tc = ((lbl == 1) | (lbl == 3)).float()
    et = (lbl == 3).float()
    return torch.stack([wt, tc, et], dim=1)

# =========================
# 3) VALIDATE FUNCTION
# =========================
@torch.no_grad()
def validate(model, loader, loss_fn, epoch):
    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model
    net.eval()
    compute_hd = (epoch % cfg.hd_every == 0)
    hd_m_raw = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    hd_m_post = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    post_filter = KeepLargestConnectedComponent(applied_labels=[1, 2, 3], is_onehot=False)

    dices, val_loss = [], 0.0
    for batch in loader:
        x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
        if y.ndim == 4: y = y.unsqueeze(1)
        y = y.long(); y[y == 4] = 3
        logits = sliding_window_inference(x, cfg.roi_size, cfg.sw_batch_size, lambda z: net(z)["logits"], overlap=cfg.sw_overlap)
        val_loss += loss_fn(logits, y).item()
        pred_raw = torch.argmax(logits, dim=1, keepdim=True)
        raw_reg, true_reg = to_regions(pred_raw.squeeze(1)), to_regions(y.squeeze(1))

        inter = (raw_reg * true_reg).sum((2,3,4))
        union = raw_reg.sum((2,3,4)) + true_reg.sum((2,3,4))
        dices.append(torch.nan_to_num(2.*inter/(union+1e-5), 0.).mean(0).cpu().numpy())

        if compute_hd:
            processed = torch.stack([post_filter(p.cpu().unsqueeze(0)).squeeze(0) for p in pred_raw.squeeze(1)]).to(cfg.device)
            post_reg = to_regions(processed)
            hd_m_raw(raw_reg, true_reg); hd_m_post(post_reg, true_reg)

    d_cls = np.mean(dices, axis=0)

    def safe_agg(metric):
        val = metric.aggregate()
        if isinstance(val, torch.Tensor): val = val.cpu().numpy()
        res = np.ones(3) * 100.0
        if np.isscalar(val): res[:] = val
        elif len(val) >= 3: res = val[:3]
        elif len(val) > 0: res[:len(val)] = val
        return np.nan_to_num(res, nan=100.0)

    h_raw = safe_agg(hd_m_raw) if compute_hd else np.array([-1.0]*3)
    h_post = safe_agg(hd_m_post) if compute_hd else np.array([-1.0]*3)
    return d_cls, h_raw, h_post, val_loss / len(loader)

# =========================================================
# 4) اصلاح شده: کلاس Lookahead با شمارشگر مستقل
# =========================================================
class Lookahead(torch.optim.Optimizer):
    def __init__(self, optimizer, k=5, alpha=0.5):
        self.optimizer = optimizer
        self.k = k
        self.alpha = alpha
        self.param_groups = self.optimizer.param_groups
        self.state = self.optimizer.state
        self.defaults = self.optimizer.defaults
        self.lookahead_step = 0 # شمارشگر اختصاصی
        self.slow_weights = [[p.data.clone().detach() for p in group['params']]
                             for group in self.param_groups]

    def step(self, closure=None):
        loss = self.optimizer.step(closure)
        self.lookahead_step += 1

        if self.lookahead_step % self.k == 0:
            for group, slow_weights in zip(self.param_groups, self.slow_weights):
                for p, slow_p in zip(group['params'], slow_weights):
                    if p.grad is None: continue
                    slow_p.add_(p.data - slow_p, alpha=self.alpha)
                    p.data.copy_(slow_p)
        return loss

    def zero_grad(self, set_to_none=True):
        self.optimizer.zero_grad(set_to_none=set_to_none)

    def state_dict(self):
        sd = self.optimizer.state_dict()
        sd['lookahead_step'] = self.lookahead_step
        return sd

    def load_state_dict(self, state_dict):
        self.lookahead_step = state_dict.pop('lookahead_step', 0)
        self.optimizer.load_state_dict(state_dict)

# =========================================================
# 5) تابع اصلی آموزش با Resume Logic هوشمند
# =========================================================
import json

def run_training(model, train_loader, val_loader):
    os.makedirs(cfg.save_dir, exist_ok=True)

    with open(os.path.join(cfg.save_dir, 'config.json'), 'w') as f:
        json.dump(vars(cfg), f, indent=4, default=lambda x: str(x))

    loss_fn = HybridLoss(cfg.device)
    loss_fn.ce_weights = torch.tensor([0.2, 1.0, 1.0, 2.0], device=cfg.device)
    loss_fn.ce_loss = nn.CrossEntropyLoss(weight=loss_fn.ce_weights)

    base_optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    optimizer = Lookahead(base_optimizer, k=5, alpha=0.5)

    def lr_lambda(epoch):
        current_ep = epoch + 1
        if current_ep <= cfg.warmup_epochs: return current_ep / cfg.warmup_epochs
        progress = (current_ep - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs)
        return 0.5 * (1.0 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(base_optimizer, lr_lambda=lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)
    model.ema_teacher = EMATeacher(model, cfg.ema_alpha)

    last_ckpt_path = os.path.join(cfg.save_dir, "LAST_CHECKPOINT.pt")
    best_model_path = os.path.join(cfg.save_dir, "BEST_MODEL.pt")
    best_ema_path = os.path.join(cfg.save_dir, "BEST_MODEL_EMA.pt")
    csv_path = os.path.join(cfg.save_dir, "training_log_detailed.csv")

    start_epoch = 1
    best_score = -1.0

    # --- بخش اصلاح شده: هوشمندی در لود کردن وزن‌ها ---
    ckpt_to_load = None

    # ۱. مطمئن شویم numpy در دسترس است (بالای تابع run_training یا اینجا)
    import numpy as np

    # ۱. اطمینان از تعریف مسیرها
    ckpt_to_load = None
    if os.path.exists(last_ckpt_path):
        ckpt_to_load = last_ckpt_path
        print(f"🔄 Resuming from LAST checkpoint...")
    elif os.path.exists(best_model_path):
        ckpt_to_load = best_model_path
        print(f"⭐ Resuming from BEST model weights...")

    if ckpt_to_load:
        try:
            import numpy as np
            # حل مسائل امنیتی نسخه جدید پایتورچ
            torch.serialization.add_safe_globals([np._core.multiarray.scalar, np.dtype])

            ckpt = torch.load(ckpt_to_load, map_location=cfg.device, weights_only=False)

            # استخراج State Dict
            if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
                raw_state_dict = ckpt["model_state_dict"]
                # بازیابی وضعیت کامل برای جلوگیری از شوک به مدل
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])
                scaler.load_state_dict(ckpt["scaler_state_dict"])
                start_epoch = ckpt.get("epoch", 0) + 1
                best_score = ckpt.get("best_score", -1.0)
            else:
                raw_state_dict = ckpt

            # اصلاح هوشمند نام لایه‌ها (Mapping)
            model_state = model.state_dict()
            new_state_dict = {}
            for k, v in raw_state_dict.items():
                name = k.replace('module.', '') # حذف پیشوند موازی‌سازی
                if name not in model_state and f"encoder.{name}" in model_state:
                    name = f"encoder.{name}" # اصلاح پیشوند انکودر
                new_state_dict[name] = v

            # لود نهایی
            msg = model.load_state_dict(new_state_dict, strict=False)
            print(f"✅ Loading Report: {msg}")

            # همگام‌سازی تیچر (بسیار مهم)
            model.ema_teacher.teacher.load_state_dict(model.state_dict())
            print(f"🚀 Weights recovered. Starting from epoch {start_epoch} with LR: {base_optimizer.param_groups[0]['lr']:.2e}")

        except Exception as e:
            print(f"⚠️ Load failed: {e}. Starting fresh.")

    # =========================


    # --- ادامه لوپ آموزش (بدون تغییر) ---
    print("\n" + "=" * 115)
    print(f"{'Ep':<3} | {'vLoss':<7} | {'Dice (WT/TC/ET/Mean)':<24} | {'Score':<6} | {'HD95 POST (W/T/E/M)':<20} | {'HD95 RAW (W/T/E/M)'}")
    print("-" * 115)

    for ep in range(start_epoch, cfg.epochs + 1):
        model.train()
        train_loss = 0.0
        for i, batch in enumerate(train_loader):
            x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
            if y.ndim == 4: y = y.unsqueeze(1)
            y = y.long(); y[y == 4] = 3

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                out = model(x)
                loss = loss_fn(out["logits"], y)
                if cfg.deep_supervision and "aux1" in out:
                    loss += cfg.ds_weights[0] * loss_fn(out["aux1"], y)
                    loss += cfg.ds_weights[1] * loss_fn(out["aux2"], y)

            scaler.scale(loss/cfg.accum_iter).backward()
            if (i+1) % cfg.accum_iter == 0 or (i+1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                model.ema_teacher.update(model)
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        d_cls, h_raw, h_post, v_loss = validate(model, val_loader, loss_fn, ep)
        d_mean, score = np.mean(d_cls), (0.2 * d_cls[0] + 0.3 * d_cls[1] + 0.5 * d_cls[2])
        current_lr = base_optimizer.param_groups[0]['lr']

        fmt_hd = lambda arr: f"{arr[0]:.1f}/{arr[1]:.1f}/{arr[2]:.1f}/{np.mean(arr):.1f}" if arr[0] >= 0 else f"{'-/-/-/-':<20}"
        print(f"{ep:02d} | {v_loss:.4f} | {d_cls[0]:.3f}/{d_cls[1]:.3f}/{d_cls[2]:.3f}/{d_mean:.3f} | {score:.4f} | {fmt_hd(h_post):<20} | {fmt_hd(h_raw)}")

        # Logging & Saving
        with open(csv_path, "a", newline="") as f:
            writer = csv.writer(f)
            if ep == 1: writer.writerow(["epoch", "lr", "train_loss", "val_loss", "dice_mean", "score"])
            writer.writerow([ep, f"{current_lr:.2e}", f"{avg_train_loss:.4f}", f"{v_loss:.4f}", f"{d_mean:.4f}", f"{score:.4f}"])

        checkpoint = {
            "epoch": ep, "best_score": max(best_score, score),
            "model_state_dict": model.state_dict(),
            "ema_teacher_state_dict": model.ema_teacher.teacher.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
        }
        torch.save(checkpoint, last_ckpt_path)
        if score > best_score:
            best_score = score
            torch.save(model.state_dict(), best_model_path)
            torch.save(model.ema_teacher.teacher.state_dict(), best_ema_path)
            print(f"💾 Best Models Updated (Score: {score:.4f})")
        scheduler.step()

# =========================
# 6) EXECUTION
# =========================
model = model1.to(cfg.device)
run_training(model, train_loader, val_loader)

# همه چیز درست فقط 6 ستون لاگ مینوشت

'''

In [ ]:
# قبل از تغییر خواندن بهترین مدل و اخرین مدل
'''

import os
import copy
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Tuple # اضافه شده برای رفع خطای احتمالی تایپ
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.metrics import HausdorffDistanceMetric
from monai.transforms import KeepLargestConnectedComponent
from monai.utils import set_determinism

# =========================
# 1) CONFIGURATION
# =========================

#cfg = CFG()
set_determinism(cfg.seed)

# =========================
# 2) EMA TEACHER & HYBRID LOSS
# =========================
class EMATeacher:
    def __init__(self, model, alpha=0.99):
        self.teacher = copy.deepcopy(model).eval()
        for p in self.teacher.parameters():
            p.requires_grad = False
        self.alpha = alpha

    @torch.no_grad()
    def update(self, student):
        for t_param, s_param in zip(self.teacher.parameters(), student.parameters()):
            t_param.data.mul_(self.alpha).add_(s_param.data, alpha=1.0 - self.alpha)

class HybridLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        #self.ce_weights = torch.tensor([0.2, 1.0, 1.5, 2.0], device=device)
        #self.ce_weights = torch.tensor([0.5, 2.0, 1.5, 4.0], device=device)

        self.dice_loss = DiceCELoss(include_background=False, to_onehot_y=True, softmax=True, batch=True)
        self.ce_loss = nn.CrossEntropyLoss(weight=self.ce_weights)

    def forward(self, logits, y):
        return self.dice_loss(logits, y) + self.ce_loss(logits, y.squeeze(1).long())

@torch.no_grad()
def to_regions(lbl):
    wt = (lbl > 0).float()
    tc = ((lbl == 1) | (lbl == 3)).float()
    et = (lbl == 3).float()
    return torch.stack([wt, tc, et], dim=1)

# =========================
# 3) VALIDATE FUNCTION
# =========================
@torch.no_grad()
def validate(model, loader, loss_fn, epoch):
    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model
    net.eval()
    compute_hd = (epoch % cfg.hd_every == 0)
    hd_m_raw = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    hd_m_post = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    post_filter = KeepLargestConnectedComponent(applied_labels=[1, 2, 3], is_onehot=False)

    dices, val_loss = [], 0.0
    for batch in loader:
        x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
        if y.ndim == 4: y = y.unsqueeze(1)
        y = y.long(); y[y == 4] = 3
        logits = sliding_window_inference(x, cfg.roi_size, cfg.sw_batch_size, lambda z: net(z)["logits"], overlap=cfg.sw_overlap)
        val_loss += loss_fn(logits, y).item()
        pred_raw = torch.argmax(logits, dim=1, keepdim=True)
        raw_reg, true_reg = to_regions(pred_raw.squeeze(1)), to_regions(y.squeeze(1))

        inter = (raw_reg * true_reg).sum((2,3,4))
        union = raw_reg.sum((2,3,4)) + true_reg.sum((2,3,4))
        dices.append(torch.nan_to_num(2.*inter/(union+1e-5), 0.).mean(0).cpu().numpy())

        if compute_hd:
            processed = torch.stack([post_filter(p.cpu().unsqueeze(0)).squeeze(0) for p in pred_raw.squeeze(1)]).to(cfg.device)
            post_reg = to_regions(processed)
            hd_m_raw(raw_reg, true_reg); hd_m_post(post_reg, true_reg)

    d_cls = np.mean(dices, axis=0)

    def safe_agg(metric):
        val = metric.aggregate()
        if isinstance(val, torch.Tensor): val = val.cpu().numpy()
        res = np.ones(3) * 100.0
        if np.isscalar(val): res[:] = val
        elif len(val) >= 3: res = val[:3]
        elif len(val) > 0: res[:len(val)] = val
        return np.nan_to_num(res, nan=100.0)

    h_raw = safe_agg(hd_m_raw) if compute_hd else np.array([-1.0]*3)
    h_post = safe_agg(hd_m_post) if compute_hd else np.array([-1.0]*3)
    return d_cls, h_raw, h_post, val_loss / len(loader)

# =========================================================
# 4) اصلاح شده: کلاس Lookahead با شمارشگر مستقل
# =========================================================
class Lookahead(torch.optim.Optimizer):
    def __init__(self, optimizer, k=5, alpha=0.5):
        self.optimizer = optimizer
        self.k = k
        self.alpha = alpha
        self.param_groups = self.optimizer.param_groups
        self.state = self.optimizer.state
        self.defaults = self.optimizer.defaults
        self.lookahead_step = 0 # شمارشگر اختصاصی
        self.slow_weights = [[p.data.clone().detach() for p in group['params']]
                             for group in self.param_groups]

    def step(self, closure=None):
        loss = self.optimizer.step(closure)
        self.lookahead_step += 1

        if self.lookahead_step % self.k == 0:
            for group, slow_weights in zip(self.param_groups, self.slow_weights):
                for p, slow_p in zip(group['params'], slow_weights):
                    if p.grad is None: continue
                    slow_p.add_(p.data - slow_p, alpha=self.alpha)
                    p.data.copy_(slow_p)
        return loss

    def zero_grad(self, set_to_none=True):
        self.optimizer.zero_grad(set_to_none=set_to_none)

    def state_dict(self):
        sd = self.optimizer.state_dict()
        sd['lookahead_step'] = self.lookahead_step
        return sd

    def load_state_dict(self, state_dict):
        self.lookahead_step = state_dict.pop('lookahead_step', 0)
        self.optimizer.load_state_dict(state_dict)

# =========================================================
# 5) تابع اصلی آموزش با Resume Logic هوشمند
# =========================================================
import json
def run_training(model, train_loader, val_loader):
    os.makedirs(cfg.save_dir, exist_ok=True)

    # ذخیره تنظیمات برای مراجعات بعدی
    with open(os.path.join(cfg.save_dir, 'config.json'), 'w') as f:
        json.dump(vars(cfg), f, indent=4, default=lambda x: str(x))

    # --- تنظیمات Loss بر اساس SOTA ---
    loss_fn = HybridLoss(cfg.device)
    # وزن‌دهی به کلاس‌ها (Background, TC, WT, ET)
    loss_fn.ce_weights = torch.tensor([0.2, 1.0, 1.0, 2.0], device=cfg.device)
    loss_fn.ce_loss = nn.CrossEntropyLoss(weight=loss_fn.ce_weights)

    # --- Optimizer & Lookahead ---
    base_optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    optimizer = Lookahead(base_optimizer, k=5, alpha=0.5)

    # --- Scheduler (Cosine Annealing with Warmup) ---
    def lr_lambda(epoch):
        current_ep = epoch + 1
        if current_ep <= cfg.warmup_epochs:
            return current_ep / cfg.warmup_epochs
        progress = (current_ep - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs)
        return 0.5 * (1.0 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(base_optimizer, lr_lambda=lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)

    # مقداردهی به مدل معلم (EMA)
    model.ema_teacher = EMATeacher(model, cfg.ema_alpha)

    # مسیرهای فایل
    last_ckpt_path = os.path.join(cfg.save_dir, "LAST_CHECKPOINT.pt")
    best_model_path = os.path.join(cfg.save_dir, "BEST_MODEL.pt")
    best_ema_path = os.path.join(cfg.save_dir, "BEST_MODEL_EMA.pt")
    csv_path = os.path.join(cfg.save_dir, "training_log_detailed.csv")

    start_epoch = 1
    best_score = -1.0

    # --- Resume Logic هوشمند ---
    ckpt_to_load = None
    if os.path.exists(last_ckpt_path):
        ckpt_to_load = last_ckpt_path
        print(f"🔄 Resuming from LAST checkpoint...")
    elif os.path.exists(best_model_path):
        ckpt_to_load = best_model_path
        print(f"⭐ Resuming from BEST model weights...")

    if ckpt_to_load:
        try:
            # حل مسائل امنیتی پایتورچ جدید برای لود کردن Numpy
            torch.serialization.add_safe_globals([np._core.multiarray.scalar, np.dtype])
            ckpt = torch.load(ckpt_to_load, map_location=cfg.device, weights_only=False)

            if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
                raw_state_dict = ckpt["model_state_dict"]
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])
                scaler.load_state_dict(ckpt["scaler_state_dict"])
                start_epoch = ckpt.get("epoch", 0) + 1
                best_score = ckpt.get("best_score", -1.0)
            else:
                raw_state_dict = ckpt

            # اصلاح نام لایه‌ها (Mapping)
            model_state = model.state_dict()
            new_state_dict = {}
            for k, v in raw_state_dict.items():
                name = k.replace('module.', '')
                if name not in model_state and f"encoder.{name}" in model_state:
                    name = f"encoder.{name}"
                new_state_dict[name] = v

            model.load_state_dict(new_state_dict, strict=False)
            model.ema_teacher.teacher.load_state_dict(model.state_dict())
            print(f"🚀 Weights recovered. Starting from epoch {start_epoch}")
        except Exception as e:
            print(f"⚠️ Load failed: {e}. Starting fresh.")

    # --- شروع لوپ آموزش ---
    print("\n" + "=" * 115)
    print(f"{'Ep':<3} | {'vLoss':<7} | {'Dice (WT/TC/ET/Mean)':<24} | {'Score':<6} | {'HD95 POST (W/T/E/M)':<20} | {'HD95 RAW (W/T/E/M)'}")
    print("-" * 115)

    for ep in range(start_epoch, cfg.epochs + 1):
        model.train()
        train_loss = 0.0

        for i, batch in enumerate(train_loader):
            x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
            if y.ndim == 4: y = y.unsqueeze(1)
            y = y.long(); y[y == 4] = 3

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                out = model(x)
                loss = loss_fn(out["logits"], y)
                if cfg.deep_supervision and "aux1" in out:
                    loss += cfg.ds_weights[0] * loss_fn(out["aux1"], y)
                    loss += cfg.ds_weights[1] * loss_fn(out["aux2"], y)

            scaler.scale(loss/cfg.accum_iter).backward()

            if (i+1) % cfg.accum_iter == 0 or (i+1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                model.ema_teacher.update(model)

            train_loss += loss.item()

        # مرحله Validation و محاسبه متریک‌ها
        avg_train_loss = train_loss / len(train_loader)
        d_cls, h_raw, h_post, v_loss = validate(model, val_loader, loss_fn, ep)
        d_mean = np.mean(d_cls)
        score = (0.2 * d_cls[0] + 0.3 * d_cls[1] + 0.5 * d_cls[2])
        current_lr = base_optimizer.param_groups[0]['lr']

        # چاپ در کنسول
        fmt_hd_print = lambda arr: f"{arr[0]:.1f}/{arr[1]:.1f}/{arr[2]:.1f}/{np.mean(arr):.1f}" if arr[0] >= 0 else f"{'-/-/-/-':<20}"
        print(f"{ep:02d} | {v_loss:.4f} | {d_cls[0]:.3f}/{d_cls[1]:.3f}/{d_cls[2]:.3f}/{d_mean:.3f} | {score:.4f} | {fmt_hd_print(h_post):<20} | {fmt_hd_print(h_raw)}")

        # =========================================================
        # لاگ‌گذاری ۱۷ ستونه (الگوی کد قبلی شما)
        # =========================================================
        # =========================================================
        # لاگ‌گذاری ۱۷ ستونه (اصلاح شده برای پایداری میانگین HD95)
        # =========================================================
        log_header = not os.path.exists(csv_path)

        # اصلاح نویز میانگین: اگر مقدار منفی بود (یعنی محاسبه نشده)، میانگین را هم 1- بگذار
        h_post_mean = np.mean(h_post) if h_post[0] >= 0 else -1.0
        h_raw_mean = np.mean(h_raw) if h_raw[0] >= 0 else -1.0

        with open(csv_path, "a", newline="") as f:
            writer = csv.writer(f)
            if log_header:
                writer.writerow([
                    "epoch", "lr", "train_loss", "val_loss",
                    "dice_WT", "dice_TC", "dice_ET", "dice_mean", "score",
                    "hd_post_WT", "hd_post_TC", "hd_post_ET", "hd_post_mean",
                    "hd_raw_WT", "hd_raw_TC", "hd_raw_ET", "hd_raw_mean"
                ])

            writer.writerow([
                ep, f"{current_lr:.2e}", f"{avg_train_loss:.4f}", f"{v_loss:.4f}",
                f"{d_cls[0]:.4f}", f"{d_cls[1]:.4f}", f"{d_cls[2]:.4f}", f"{d_mean:.4f}", f"{score:.4f}",
                f"{h_post[0]:.2f}", f"{h_post[1]:.2f}", f"{h_post[2]:.2f}", f"{h_post_mean:.2f}",
                f"{h_raw[0]:.2f}", f"{h_raw[1]:.2f}", f"{h_raw[2]:.2f}", f"{h_raw_mean:.2f}"
            ])

        # ذخیره چک‌پوینت‌ها (دقیقاً بدون تغییر طبق کد خودت)
        checkpoint = {
            "epoch": ep, "best_score": max(best_score, score),
            "model_state_dict": model.state_dict(),
            "ema_teacher_state_dict": model.ema_teacher.teacher.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
        }
        torch.save(checkpoint, last_ckpt_path)


        if score > best_score:
            best_score = score
            torch.save(model.state_dict(), best_model_path)
            torch.save(model.ema_teacher.teacher.state_dict(), best_ema_path)
            print(f"💾 Best Models Updated (Score: {score:.4f})")

        scheduler.step()


# =========================
# 6) EXECUTION
# =========================
model = model1.to(cfg.device)
run_training(model, train_loader, val_loader)


'''

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

def debug_model_output(model, loader, cfg, num_samples=1):
    model.eval()
    device = cfg.device

    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= num_samples: break

            x = batch["image"].to(device)
            y = batch["label"].to(device)
            y[y == 4] = 3 # اصلاح لیبل

            # ۱. اجرای مدل
            with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                logits = sliding_window_inference(
                    x, cfg.roi_size, cfg.sw_batch_size,
                    lambda z: model(z)["logits"], overlap=cfg.sw_overlap
                )

            probs = torch.softmax(logits, dim=1)
            pred = torch.argmax(probs, dim=1, keepdim=True)

            # تبدیل به ریجن‌های سه گانه (WT, TC, ET)
            y_reg = to_regions(y.squeeze(1)).cpu().numpy()[0] # [3, H, W, D]
            p_reg = to_regions(pred.squeeze(1)).cpu().numpy()[0] # [3, H, W, D]

            class_names = ["WT (Whole)", "TC (Core)", "ET (Enhancing)"]

            print(f"--- Analysis for Sample {i+1} ---")

            for c in range(3):
                gt_pixels = np.sum(y_reg[c])
                pred_pixels = np.sum(p_reg[c])

                # محاسبه Dice دستی
                intersection = np.sum(y_reg[c] * p_reg[c])
                dice = (2. * intersection) / (gt_pixels + pred_pixels + 1e-5)

                print(f"\nClass: {class_names[c]}")
                print(f"  - GT volume: {gt_pixels} pixels")
                print(f"  - Pred volume: {pred_pixels} pixels")
                print(f"  - Manual Dice: {dice:.4f}")

                if gt_pixels > 0 and pred_pixels > 0:
                    # پیدا کردن مختصات مرکز توده
                    coords_gt = np.argwhere(y_reg[c])
                    coords_p = np.argwhere(p_reg[c])

                    centroid_gt = coords_gt.mean(axis=0)
                    centroid_p = coords_p.mean(axis=0)
                    dist = np.linalg.norm(centroid_gt - centroid_p)

                    print(f"  - Centroid Distance: {dist:.2f} pixels")

                    # بررسی فاصله مرزها (بسیار ساده شده برای تست)
                    max_dist = np.max(np.abs(np.max(coords_gt, axis=0) - np.max(coords_p, axis=0)))
                    print(f"  - Boundary Offset (approx): {max_dist:.2f} pixels")
                else:
                    print("  - [!] One of the masks is EMPTY. HD95 will be 100.0")

            # ۲. تصویرسازی برای اطمینان چشمی
            slice_idx = 75 # یک اسلایس وسط مغز
            plt.figure(figsize=(15, 5))
            plt.subplot(1, 3, 1); plt.imshow(x[0, 0, :, :, slice_idx].cpu(), cmap='gray'); plt.title("T1ce Image")
            plt.subplot(1, 3, 2); plt.imshow(y_reg[0, :, :, slice_idx]); plt.title("GT WT")
            plt.subplot(1, 3, 3); plt.imshow(p_reg[0, :, :, slice_idx]); plt.title("Pred WT")
            plt.show()

# برای اجرا در سلول جدید:
model = model1.to(cfg.device)
#debug_model_output(model, val_loader, cfg)

In [ ]:
import torch
import numpy as np

def debug_prediction_integrity(model, loader, cfg, num_samples=3):
    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model
    net.eval()

    print(f"🔍 Starting Deep Inspection on {num_samples} samples...\n")
    print(f"{'Sample':<8} | {'Class':<4} | {'GT Pixels':<12} | {'Pred Pixels':<12} | {'Manual Dice':<12} | {'Status'}")
    print("-" * 85)

    samples_processed = 0
    with torch.no_grad():
        for batch in loader:
            if samples_processed >= num_samples: break

            x = batch["image"].to(cfg.device)
            y = batch["label"].to(cfg.device)

            # اصلاح لیبل مشابه کد اصلی
            y = y.long()
            y[y == 4] = 3
            if y.ndim == 4: y = y.unsqueeze(1)

            # اینفرنس
            with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                logits = sliding_window_inference(
                    x, cfg.roi_size, cfg.sw_batch_size,
                    lambda z: net(z)["logits"], overlap=cfg.sw_overlap
                )

            # تبدیل به نواحی (WT, TC, ET)
            probs = torch.softmax(logits, dim=1)
            pred_mask = torch.argmax(probs, dim=1, keepdim=True)

            true_reg = to_regions(y.squeeze(1)) # (B, 3, H, W, D)
            pred_reg = to_regions(pred_mask.squeeze(1)) # (B, 3, H, W, D)

            class_names = ["WT", "TC", "ET"]

            for b in range(x.shape[0]):
                if samples_processed >= num_samples: break

                for c in range(3):
                    gt_sum = true_reg[b, c].sum().item()
                    pd_sum = pred_reg[b, c].sum().item()

                    # محاسبه Dice دستی
                    inter = (true_reg[b, c] * pred_reg[b, c]).sum().item()
                    union = gt_sum + pd_sum
                    dice = (2.0 * inter) / (union + 1e-6) if union > 0 else 1.0

                    # تعیین وضعیت
                    status = "✅ OK"
                    if gt_sum == 0 and pd_sum > 0: status = "⚠️ FP (False Pos)"
                    if gt_sum > 0 and pd_sum == 0: status = "❌ FN (Missed)"
                    if gt_sum == 0 and pd_sum == 0: status = "⚪ Empty"

                    print(f"{samples_processed:<8} | {class_names[c]:<4} | {int(gt_sum):<12} | {int(pd_sum):<12} | {dice:<12.4f} | {status}")

                samples_processed += 1
                print("-" * 85)

# اجرای تست
# نکته: مطمئن باش مدل و val_loader قبلاً تعریف شده باشند
model = model1.to(cfg.device)
#debug_prediction_integrity(model, val_loader, cfg, num_samples=5)

In [ ]:
# قبل از تغییرات در ایپاک 108
'''
import os

import copy

import csv

import numpy as np

import torch

import torch.nn as nn

import torch.nn.functional as F

from dataclasses import dataclass

from typing import Tuple # اضافه شده برای رفع خطای احتمالی تایپ

from monai.losses import DiceCELoss

from monai.inferers import sliding_window_inference

from monai.metrics import HausdorffDistanceMetric

from monai.transforms import KeepLargestConnectedComponent

from monai.utils import set_determinism



# =========================

# 1) CONFIGURATION

# =========================



#cfg = CFG()

set_determinism(cfg.seed)



# =========================

# 2) EMA TEACHER & HYBRID LOSS

# =========================

class EMATeacher:

    def __init__(self, model, alpha=0.99):

        self.teacher = copy.deepcopy(model).eval()

        for p in self.teacher.parameters():

            p.requires_grad = False

        self.alpha = alpha



    @torch.no_grad()

    def update(self, student):

        for t_param, s_param in zip(self.teacher.parameters(), student.parameters()):

            t_param.data.mul_(self.alpha).add_(s_param.data, alpha=1.0 - self.alpha)



class HybridLoss(nn.Module):

    def __init__(self, device):

        super().__init__()

        #self.ce_weights = torch.tensor([0.2, 1.0, 1.5, 2.0], device=device)

        self.ce_weights = torch.tensor([0.1, 1.0, 2.0, 8.0], device=device)



        self.dice_loss = DiceCELoss(include_background=False, to_onehot_y=True, softmax=True, batch=True)

        self.ce_loss = nn.CrossEntropyLoss(weight=self.ce_weights)



    def forward(self, logits, y):

        return (1.3 * self.dice_loss(logits, y)) + self.ce_loss(logits, y.squeeze(1).long())



@torch.no_grad()

def to_regions(lbl):

    wt = (lbl > 0).float()

    tc = ((lbl == 1) | (lbl == 3)).float()

    et = (lbl == 3).float()

    return torch.stack([wt, tc, et], dim=1)



# =========================

# 3) VALIDATE FUNCTION

# =========================



# خط ۱۳ یا حوالی آن را به این شکل تغییر بده:

from monai.transforms import RemoveSmallObjects  # جایگزین KeepLargestConnectedComponent



@torch.no_grad()

def validate(model, loader, loss_fn, epoch):

    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model

    net.eval()



    # ۱. آزاد سازی حافظه قبل از شروع برای جلوگیری از OOM

    torch.cuda.empty_cache()



    compute_hd = (epoch % cfg.hd_every == 0)

    hd_m_raw = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")

    hd_m_post = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")

    post_filter = KeepLargestConnectedComponent(applied_labels=[1, 2, 3], is_onehot=False)





    # به جای KeepLargest، از حذف اشیاء زیر ۵۰ پیکسل استفاده می‌کنیم

    # این کار باعث می‌شود خوشه‌های اصلی تومور حفظ شوند و فقط نویزهای تک‌-پیکسلی حذف شوند

    #post_filter = RemoveSmallObjects(min_size=1000, connectivity=1)



    dices, val_loss = [], 0.0



    # ۲. اضافه کردن enumerate برای کنترل تعداد نمونه‌ها

    for i, batch in enumerate(loader):

        # ۳. محدود کردن ولیدیشن به ۴۰ مورد (قابل تغییر در CFG) برای افزایش سرعت

        if i >= getattr(cfg, 'val_sample_limit', 40):

            break



        x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)

        if y.ndim == 4: y = y.unsqueeze(1)

        y = y.long(); y[y == 4] = 3



        # ۴. استفاده از autocast برای کاهش مصرف حافظه گرافیکی (VRAM)

        with torch.amp.autocast("cuda", enabled=cfg.use_amp):

            logits = sliding_window_inference(

                x, cfg.roi_size, cfg.sw_batch_size,

                lambda z: net(z)["logits"],

                overlap=cfg.sw_overlap

            )

            val_loss += loss_fn(logits, y).item()



        '''pred_raw = torch.argmax(logits, dim=1, keepdim=True)



        # ۵. پاک کردن logits بلافاصله پس از استفاده برای آزاد شدن فضا

        del logits'''

        # --- بخش اصلاح شده شروع می‌شود ---



        # تبدیل Logits به احتمال (Probability)

        probs = torch.softmax(logits, dim=1)



        # حذف پیکسل‌های با اطمینان پایین (مثلا زیر 0.7)

        # این کار نویزهای پراکنده که مدل در موردشان مطمئن نیست را پاک می‌کند

        thresh = 0.8

        probs[probs < thresh] = 0



        # حالا Argmax می‌گیریم

        pred_raw = torch.argmax(probs, dim=1, keepdim=True)



        # پاک کردن متغیرهای سنگین

        del logits, probs



        raw_reg, true_reg = to_regions(pred_raw.squeeze(1)), to_regions(y.squeeze(1))



        inter = (raw_reg * true_reg).sum((2,3,4))

        union = raw_reg.sum((2,3,4)) + true_reg.sum((2,3,4))

        dices.append(torch.nan_to_num(2.*inter/(union+1e-5), 0.).mean(0).cpu().numpy())



        if compute_hd:

            # انتقال به CPU برای پردازش مورفولوژیک

            preds_cpu = pred_raw.squeeze(1).cpu()



            # اعمال فیلتر حذف اشیاء کوچک روی هر نمونه در بچ

            processed = torch.stack([post_filter(p.unsqueeze(0)).squeeze(0) for p in preds_cpu]).to(cfg.device)



            post_reg = to_regions(processed)

            hd_m_raw(raw_reg, true_reg)

            hd_m_post(post_reg, true_reg)

            del processed, preds_cpu # پاکسازی حافظه



    d_cls = np.mean(dices, axis=0)



    def safe_agg(metric):

        val = metric.aggregate()

        if isinstance(val, torch.Tensor): val = val.cpu().numpy()

        res = np.ones(3) * 100.0

        if np.isscalar(val): res[:] = val

        elif len(val) >= 3: res = val[:3]

        elif len(val) > 0: res[:len(val)] = val

        return np.nan_to_num(res, nan=100.0)



    h_raw = safe_agg(hd_m_raw) if compute_hd else np.array([-1.0]*3)

    h_post = safe_agg(hd_m_post) if compute_hd else np.array([-1.0]*3)



    # ۶. تخلیه نهایی حافظه قبل از بازگشت به آموزش

    torch.cuda.empty_cache()



    return d_cls, h_raw, h_post, val_loss / (i + 1) # تقسیم بر تعداد واقعی موارد پردازش شده



# محدود کردن تعداد سمپل ولیدیشن

'''@torch.no_grad()

def validate(model, loader, loss_fn, epoch):

    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model

    net.eval()



    # --- آزاد سازی حافظه قبل از شروع ---

    torch.cuda.empty_cache()



    compute_hd = (epoch % cfg.hd_every == 0)

    hd_m_raw = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")

    hd_m_post = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")

    post_filter = KeepLargestConnectedComponent(applied_labels=[1, 2, 3], is_onehot=False)



    dices, val_loss = [], 0.0

    for batch in loader:

        x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)

        if y.ndim == 4: y = y.unsqueeze(1)

        y = y.long(); y[y == 4] = 3

        logits = sliding_window_inference(x, cfg.roi_size, cfg.sw_batch_size, lambda z: net(z)["logits"], overlap=cfg.sw_overlap)

        val_loss += loss_fn(logits, y).item()

        pred_raw = torch.argmax(logits, dim=1, keepdim=True)

        raw_reg, true_reg = to_regions(pred_raw.squeeze(1)), to_regions(y.squeeze(1))



        inter = (raw_reg * true_reg).sum((2,3,4))

        union = raw_reg.sum((2,3,4)) + true_reg.sum((2,3,4))

        dices.append(torch.nan_to_num(2.*inter/(union+1e-5), 0.).mean(0).cpu().numpy())



        if compute_hd:

            processed = torch.stack([post_filter(p.cpu().unsqueeze(0)).squeeze(0) for p in pred_raw.squeeze(1)]).to(cfg.device)

            post_reg = to_regions(processed)

            hd_m_raw(raw_reg, true_reg); hd_m_post(post_reg, true_reg)



    d_cls = np.mean(dices, axis=0)



    def safe_agg(metric):

        val = metric.aggregate()

        if isinstance(val, torch.Tensor): val = val.cpu().numpy()

        res = np.ones(3) * 100.0

        if np.isscalar(val): res[:] = val

        elif len(val) >= 3: res = val[:3]

        elif len(val) > 0: res[:len(val)] = val

        return np.nan_to_num(res, nan=100.0)



    h_raw = safe_agg(hd_m_raw) if compute_hd else np.array([-1.0]*3)

    h_post = safe_agg(hd_m_post) if compute_hd else np.array([-1.0]*3)

    return d_cls, h_raw, h_post, val_loss / len(loader)

'''

# =========================================================

# 4) اصلاح شده: کلاس Lookahead با شمارشگر مستقل

# =========================================================

class Lookahead(torch.optim.Optimizer):

    def __init__(self, optimizer, k=5, alpha=0.5):

        self.optimizer = optimizer

        self.k = k

        self.alpha = alpha

        self.param_groups = self.optimizer.param_groups

        self.state = self.optimizer.state

        self.defaults = self.optimizer.defaults

        self.lookahead_step = 0 # شمارشگر اختصاصی

        self.slow_weights = [[p.data.clone().detach() for p in group['params']]

                             for group in self.param_groups]



    def step(self, closure=None):

        loss = self.optimizer.step(closure)

        self.lookahead_step += 1



        if self.lookahead_step % self.k == 0:

            for group, slow_weights in zip(self.param_groups, self.slow_weights):

                for p, slow_p in zip(group['params'], slow_weights):

                    if p.grad is None: continue

                    slow_p.add_(p.data - slow_p, alpha=self.alpha)

                    p.data.copy_(slow_p)

        return loss



    def zero_grad(self, set_to_none=True):

        self.optimizer.zero_grad(set_to_none=set_to_none)



    def state_dict(self):

        sd = self.optimizer.state_dict()

        sd['lookahead_step'] = self.lookahead_step

        return sd



    def load_state_dict(self, state_dict):

        self.lookahead_step = state_dict.pop('lookahead_step', 0)

        self.optimizer.load_state_dict(state_dict)

        # 🚀 اضافه کردن این خط: همگام‌سازی وزن‌های کند با وزن‌های لود شده

        self.slow_weights = [[p.data.clone().detach() for p in group['params']]

                             for group in self.param_groups]



# =========================================================

# 5) تابع اصلی آموزش با Resume Logic هوشمند

# =========================================================

import json

def run_training(model, train_loader, val_loader):

    os.makedirs(cfg.save_dir, exist_ok=True)



    # ذخیره تنظیمات برای مراجعات بعدی

    with open(os.path.join(cfg.save_dir, 'config.json'), 'w') as f:

        json.dump(vars(cfg), f, indent=4, default=lambda x: str(x))



    # --- تنظیمات Loss بر اساس SOTA ---

    loss_fn = HybridLoss(cfg.device)

    # وزن‌دهی به کلاس‌ها (Background, TC, WT, ET)

    loss_fn.ce_weights = torch.tensor([0.1, 1.0, 2.0, 8.0], device=cfg.device)

    loss_fn.ce_loss = nn.CrossEntropyLoss(weight=loss_fn.ce_weights)



    # --- Optimizer & Lookahead ---

    base_optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    optimizer = Lookahead(base_optimizer, k=5, alpha=0.5)



    # --- Scheduler (Cosine Annealing with Warmup) ---

    '''def lr_lambda(epoch):

        current_ep = epoch + 1

        if current_ep <= cfg.warmup_epochs:

            return current_ep / cfg.warmup_epochs

        progress = (current_ep - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs)

        #return 0.5 * (1.0 + np.cos(np.pi * progress))

        return 1.0'''



    def lr_lambda(epoch):

        # از اپوک 108 به بعد، نرخ یادگیری را هر اپوک 10 درصد کم کن

        if epoch >= 108:

            exponent = epoch - 108

            return 0.9 ** exponent

        return 1.0



    scheduler = torch.optim.lr_scheduler.LambdaLR(base_optimizer, lr_lambda=lr_lambda)

    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)



    # مقداردهی به مدل معلم (EMA)

    model.ema_teacher = EMATeacher(model, cfg.ema_alpha)



    # مسیرهای فایل

    last_ckpt_path = os.path.join(cfg.save_dir, "LAST_CHECKPOINT.pt")

    best_model_path = os.path.join(cfg.save_dir, "BEST_MODEL.pt")

    best_ema_path = os.path.join(cfg.save_dir, "BEST_MODEL_EMA.pt")

    csv_path = os.path.join(cfg.save_dir, "training_log_detailed.csv")



    start_epoch = 1

    best_score = -1.0





    # --- Resume Logic هوشمند ---

    ckpt_to_load = None

    if os.path.exists(last_ckpt_path):

        ckpt_to_load = last_ckpt_path

        print(f"🔄 Resuming from LAST checkpoint...")

    elif os.path.exists(best_model_path):

        ckpt_to_load = best_model_path

        print(f"⭐ Resuming from BEST model weights...")



    if ckpt_to_load:

        # اضافه کردن weights_only=False برای رفع ارور UnpicklingError

        ckpt = torch.load(ckpt_to_load, map_location=cfg.device, weights_only=False)

        raw_state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt









        # لود کردن وزن‌ها با متد تطبیق نام لایه‌ها

        curr_state = model.state_dict()

        new_state = {}

        matched = 0

        for k, v in raw_state.items():

            name = k.replace('module.', '').replace('model.', '')

            if name in curr_state:

                new_state[name] = v

                matched += 1

            elif f"encoder.{name}" in curr_state:

                new_state[f"encoder.{name}"] = v

                matched += 1



        model.load_state_dict(new_state, strict=False)

        print(f"✅ Loaded {matched}/{len(curr_state)} layers.")





        # لود کردن وضعیت‌های آموزش

        if isinstance(ckpt, dict):

            # اولویت اول: خواندن ایپاک از خودِ فایل (چه Best باشد چه Last)

            if "epoch" in ckpt:

                start_epoch = ckpt["epoch"] + 1

                best_score = ckpt.get("best_score", -1.0)

                print(f"📖 Found epoch in checkpoint: Resuming from {start_epoch}")

            else:

                # اگر فایل دیکشنری بود ولی عدد ایپاک نداشت (مثلاً فقط وزن بود)

                start_epoch = 1

                best_score = -1.0

                print("⚠️ Checkpoint has no epoch info. Starting from 1.")



            # لود کردن بهینه وضعیت Optimizer و Scheduler فقط در صورت وجود

            if ckpt_to_load == last_ckpt_path and "optimizer_state_dict" in ckpt:

                optimizer.load_state_dict(ckpt["optimizer_state_dict"])

                scheduler.load_state_dict(ckpt["scheduler_state_dict"])

                if "scaler_state_dict" in ckpt:

                    scaler.load_state_dict(ckpt["scaler_state_dict"])



                # 🔥 --- تغییر مهم: شوک به نرخ یادگیری (LR Jump) --- 🔥

                # هدف: بیدار کردن مدل برای اصلاح کلاس ET

                '''new_lr = 3e-05  # مقدار پیشنهادی برای شوک

                for param_group in base_optimizer.param_groups:

                    param_group['lr'] = new_lr



                # بروزرسانی مقادیر پایه در scheduler برای جلوگیری از تداخل

                if hasattr(scheduler, 'base_lrs'):

                    scheduler.base_lrs = [new_lr for _ in base_optimizer.param_groups]

                print(f"🚀 LR Reset to: {new_lr:.2e} for fine-tuning boost!")

                # ---------------------------------------------------- 🔥

                '''

                # 🔥 --- تغییر نهایی: کاهش برای تثبیت (Annealing) --- 🔥

                new_lr = 1e-05  # کاهش از 3e-05 به 1e-05

                for param_group in base_optimizer.param_groups:

                    param_group['lr'] = new_lr



                if hasattr(scheduler, 'base_lrs'):

                    scheduler.base_lrs = [new_lr for _ in base_optimizer.param_groups]



                print(f"📉 LR Reduced to: {new_lr:.2e} for final stabilization.")



                if "scaler_state_dict" in ckpt:

                    scaler.load_state_dict(ckpt["scaler_state_dict"])





        else:

            # اگر فایل اصلاً دیکشنری نبود و فقط وزن خالص (OrderedDict) بود

            start_epoch = 1

            best_score = -1.0

            print("❗ Pure weights detected. Starting from epoch 1.")



    # همگام‌سازی‌های حیاتی

    model.ema_teacher.teacher.load_state_dict(model.state_dict())

    scheduler.last_epoch = start_epoch - 1

    # بروزرسانی LR فیزیکی در گروه‌های پارامتر

    '''

    current_lr_val = scheduler.get_last_lr()[0]

    for param_group in base_optimizer.param_groups:

        param_group['lr'] = current_lr_val





    print(f"📈 Ready! Start Epoch: {start_epoch} | Initial LR: {current_lr_val:.2e}")'''



    # همگام‌سازی‌های حیاتی

    model.ema_teacher.teacher.load_state_dict(model.state_dict())

    scheduler.last_epoch = start_epoch - 1



    # بروزرسانی LR: اگر ریست انجام شده، از آن استفاده کن، در غیر این صورت از شجولر

    current_lr_val = base_optimizer.param_groups[0]['lr']



    print(f"📈 Ready! Start Epoch: {start_epoch} | Initial LR: {current_lr_val:.2e}")



'''

In [ ]:
#  قبل از تغییر لاس و بعد از ایپاک 110
'''
import os
import copy
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Tuple # اضافه شده برای رفع خطای احتمالی تایپ
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.metrics import HausdorffDistanceMetric
from monai.transforms import KeepLargestConnectedComponent
from monai.utils import set_determinism

# =========================
# 1) CONFIGURATION
# =========================

#cfg = CFG()
set_determinism(cfg.seed)

# =========================
# 2) EMA TEACHER & HYBRID LOSS
# =========================
class EMATeacher:
    def __init__(self, model, alpha=0.99):
        self.teacher = copy.deepcopy(model).eval()
        for p in self.teacher.parameters():
            p.requires_grad = False
        self.alpha = alpha

    @torch.no_grad()
    def update(self, student):
        for t_param, s_param in zip(self.teacher.parameters(), student.parameters()):
            t_param.data.mul_(self.alpha).add_(s_param.data, alpha=1.0 - self.alpha)

class HybridLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        #self.ce_weights = torch.tensor([0.2, 1.0, 1.5, 2.0], device=device)
        self.ce_weights = torch.tensor([0.1, 4.0, 8.0, 25.0], device=device)

        self.dice_loss = DiceCELoss(include_background=False, to_onehot_y=True, softmax=True, batch=True)
        self.ce_loss = nn.CrossEntropyLoss(weight=self.ce_weights)

    def forward(self, logits, y):
        return (1.5 * self.dice_loss(logits, y)) + self.ce_loss(logits, y.squeeze(1).long())

@torch.no_grad()
def to_regions(lbl):
    wt = (lbl > 0).float()
    tc = ((lbl == 1) | (lbl == 3)).float()
    et = (lbl == 3).float()
    return torch.stack([wt, tc, et], dim=1)

# =========================
# 3) VALIDATE FUNCTION
# =========================
@torch.no_grad()
def validate(model, loader, loss_fn, epoch):
    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model
    net.eval()
    torch.cuda.empty_cache()

    compute_hd = (epoch % cfg.hd_every == 0)

    # تنظیم دقیق متریک برای تطابق با نتایج دستی
    # ما include_background را True میگذاریم و خودمان فقط ۳ کانال اصلی را میدهیم
    hd_metric = HausdorffDistanceMetric(include_background=True, percentile=95, reduction=None)

    #post_filter = KeepLargestConnectedComponent(applied_labels=[1, 2, 3], is_onehot=False)

    # هر توده زیر 32 پیکسل احتمالا نویز است و حذف آن HD95 را عالی و Dice را شفاف می‌کند.
    post_filter = RemoveSmallObjects(min_size=16, connectivity=1)

    dices, val_loss = [], 0.0

    for i, batch in enumerate(loader):
        if i >= getattr(cfg, 'val_sample_limit', 251): break

        x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)

        # --- گام جدید: دریافت ماسک مغز ---
        # این ماسک در لودر حذف نشده و حالا در دسترس است
        brain_mask = batch["brain_mask_tmp"].to(cfg.device)


        if y.ndim == 4: y = y.unsqueeze(1)
        y = y.long(); y[y == 4] = 3

        with torch.amp.autocast("cuda", enabled=cfg.use_amp):
            logits = sliding_window_inference(
                x, cfg.roi_size, cfg.sw_batch_size,
                lambda z: net(z)["logits"], overlap=cfg.sw_overlap
            )
            val_loss += loss_fn(logits, y).item()

        probs = torch.softmax(logits, dim=1)
        pred_raw = torch.argmax(probs, dim=1, keepdim=True)

        # --- گام طلایی: پاکسازی با ماسک مغز ---
        # تمام پیش‌بینی‌های خارج از فضای مغز صفر می‌شوند
        pred_raw = pred_raw * brain_mask

        # تبدیل به مناطق (WT, TC, ET) - دقیقاً مشابه تست دستی موفق
        raw_reg = to_regions(pred_raw.squeeze(1)).float()
        true_reg = to_regions(y.squeeze(1)).float()

        # محاسبه Dice (دستی و مطمئن)
        inter = (raw_reg * true_reg).sum((2,3,4))
        union = raw_reg.sum((2,3,4)) + true_reg.sum((2,3,4))
        dices.append(torch.nan_to_num(2.*inter/(union+1e-5), 0.).mean(0).cpu().numpy())

        if compute_hd:
            # اعمال فیلتر برای حذف نویزهای تک پیکسلی
            preds_cpu = pred_raw.squeeze(1).cpu()
            processed = torch.stack([post_filter(p.unsqueeze(0)).squeeze(0) for p in preds_cpu]).to(cfg.device)
            post_reg = to_regions(processed).float()

            # تضمین ابعاد ۵ بعدی برای جلوگیری از گیج شدن MONAI
            if post_reg.ndim == 4: post_reg = post_reg.unsqueeze(0)
            if true_reg.ndim == 4: true_reg = true_reg.unsqueeze(0)

            # ارسال به بافر
            hd_metric(post_reg, true_reg)

    d_cls = np.mean(dices, axis=0)

    # استخراج میانگین واقعی بدون تله‌های Inf/100
    if compute_hd:
        hd_vals = hd_metric.get_buffer()
        if isinstance(hd_vals, torch.Tensor): hd_vals = hd_vals.cpu().numpy()

        h_post = []
        for c in range(3):
            # حذف مقادیر پرت (فقط کیس‌هایی که تومور داشتند را حساب کن)
            valid_vals = hd_vals[:, c][np.isfinite(hd_vals[:, c])]
            if len(valid_vals) > 0:
                # میانگین کیس‌های معتبر (مانند 1.0 و 4.0 که در تست دستی دیدی)
                h_post.append(np.mean(valid_vals))
            else:
                h_post.append(100.0)
        hd_metric.reset()
    else:
        h_post = np.array([-1.0]*3)

    return d_cls, np.array(h_post), np.array(h_post), val_loss / (i + 1)

# خط ۱۳ یا حوالی آن را به این شکل تغییر بده:
from monai.transforms import RemoveSmallObjects  # جایگزین KeepLargestConnectedComponent
# عیب یابی نشده بعد از ایپاک 108
'''@torch.no_grad()
def validate(model, loader, loss_fn, epoch):
    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model
    net.eval()

    torch.cuda.empty_cache()

    compute_hd = (epoch % cfg.hd_every == 0)
    hd_m_raw = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    hd_m_post = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    post_filter = KeepLargestConnectedComponent(applied_labels=[1, 2, 3], is_onehot=False)

    dices, val_loss = [], 0.0

    for i, batch in enumerate(loader):
        if i >= getattr(cfg, 'val_sample_limit', 251): # تنظیم روی حداکثر
            break

        x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
        if y.ndim == 4: y = y.unsqueeze(1)
        y = y.long(); y[y == 4] = 3

        with torch.amp.autocast("cuda", enabled=cfg.use_amp):
            logits = sliding_window_inference(
                x, cfg.roi_size, cfg.sw_batch_size,
                lambda z: net(z)["logits"],
                overlap=cfg.sw_overlap
            )
            val_loss += loss_fn(logits, y).item()

        # ۱. تبدیل به احتمال و اعمال آستانه 0.5 برای تعادل دایس
        probs = torch.softmax(logits, dim=1)
        thresh = 0.5
        probs[probs < thresh] = 0
        pred_raw = torch.argmax(probs, dim=1, keepdim=True)

        del logits, probs

        # ۲. محاسبه مناطق
        raw_reg, true_reg = to_regions(pred_raw.squeeze(1)), to_regions(y.squeeze(1))

        # ۳. محاسبه Dice
        inter = (raw_reg * true_reg).sum((2,3,4))
        union = raw_reg.sum((2,3,4)) + true_reg.sum((2,3,4))
        dices.append(torch.nan_to_num(2.*inter/(union+1e-5), 0.).mean(0).cpu().numpy())

        # ۴. محاسبه HD95
        if compute_hd:
            preds_cpu = pred_raw.squeeze(1).cpu()
            processed = torch.stack([post_filter(p.unsqueeze(0)).squeeze(0) for p in preds_cpu]).to(cfg.device)
            post_reg = to_regions(processed)

            hd_m_raw(raw_reg, true_reg)
            hd_m_post(post_reg, true_reg)
            del processed, preds_cpu

    # میانگین‌گیری دایس
    d_cls = np.mean(dices, axis=0)

    # ۵. جراحی بخش safe_agg برای جلوگیری از IndexError
    def safe_agg(metric):
        try:
            val = metric.aggregate()
            if isinstance(val, torch.Tensor):
                val = val.detach().cpu().numpy()

            # تبدیل به آرایه یک بعدی در هر صورت
            val = np.atleast_1d(val)

            # ساخت خروجی پیش‌فرض ۳ تایی
            res = np.ones(3) * 100.0

            if val.size >= 3:
                res = val[:3] # سه کلاس اول (WT, TC, ET)
            elif val.size > 0:
                res[:val.size] = val # پر کردن کلاس‌های موجود

            # پاکسازی NaNها
            res = np.nan_to_num(res, nan=100.0)
            return res
        except:
            return np.array([100.0, 100.0, 100.0])
        finally:
            metric.reset()

    h_raw = safe_agg(hd_m_raw) if compute_hd else np.array([-1.0]*3)
    h_post = safe_agg(hd_m_post) if compute_hd else np.array([-1.0]*3)

    torch.cuda.empty_cache()
    # خروجی نهایی تضمین شده
    return d_cls, h_raw, h_post, val_loss / (i + 1)
'''
'''
# second version 1-108 epoch
@torch.no_grad()
def validate(model, loader, loss_fn, epoch):
    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model
    net.eval()

    # ۱. آزاد سازی حافظه قبل از شروع برای جلوگیری از OOM
    torch.cuda.empty_cache()

    compute_hd = (epoch % cfg.hd_every == 0)
    hd_m_raw = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    hd_m_post = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    post_filter = KeepLargestConnectedComponent(applied_labels=[1, 2, 3], is_onehot=False)


    # به جای KeepLargest، از حذف اشیاء زیر ۵۰ پیکسل استفاده می‌کنیم
    # این کار باعث می‌شود خوشه‌های اصلی تومور حفظ شوند و فقط نویزهای تک‌-پیکسلی حذف شوند
    #post_filter = RemoveSmallObjects(min_size=1000, connectivity=1)

    dices, val_loss = [], 0.0

    # ۲. اضافه کردن enumerate برای کنترل تعداد نمونه‌ها
    for i, batch in enumerate(loader):
        # ۳. محدود کردن ولیدیشن به ۴۰ مورد (قابل تغییر در CFG) برای افزایش سرعت
        if i >= getattr(cfg, 'val_sample_limit', 40):
            break

        x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
        if y.ndim == 4: y = y.unsqueeze(1)
        y = y.long(); y[y == 4] = 3

        # ۴. استفاده از autocast برای کاهش مصرف حافظه گرافیکی (VRAM)
        with torch.amp.autocast("cuda", enabled=cfg.use_amp):
            logits = sliding_window_inference(
                x, cfg.roi_size, cfg.sw_batch_size,
                lambda z: net(z)["logits"],
                overlap=cfg.sw_overlap
            )
            val_loss += loss_fn(logits, y).item()

        #pred_raw = torch.argmax(logits, dim=1, keepdim=True)

        # ۵. پاک کردن logits بلافاصله پس از استفاده برای آزاد شدن فضا
        #del logits

        # --- بخش اصلاح شده شروع می‌شود ---

        # تبدیل Logits به احتمال (Probability)
        probs = torch.softmax(logits, dim=1)

        # حذف پیکسل‌های با اطمینان پایین (مثلا زیر 0.7)
        # این کار نویزهای پراکنده که مدل در موردشان مطمئن نیست را پاک می‌کند
        thresh = 0.6
        probs[probs < thresh] = 0

        # حالا Argmax می‌گیریم
        pred_raw = torch.argmax(probs, dim=1, keepdim=True)

        # پاک کردن متغیرهای سنگین
        del logits, probs

        raw_reg, true_reg = to_regions(pred_raw.squeeze(1)), to_regions(y.squeeze(1))

        inter = (raw_reg * true_reg).sum((2,3,4))
        union = raw_reg.sum((2,3,4)) + true_reg.sum((2,3,4))
        dices.append(torch.nan_to_num(2.*inter/(union+1e-5), 0.).mean(0).cpu().numpy())

        if compute_hd:
            # انتقال به CPU برای پردازش مورفولوژیک
            preds_cpu = pred_raw.squeeze(1).cpu()

            # اعمال فیلتر حذف اشیاء کوچک روی هر نمونه در بچ
            processed = torch.stack([post_filter(p.unsqueeze(0)).squeeze(0) for p in preds_cpu]).to(cfg.device)

            post_reg = to_regions(processed)
            hd_m_raw(raw_reg, true_reg)
            hd_m_post(post_reg, true_reg)
            del processed, preds_cpu # پاکسازی حافظه

    d_cls = np.mean(dices, axis=0)

    def safe_agg(metric):
        val = metric.aggregate()
        if isinstance(val, torch.Tensor): val = val.cpu().numpy()
        res = np.ones(3) * 100.0
        if np.isscalar(val): res[:] = val
        elif len(val) >= 3: res = val[:3]
        elif len(val) > 0: res[:len(val)] = val
        return np.nan_to_num(res, nan=100.0)

    h_raw = safe_agg(hd_m_raw) if compute_hd else np.array([-1.0]*3)
    h_post = safe_agg(hd_m_post) if compute_hd else np.array([-1.0]*3)

    # ۶. تخلیه نهایی حافظه قبل از بازگشت به آموزش
    torch.cuda.empty_cache()

    return d_cls, h_raw, h_post, val_loss / (i + 1) # تقسیم بر تعداد واقعی موارد پردازش شده
'''

# محدود کردن تعداد سمپل ولیدیشن
'''@torch.no_grad()
def validate(model, loader, loss_fn, epoch):
    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model
    net.eval()

    # --- آزاد سازی حافظه قبل از شروع ---
    torch.cuda.empty_cache()

    compute_hd = (epoch % cfg.hd_every == 0)
    hd_m_raw = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    hd_m_post = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
    post_filter = KeepLargestConnectedComponent(applied_labels=[1, 2, 3], is_onehot=False)

    dices, val_loss = [], 0.0
    for batch in loader:
        x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
        if y.ndim == 4: y = y.unsqueeze(1)
        y = y.long(); y[y == 4] = 3
        logits = sliding_window_inference(x, cfg.roi_size, cfg.sw_batch_size, lambda z: net(z)["logits"], overlap=cfg.sw_overlap)
        val_loss += loss_fn(logits, y).item()
        pred_raw = torch.argmax(logits, dim=1, keepdim=True)
        raw_reg, true_reg = to_regions(pred_raw.squeeze(1)), to_regions(y.squeeze(1))

        inter = (raw_reg * true_reg).sum((2,3,4))
        union = raw_reg.sum((2,3,4)) + true_reg.sum((2,3,4))
        dices.append(torch.nan_to_num(2.*inter/(union+1e-5), 0.).mean(0).cpu().numpy())

        if compute_hd:
            processed = torch.stack([post_filter(p.cpu().unsqueeze(0)).squeeze(0) for p in pred_raw.squeeze(1)]).to(cfg.device)
            post_reg = to_regions(processed)
            hd_m_raw(raw_reg, true_reg); hd_m_post(post_reg, true_reg)

    d_cls = np.mean(dices, axis=0)

    def safe_agg(metric):
        val = metric.aggregate()
        if isinstance(val, torch.Tensor): val = val.cpu().numpy()
        res = np.ones(3) * 100.0
        if np.isscalar(val): res[:] = val
        elif len(val) >= 3: res = val[:3]
        elif len(val) > 0: res[:len(val)] = val
        return np.nan_to_num(res, nan=100.0)

    h_raw = safe_agg(hd_m_raw) if compute_hd else np.array([-1.0]*3)
    h_post = safe_agg(hd_m_post) if compute_hd else np.array([-1.0]*3)
    return d_cls, h_raw, h_post, val_loss / len(loader)
'''
# =========================================================
# 4) اصلاح شده: کلاس Lookahead با شمارشگر مستقل
# =========================================================
class Lookahead(torch.optim.Optimizer):
    def __init__(self, optimizer, k=5, alpha=0.5):
        self.optimizer = optimizer
        self.k = k
        self.alpha = alpha
        self.param_groups = self.optimizer.param_groups
        self.state = self.optimizer.state
        self.defaults = self.optimizer.defaults
        self.lookahead_step = 0 # شمارشگر اختصاصی
        self.slow_weights = [[p.data.clone().detach() for p in group['params']]
                             for group in self.param_groups]

    def step(self, closure=None):
        loss = self.optimizer.step(closure)
        self.lookahead_step += 1

        if self.lookahead_step % self.k == 0:
            for group, slow_weights in zip(self.param_groups, self.slow_weights):
                for p, slow_p in zip(group['params'], slow_weights):
                    if p.grad is None: continue
                    slow_p.add_(p.data - slow_p, alpha=self.alpha)
                    p.data.copy_(slow_p)
        return loss

    def zero_grad(self, set_to_none=True):
        self.optimizer.zero_grad(set_to_none=set_to_none)

    def state_dict(self):
        sd = self.optimizer.state_dict()
        sd['lookahead_step'] = self.lookahead_step
        return sd

    def load_state_dict(self, state_dict):
        self.lookahead_step = state_dict.pop('lookahead_step', 0)
        self.optimizer.load_state_dict(state_dict)
        # 🚀 اضافه کردن این خط: همگام‌سازی وزن‌های کند با وزن‌های لود شده
        self.slow_weights = [[p.data.clone().detach() for p in group['params']]
                             for group in self.param_groups]

# =========================================================
# 5) تابع اصلی آموزش با Resume Logic هوشمند
# =========================================================
import json
def run_training(model, train_loader, val_loader):
    os.makedirs(cfg.save_dir, exist_ok=True)

    # ذخیره تنظیمات برای مراجعات بعدی
    with open(os.path.join(cfg.save_dir, 'config.json'), 'w') as f:
        json.dump(vars(cfg), f, indent=4, default=lambda x: str(x))

    # --- تنظیمات Loss بر اساس SOTA ---
    loss_fn = HybridLoss(cfg.device)
    # وزن‌دهی به کلاس‌ها (Background, TC, WT, ET)
    loss_fn.ce_weights = torch.tensor([0.1, 4.0, 8.0, 25.0], device=cfg.device)
    loss_fn.ce_loss = nn.CrossEntropyLoss(weight=loss_fn.ce_weights)

    # --- Optimizer & Lookahead ---
    base_optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    optimizer = Lookahead(base_optimizer, k=5, alpha=0.5)

    # --- Scheduler (Cosine Annealing with Warmup) ---
    '''def lr_lambda(epoch):
        current_ep = epoch + 1
        if current_ep <= cfg.warmup_epochs:
            return current_ep / cfg.warmup_epochs
        progress = (current_ep - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs)
        #return 0.5 * (1.0 + np.cos(np.pi * progress))
        return 1.0'''

    '''def lr_lambda(epoch):
        # از اپوک 108 به بعد، نرخ یادگیری را هر اپوک 10 درصد کم کن
        if epoch >= 108:
            exponent = epoch - 108
            return 0.9 ** exponent
        return 1.0'''

    '''def lr_lambda(epoch):
    # یک شوک کوچک در اپوک‌های ۱۱۲ تا ۱۱۵ برای بازنگری در وزن‌های ET
    if 112 <= epoch <= 115:
        return 2.5 # یعنی LR را به 2.5e-5 می‌رساند
    if epoch > 115:
        return 0.9 ** (epoch - 115) # دوباره کاهش ملایم
    return 1.0'''

    def lr_lambda(epoch):
      return 1.0 # یعنی LR همان 1e-5 ثابت بماند تا اپوک 150

    scheduler = torch.optim.lr_scheduler.LambdaLR(base_optimizer, lr_lambda=lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)

    # مقداردهی به مدل معلم (EMA)
    model.ema_teacher = EMATeacher(model, cfg.ema_alpha)

    # مسیرهای فایل
    last_ckpt_path = os.path.join(cfg.save_dir, "LAST_CHECKPOINT.pt")
    best_model_path = os.path.join(cfg.save_dir, "BEST_MODEL.pt")
    best_ema_path = os.path.join(cfg.save_dir, "BEST_MODEL_EMA.pt")
    csv_path = os.path.join(cfg.save_dir, "training_log_detailed.csv")

    start_epoch = 1
    best_score = -1.0


    # --- Resume Logic هوشمند ---
    ckpt_to_load = None
    if os.path.exists(last_ckpt_path):
        ckpt_to_load = last_ckpt_path
        print(f"🔄 Resuming from LAST checkpoint...")
    elif os.path.exists(best_model_path):
        ckpt_to_load = best_model_path
        print(f"⭐ Resuming from BEST model weights...")

    if ckpt_to_load:
        # اضافه کردن weights_only=False برای رفع ارور UnpicklingError
        ckpt = torch.load(ckpt_to_load, map_location=cfg.device, weights_only=False)
        raw_state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt




        # لود کردن وزن‌ها با متد تطبیق نام لایه‌ها
        curr_state = model.state_dict()
        new_state = {}
        matched = 0
        for k, v in raw_state.items():
            name = k.replace('module.', '').replace('model.', '')
            if name in curr_state:
                new_state[name] = v
                matched += 1
            elif f"encoder.{name}" in curr_state:
                new_state[f"encoder.{name}"] = v
                matched += 1

        model.load_state_dict(new_state, strict=False)
        print(f"✅ Loaded {matched}/{len(curr_state)} layers.")


        # لود کردن وضعیت‌های آموزش
        if isinstance(ckpt, dict):
            # اولویت اول: خواندن ایپاک از خودِ فایل (چه Best باشد چه Last)
            if "epoch" in ckpt:
                start_epoch = ckpt["epoch"] + 1
                best_score = ckpt.get("best_score", -1.0)
                print(f"📖 Found epoch in checkpoint: Resuming from {start_epoch}")
            else:
                # اگر فایل دیکشنری بود ولی عدد ایپاک نداشت (مثلاً فقط وزن بود)
                start_epoch = 1
                best_score = -1.0
                print("⚠️ Checkpoint has no epoch info. Starting from 1.")

            # لود کردن بهینه وضعیت Optimizer و Scheduler فقط در صورت وجود
            if ckpt_to_load == last_ckpt_path and "optimizer_state_dict" in ckpt:
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])
                if "scaler_state_dict" in ckpt:
                    scaler.load_state_dict(ckpt["scaler_state_dict"])

                # 🔥 --- تغییر مهم: شوک به نرخ یادگیری (LR Jump) --- 🔥
                # هدف: بیدار کردن مدل برای اصلاح کلاس ET
                '''new_lr = 3e-05  # مقدار پیشنهادی برای شوک
                for param_group in base_optimizer.param_groups:
                    param_group['lr'] = new_lr

                # بروزرسانی مقادیر پایه در scheduler برای جلوگیری از تداخل
                if hasattr(scheduler, 'base_lrs'):
                    scheduler.base_lrs = [new_lr for _ in base_optimizer.param_groups]
                print(f"🚀 LR Reset to: {new_lr:.2e} for fine-tuning boost!")
                # ---------------------------------------------------- 🔥
                '''
                # 🔥 --- تغییر نهایی: کاهش برای تثبیت (Annealing) --- 🔥
                new_lr = 2e-05  # کاهش از 3e-05 به 1e-05
                for param_group in base_optimizer.param_groups:
                    param_group['lr'] = new_lr

                if hasattr(scheduler, 'base_lrs'):
                    scheduler.base_lrs = [new_lr for _ in base_optimizer.param_groups]

                print(f"📉 LR Reduced to: {new_lr:.2e} for final stabilization.")

                if "scaler_state_dict" in ckpt:
                    scaler.load_state_dict(ckpt["scaler_state_dict"])


        else:
            # اگر فایل اصلاً دیکشنری نبود و فقط وزن خالص (OrderedDict) بود
            start_epoch = 1
            best_score = -1.0
            print("❗ Pure weights detected. Starting from epoch 1.")

    # همگام‌سازی‌های حیاتی
    model.ema_teacher.teacher.load_state_dict(model.state_dict())
    scheduler.last_epoch = start_epoch - 1
    # بروزرسانی LR فیزیکی در گروه‌های پارامتر
    '''
    current_lr_val = scheduler.get_last_lr()[0]
    for param_group in base_optimizer.param_groups:
        param_group['lr'] = current_lr_val


    print(f"📈 Ready! Start Epoch: {start_epoch} | Initial LR: {current_lr_val:.2e}")'''

    # همگام‌سازی‌های حیاتی
    model.ema_teacher.teacher.load_state_dict(model.state_dict())
    scheduler.last_epoch = start_epoch - 1

    # بروزرسانی LR: اگر ریست انجام شده، از آن استفاده کن، در غیر این صورت از شجولر
    current_lr_val = base_optimizer.param_groups[0]['lr']

    print(f"📈 Ready! Start Epoch: {start_epoch} | Initial LR: {current_lr_val:.2e}")



    # --- شروع لوپ آموزش ---
    print("\n" + "=" * 115)
    print(f"{'Ep':<3} | {'vLoss':<7} | {'Dice (WT/TC/ET/Mean)':<24} | {'Score':<6} | {'HD95 POST (W/T/E/M)':<20} | {'HD95 RAW (W/T/E/M)'}")
    print("-" * 115)

    for ep in range(start_epoch, cfg.epochs + 1):
        model.train()
        train_loss = 0.0

        for i, batch in enumerate(train_loader):
            x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
            if y.ndim == 4: y = y.unsqueeze(1)
            y = y.long(); y[y == 4] = 3

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                out = model(x)
                loss = loss_fn(out["logits"], y)
                if cfg.deep_supervision and "aux1" in out:
                    loss += cfg.ds_weights[0] * loss_fn(out["aux1"], y)
                    loss += cfg.ds_weights[1] * loss_fn(out["aux2"], y)

            scaler.scale(loss/cfg.accum_iter).backward()

            if (i+1) % cfg.accum_iter == 0 or (i+1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                model.ema_teacher.update(model)

            train_loss += loss.item()
        # 🚀 اضافه کردن این بخش دقیقاً قبل از ولیدیشن 🚀
        del loss, out, x, y  # پاک کردن متغیرهای سنگین آموزش
        import gc
        gc.collect()         # پاکسازی RAM سیستم
        torch.cuda.empty_cache() # پاکسازی VRAM


        # مرحله Validation و محاسبه متریک‌ها
        avg_train_loss = train_loss / len(train_loader)
        d_cls, h_raw, h_post, v_loss = validate(model, val_loader, loss_fn, ep)
        d_mean = np.mean(d_cls)
        score = (0.2 * d_cls[0] + 0.3 * d_cls[1] + 0.5 * d_cls[2])
        current_lr = base_optimizer.param_groups[0]['lr']

        # چاپ در کنسول
        fmt_hd_print = lambda arr: f"{arr[0]:.1f}/{arr[1]:.1f}/{arr[2]:.1f}/{np.mean(arr):.1f}" if arr[0] >= 0 else f"{'-/-/-/-':<20}"
        print(f"{ep:02d} | {v_loss:.4f} | {d_cls[0]:.3f}/{d_cls[1]:.3f}/{d_cls[2]:.3f}/{d_mean:.3f} | {score:.4f} | {fmt_hd_print(h_post):<20} | {fmt_hd_print(h_raw)}")

        # =========================================================
        # لاگ‌گذاری ۱۷ ستونه (اصلاح شده برای پایداری میانگین HD95)
        # =========================================================
        log_header = not os.path.exists(csv_path)

        # اصلاح نویز میانگین: اگر مقدار منفی بود (یعنی محاسبه نشده)، میانگین را هم 1- بگذار
        h_post_mean = np.mean(h_post) if h_post[0] >= 0 else -1.0
        h_raw_mean = np.mean(h_raw) if h_raw[0] >= 0 else -1.0

        with open(csv_path, "a", newline="") as f:
            writer = csv.writer(f)
            if log_header:
                writer.writerow([
                    "epoch", "lr", "train_loss", "val_loss",
                    "dice_WT", "dice_TC", "dice_ET", "dice_mean", "score",
                    "hd_post_WT", "hd_post_TC", "hd_post_ET", "hd_post_mean",
                    "hd_raw_WT", "hd_raw_TC", "hd_raw_ET", "hd_raw_mean"
                ])

            writer.writerow([
                ep, f"{current_lr:.2e}", f"{avg_train_loss:.4f}", f"{v_loss:.4f}",
                f"{d_cls[0]:.4f}", f"{d_cls[1]:.4f}", f"{d_cls[2]:.4f}", f"{d_mean:.4f}", f"{score:.4f}",
                f"{h_post[0]:.2f}", f"{h_post[1]:.2f}", f"{h_post[2]:.2f}", f"{h_post_mean:.2f}",
                f"{h_raw[0]:.2f}", f"{h_raw[1]:.2f}", f"{h_raw[2]:.2f}", f"{h_raw_mean:.2f}"
            ])


        # ۱. مدیریت Best Score و تعیین وضعیت مدل فعلی
        # =========================================================
        # ذخیره هوشمند چک‌پوینت (جایگزین از خط ۳۶۰ به بعد)
        # =========================================================

        # ۱. مدیریت Best Score و تعیین وضعیت مدل فعلی
        is_best = False
        if score > best_score:
            best_score = score
            is_best = True

        # ۲. ساخت پکیج چک‌پوینت (یکبار برای همه مصارف)
        checkpoint = {
            "epoch": ep,
            "best_score": best_score,
            "model_state_dict": model.state_dict(),
            "ema_teacher_state_dict": model.ema_teacher.teacher.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
        }

        # ۳. ذخیره آخرین وضعیت (همیشه آپدیت می‌شود برای Resume)
        torch.save(checkpoint, last_ckpt_path)

        # ۴. اگر رکورد جدیدی ثبت شد، در فایل BEST هم کپی کن
        if is_best:
            torch.save(checkpoint, best_model_path)

            # ذخیره نسخه معلم (EMA) برای تست نهایی
            checkpoint_ema = checkpoint.copy()
            checkpoint_ema["model_state_dict"] = model.ema_teacher.teacher.state_dict()
            torch.save(checkpoint_ema, best_ema_path)

            print(f"💾 ⭐ BEST MODEL UPDATED (Score: {score:.4f} at Epoch {ep})")

        # ۵. آپدیت نرخ یادگیری برای اپوک بعدی
        scheduler.step()

# =========================
# 6) EXECUTION
# =========================
model = model1.to(cfg.device)
run_training(model, train_loader, val_loader)

'''


IndentationError: unindent does not match any outer indentation level (<tokenize>, line 476)

In [ ]:
س

In [ ]:
"""import torch
print(torch.cuda.get_arch_list())
# اگر 'sm_120' را در لیست دیدید، یعنی مشکل حل شده است."""

['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']


In [ ]:
"""
# بعد از تغییر لاس و اضافه کردن و بعد از ایپاک 110
import os
import copy
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Tuple # اضافه شده برای رفع خطای احتمالی تایپ
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.metrics import HausdorffDistanceMetric
from monai.transforms import KeepLargestConnectedComponent
from monai.utils import set_determinism

# =========================
# 1) CONFIGURATION
# =========================

#cfg = CFG()
set_determinism(cfg.seed)

# =========================
# 2) EMA TEACHER & HYBRID LOSS
# =========================
class EMATeacher:
    def __init__(self, model, alpha=0.99):
        self.teacher = copy.deepcopy(model).eval()
        for p in self.teacher.parameters():
            p.requires_grad = False
        self.alpha = alpha

    @torch.no_grad()
    def update(self, student):
        for t_param, s_param in zip(self.teacher.parameters(), student.parameters()):
            t_param.data.mul_(self.alpha).add_(s_param.data, alpha=1.0 - self.alpha)

from monai.losses import DiceLoss

class HybridLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        # استفاده خالص و پایدار از دایس روی ریجن‌ها
        self.dice_loss = DiceLoss(
            include_background=False,
            to_onehot_y=False,
            softmax=False, # چون خودمان احتمالات را می‌سازیم
            squared_pred=True # برای پایداری بیشتر در آموزش
        )

    def forward(self, logits, y):
        # ۱. تبدیل تارگت به ریجن‌های BraTS
        y_wt = (y > 0).float()
        y_tc = ((y == 1) | (y == 3)).float()
        y_et = (y == 3).float()
        y_regions = torch.cat([y_wt, y_tc, y_et], dim=1)

        # ۲. تبدیل خروجی مدل به احتمالات و سپس ترکیب برای ریجن‌ها
        probs = torch.softmax(logits, dim=1)
        p_wt = probs[:, 1:].sum(dim=1, keepdim=True)
        p_tc = probs[:, 1:2] + probs[:, 3:4]
        p_et = probs[:, 3:4]
        preds_regions = torch.cat([p_wt, p_tc, p_et], dim=1)

        # ۳. محاسبه خطا مستقیماً روی ریجن‌ها
        return self.dice_loss(preds_regions, y_regions)
"""

@torch.no_grad()
def to_regions(lbl):
    wt = (lbl > 0).float()
    tc = ((lbl == 1) | (lbl == 3)).float()
    et = (lbl == 3).float()
    return torch.stack([wt, tc, et], dim=1)

from monai.losses import DiceLoss, FocalLoss

class HybridDiceFocalLoss(nn.Module):
    def __init__(self, device):
        super().__init__()

        # Dice برای regions (مثل قبل)
        self.dice_loss = DiceLoss(
            include_background=False,
            to_onehot_y=False,
            softmax=False,
            squared_pred=True
        )

        # Focal برای کلاس‌های سخت (ET)
        self.focal_loss = FocalLoss(
            include_background=False,  # BG را نادیده بگیر
            to_onehot_y=True,          # target را one-hot کن
            alpha=0.25,                # وزن کلاس‌های مثبت
            gamma=2.0                  # تمرکز روی hard examples
        )

    def forward(self, logits, y):
        # ۱. Region Dice (مثل Loss v2 فعلی)
        y_wt = (y > 0).float()
        y_tc = ((y == 1) | (y == 3)).float()
        y_et = (y == 3).float()
        y_regions = torch.cat([y_wt, y_tc, y_et], dim=1)

        probs = torch.softmax(logits, dim=1)
        p_wt = probs[:, 1:].sum(dim=1, keepdim=True)
        p_tc = probs[:, 1:2] + probs[:, 3:4]
        p_et = probs[:, 3:4]
        preds_regions = torch.cat([p_wt, p_tc, p_et], dim=1)

        dice = self.dice_loss(preds_regions, y_regions)

        # ۲. Focal Loss روی کلاس‌ها
        focal = self.focal_loss(logits, y)

        # ۳. ترکیب: Dice اولویت دارد (2×)
        return 2.0 * dice + focal
        """


# =========================
# 3) VALIDATE FUNCTION
# =========================
# خط ۱۳ یا حوالی آن را به این شکل تغییر بده:
from monai.transforms import RemoveSmallObjects  # جایگزین KeepLargestConnectedComponent

@torch.no_grad()
def validate(model, loader, loss_fn, epoch):
    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model
    net.eval()
    torch.cuda.empty_cache()

    compute_hd = (epoch % cfg.hd_every == 0)

    # تنظیم دقیق متریک برای تطابق با نتایج دستی
    # ما include_background را True میگذاریم و خودمان فقط ۳ کانال اصلی را میدهیم
    hd_metric = HausdorffDistanceMetric(include_background=True, percentile=95, reduction=None)

    #post_filter = KeepLargestConnectedComponent(applied_labels=[1, 2, 3], is_onehot=False)

    # هر توده زیر 32 پیکسل احتمالا نویز است و حذف آن HD95 را عالی و Dice را شفاف می‌کند.
    post_filter = RemoveSmallObjects(min_size=16, connectivity=1)

    dices, val_loss = [], 0.0

    for i, batch in enumerate(loader):
        if i >= getattr(cfg, 'val_sample_limit', 251): break

        x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)

        # --- گام جدید: دریافت ماسک مغز ---
        # این ماسک در لودر حذف نشده و حالا در دسترس است
        brain_mask = batch["brain_mask_tmp"].to(cfg.device)


        if y.ndim == 4: y = y.unsqueeze(1)
        y = y.long(); y[y == 4] = 3

        with torch.amp.autocast("cuda", enabled=cfg.use_amp):
            logits = sliding_window_inference(
                x, cfg.roi_size, cfg.sw_batch_size,
                lambda z: net(z)["logits"], overlap=cfg.sw_overlap
            )
            val_loss += loss_fn(logits, y).item()

        probs = torch.softmax(logits, dim=1)
        pred_raw = torch.argmax(probs, dim=1, keepdim=True)

        # --- گام طلایی: پاکسازی با ماسک مغز ---
        # تمام پیش‌بینی‌های خارج از فضای مغز صفر می‌شوند
        pred_raw = pred_raw * brain_mask

        # تبدیل به مناطق (WT, TC, ET) - دقیقاً مشابه تست دستی موفق
        raw_reg = to_regions(pred_raw.squeeze(1)).float()
        true_reg = to_regions(y.squeeze(1)).float()

        # محاسبه Dice (دستی و مطمئن)
        inter = (raw_reg * true_reg).sum((2,3,4))
        union = raw_reg.sum((2,3,4)) + true_reg.sum((2,3,4))
        dices.append(torch.nan_to_num(2.*inter/(union+1e-5), 0.).mean(0).cpu().numpy())

        if compute_hd:
            """# اعمال فیلتر برای حذف نویزهای تک پیکسلی
            preds_cpu = pred_raw.squeeze(1).cpu()
            processed = torch.stack([post_filter(p.unsqueeze(0)).squeeze(0) for p in preds_cpu]).to(cfg.device)
            post_reg = to_regions(processed).float()

            # تضمین ابعاد ۵ بعدی برای جلوگیری از گیج شدن MONAI
            if post_reg.ndim == 4: post_reg = post_reg.unsqueeze(0)
            if true_reg.ndim == 4: true_reg = true_reg.unsqueeze(0)

            # ارسال به بافر
            hd_metric(post_reg, true_reg)"""

            # ۱. اعمال فیلتر روی CPU (پیش از این هم روی CPU بود)
            preds_cpu = pred_raw.squeeze(1).cpu()
            processed = torch.stack([post_filter(p.unsqueeze(0)).squeeze(0) for p in preds_cpu])

            # ۲. تبدیل به ریجن‌ها (حالا post_reg و true_reg روی CPU هستند)
            post_reg = to_regions(processed).float() # خروجی به صورت خودکار روی CPU است
            # مطمئن شویم true_reg هم به CPU منتقل شده است
            true_reg_cpu = true_reg.cpu().float()

            # ۳. تضمین ابعاد ۵ بعدی
            if post_reg.ndim == 4: post_reg = post_reg.unsqueeze(0)
            if true_reg_cpu.ndim == 4: true_reg_cpu = true_reg_cpu.unsqueeze(0)

            # ۴. ارسال به متریک (متریک روی هر چه به آن بدهید محاسبه می‌کند)
            # چون هر دو Tensor روی CPU هستند، محاسبات روی CPU انجام می‌شود و ارور CUDA نمی‌گیرید
            hd_metric(post_reg, true_reg_cpu)

    d_cls = np.mean(dices, axis=0)

    # استخراج میانگین واقعی بدون تله‌های Inf/100
    if compute_hd:
        hd_vals = hd_metric.get_buffer()
        if isinstance(hd_vals, torch.Tensor): hd_vals = hd_vals.cpu().numpy()

        h_post = []
        for c in range(3):
            # حذف مقادیر پرت (فقط کیس‌هایی که تومور داشتند را حساب کن)
            valid_vals = hd_vals[:, c][np.isfinite(hd_vals[:, c])]
            if len(valid_vals) > 0:
                # میانگین کیس‌های معتبر (مانند 1.0 و 4.0 که در تست دستی دیدی)
                h_post.append(np.mean(valid_vals))
            else:
                h_post.append(100.0)
        hd_metric.reset()
    else:
        h_post = np.array([-1.0]*3)

    return d_cls, np.array(h_post), np.array(h_post), val_loss / (i + 1)


# =========================================================
# 4) اصلاح شده: کلاس Lookahead با شمارشگر مستقل
# =========================================================
class Lookahead(torch.optim.Optimizer):
    def __init__(self, optimizer, k=5, alpha=0.5):
        self.optimizer = optimizer
        self.k = k
        self.alpha = alpha
        self.param_groups = self.optimizer.param_groups
        self.state = self.optimizer.state
        self.defaults = self.optimizer.defaults
        self.lookahead_step = 0 # شمارشگر اختصاصی
        self.slow_weights = [[p.data.clone().detach() for p in group['params']]
                             for group in self.param_groups]

    def step(self, closure=None):
        loss = self.optimizer.step(closure)
        self.lookahead_step += 1

        if self.lookahead_step % self.k == 0:
            for group, slow_weights in zip(self.param_groups, self.slow_weights):
                for p, slow_p in zip(group['params'], slow_weights):
                    if p.grad is None: continue
                    slow_p.add_(p.data - slow_p, alpha=self.alpha)
                    p.data.copy_(slow_p)
        return loss

    def zero_grad(self, set_to_none=True):
        self.optimizer.zero_grad(set_to_none=set_to_none)

    def state_dict(self):
        sd = self.optimizer.state_dict()
        sd['lookahead_step'] = self.lookahead_step
        return sd

    def load_state_dict(self, state_dict):
        self.lookahead_step = state_dict.pop('lookahead_step', 0)
        self.optimizer.load_state_dict(state_dict)
        # 🚀 اضافه کردن این خط: همگام‌سازی وزن‌های کند با وزن‌های لود شده
        self.slow_weights = [[p.data.clone().detach() for p in group['params']]
                             for group in self.param_groups]

# =========================================================
# 5) تابع اصلی آموزش با Resume Logic هوشمند
# =========================================================
import json
def run_training(model, train_loader, val_loader):
    os.makedirs(cfg.save_dir, exist_ok=True)

    # ذخیره تنظیمات برای مراجعات بعدی
    with open(os.path.join(cfg.save_dir, 'config.json'), 'w') as f:
        json.dump(vars(cfg), f, indent=4, default=lambda x: str(x))

    '''# --- تنظیمات Loss بر اساس SOTA ---
    loss_fn = HybridLoss(cfg.device)
    # وزن‌دهی به کلاس‌ها (Background, TC, WT, ET)
    loss_fn.ce_weights = torch.tensor([0.1, 4.0, 8.0, 25.0], device=cfg.device)
    loss_fn.ce_loss = nn.CrossEntropyLoss(weight=loss_fn.ce_weights)
    '''
    # --- تنظیمات Loss بر اساس SOTA (مبتنی بر Region) ---
    loss_fn = HybridLoss(cfg.device)
    # خطوط وزن‌دهی دستی کاملاً حذف شدند تا مدل خودش ساختار تومور را یاد بگیرد


    # --- Optimizer & Lookahead ---
    base_optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    optimizer = Lookahead(base_optimizer, k=5, alpha=0.5)

    def lr_lambda(epoch):
      return 1.0 # یعنی LR همان 1e-5 ثابت بماند تا اپوک 150

    scheduler = torch.optim.lr_scheduler.LambdaLR(base_optimizer, lr_lambda=lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)

    # مقداردهی به مدل معلم (EMA)
    model.ema_teacher = EMATeacher(model, cfg.ema_alpha)

    # مسیرهای فایل
    last_ckpt_path = os.path.join(cfg.save_dir, "LAST_CHECKPOINT.pt")
    best_model_path = os.path.join(cfg.save_dir, "BEST_MODEL.pt")
    best_ema_path = os.path.join(cfg.save_dir, "BEST_MODEL_EMA.pt")
    csv_path = os.path.join(cfg.save_dir, "training_log_detailed.csv")

    start_epoch = 1
    best_score = -1.0


    # --- Resume Logic هوشمند ---
    ckpt_to_load = None
    if os.path.exists(last_ckpt_path):
        ckpt_to_load = last_ckpt_path
        print(f"🔄 Resuming from LAST checkpoint...")
    elif os.path.exists(best_model_path):
        ckpt_to_load = best_model_path
        print(f"⭐ Resuming from BEST model weights...")

    if ckpt_to_load:
        # اضافه کردن weights_only=False برای رفع ارور UnpicklingError
        ckpt = torch.load(ckpt_to_load, map_location=cfg.device, weights_only=False)
        raw_state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt




        # لود کردن وزن‌ها با متد تطبیق نام لایه‌ها
        curr_state = model.state_dict()
        new_state = {}
        matched = 0
        for k, v in raw_state.items():
            name = k.replace('module.', '').replace('model.', '')
            if name in curr_state:
                new_state[name] = v
                matched += 1
            elif f"encoder.{name}" in curr_state:
                new_state[f"encoder.{name}"] = v
                matched += 1

        model.load_state_dict(new_state, strict=False)
        print(f"✅ Loaded {matched}/{len(curr_state)} layers.")


        # لود کردن وضعیت‌های آموزش
        if isinstance(ckpt, dict):
            # اولویت اول: خواندن ایپاک از خودِ فایل (چه Best باشد چه Last)
            if "epoch" in ckpt:
                start_epoch = ckpt["epoch"] + 1
                best_score = ckpt.get("best_score", -1.0)
                print(f"📖 Found epoch in checkpoint: Resuming from {start_epoch}")
            else:
                # اگر فایل دیکشنری بود ولی عدد ایپاک نداشت (مثلاً فقط وزن بود)
                start_epoch = 1
                best_score = -1.0
                print("⚠️ Checkpoint has no epoch info. Starting from 1.")

            # لود کردن بهینه وضعیت Optimizer و Scheduler فقط در صورت وجود
            if ckpt_to_load == last_ckpt_path and "optimizer_state_dict" in ckpt:
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])
                if "scaler_state_dict" in ckpt:
                    scaler.load_state_dict(ckpt["scaler_state_dict"])


                # 🔥 --- تغییر نهایی: کاهش برای تثبیت (Annealing) --- 🔥
                new_lr = 1e-05  # کاهش از 3e-05 به 1e-05
                for param_group in base_optimizer.param_groups:
                    param_group['lr'] = new_lr

                if hasattr(scheduler, 'base_lrs'):
                    scheduler.base_lrs = [new_lr for _ in base_optimizer.param_groups]

                print(f"📉 LR Reduced to: {new_lr:.2e} for final stabilization.")

                if "scaler_state_dict" in ckpt:
                    scaler.load_state_dict(ckpt["scaler_state_dict"])


        else:
            # اگر فایل اصلاً دیکشنری نبود و فقط وزن خالص (OrderedDict) بود
            start_epoch = 1
            best_score = -1.0
            print("❗ Pure weights detected. Starting from epoch 1.")

    # همگام‌سازی‌های حیاتی
    model.ema_teacher.teacher.load_state_dict(model.state_dict())
    scheduler.last_epoch = start_epoch - 1
    # بروزرسانی LR فیزیکی در گروه‌های پارامتر
    '''
    current_lr_val = scheduler.get_last_lr()[0]
    for param_group in base_optimizer.param_groups:
        param_group['lr'] = current_lr_val


    print(f"📈 Ready! Start Epoch: {start_epoch} | Initial LR: {current_lr_val:.2e}")'''

    # همگام‌سازی‌های حیاتی
    model.ema_teacher.teacher.load_state_dict(model.state_dict())
    scheduler.last_epoch = start_epoch - 1

    # بروزرسانی LR: اگر ریست انجام شده، از آن استفاده کن، در غیر این صورت از شجولر
    current_lr_val = base_optimizer.param_groups[0]['lr']

    print(f"📈 Ready! Start Epoch: {start_epoch} | Initial LR: {current_lr_val:.2e}")



    # --- شروع لوپ آموزش ---
    print("\n" + "=" * 115)
    print(f"{'Ep':<3} | {'vLoss':<7} | {'Dice (WT/TC/ET/Mean)':<24} | {'Score':<6} | {'HD95 POST (W/T/E/M)':<20} | {'HD95 RAW (W/T/E/M)'}")
    print("-" * 115)

    for ep in range(start_epoch, cfg.epochs + 1):
        torch.cuda.empty_cache()
        model.train()
        train_loss = 0.0

        for i, batch in enumerate(train_loader):
            x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
            if y.ndim == 4: y = y.unsqueeze(1)
            #y = y.long(); y[y == 4] = 3
            y_cpu = y.cpu().long()
            y_cpu[y_cpu == 4] = 3
            y = y_cpu.to(cfg.device)


            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=False):
                out = model(x)
                loss = loss_fn(out["logits"], y)
                if cfg.deep_supervision and "aux1" in out:
                    loss += cfg.ds_weights[0] * loss_fn(out["aux1"], y)
                    loss += cfg.ds_weights[1] * loss_fn(out["aux2"], y)

            scaler.scale(loss/cfg.accum_iter).backward()

            if (i+1) % cfg.accum_iter == 0 or (i+1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                model.ema_teacher.update(model)

            train_loss += loss.item()
        # 🚀 اضافه کردن این بخش دقیقاً قبل از ولیدیشن 🚀
        del loss, out, x, y  # پاک کردن متغیرهای سنگین آموزش
        import gc
        gc.collect()         # پاکسازی RAM سیستم
        torch.cuda.empty_cache() # پاکسازی VRAM


        # مرحله Validation و محاسبه متریک‌ها
        avg_train_loss = train_loss / len(train_loader)
        d_cls, h_raw, h_post, v_loss = validate(model, val_loader, loss_fn, ep)
        d_mean = np.mean(d_cls)
        score = (0.2 * d_cls[0] + 0.3 * d_cls[1] + 0.5 * d_cls[2])
        current_lr = base_optimizer.param_groups[0]['lr']

        # چاپ در کنسول
        fmt_hd_print = lambda arr: f"{arr[0]:.1f}/{arr[1]:.1f}/{arr[2]:.1f}/{np.mean(arr):.1f}" if arr[0] >= 0 else f"{'-/-/-/-':<20}"
        print(f"{ep:02d} | {v_loss:.4f} | {d_cls[0]:.3f}/{d_cls[1]:.3f}/{d_cls[2]:.3f}/{d_mean:.3f} | {score:.4f} | {fmt_hd_print(h_post):<20} | {fmt_hd_print(h_raw)}")

        # =========================================================
        # لاگ‌گذاری ۱۷ ستونه (اصلاح شده برای پایداری میانگین HD95)
        # =========================================================
        log_header = not os.path.exists(csv_path)

        # اصلاح نویز میانگین: اگر مقدار منفی بود (یعنی محاسبه نشده)، میانگین را هم 1- بگذار
        h_post_mean = np.mean(h_post) if h_post[0] >= 0 else -1.0
        h_raw_mean = np.mean(h_raw) if h_raw[0] >= 0 else -1.0

        with open(csv_path, "a", newline="") as f:
            writer = csv.writer(f)
            if log_header:
                writer.writerow([
                    "epoch", "lr", "train_loss", "val_loss",
                    "dice_WT", "dice_TC", "dice_ET", "dice_mean", "score",
                    "hd_post_WT", "hd_post_TC", "hd_post_ET", "hd_post_mean",
                    "hd_raw_WT", "hd_raw_TC", "hd_raw_ET", "hd_raw_mean"
                ])

            writer.writerow([
                ep, f"{current_lr:.2e}", f"{avg_train_loss:.4f}", f"{v_loss:.4f}",
                f"{d_cls[0]:.4f}", f"{d_cls[1]:.4f}", f"{d_cls[2]:.4f}", f"{d_mean:.4f}", f"{score:.4f}",
                f"{h_post[0]:.2f}", f"{h_post[1]:.2f}", f"{h_post[2]:.2f}", f"{h_post_mean:.2f}",
                f"{h_raw[0]:.2f}", f"{h_raw[1]:.2f}", f"{h_raw[2]:.2f}", f"{h_raw_mean:.2f}"
            ])


        # ۱. مدیریت Best Score و تعیین وضعیت مدل فعلی
        # =========================================================
        # ذخیره هوشمند چک‌پوینت (جایگزین از خط ۳۶۰ به بعد)
        # =========================================================

        # ۱. مدیریت Best Score و تعیین وضعیت مدل فعلی
        is_best = False
        if score > best_score:
            best_score = score
            is_best = True

        # ۲. ساخت پکیج چک‌پوینت (یکبار برای همه مصارف)
        checkpoint = {
            "epoch": ep,
            "best_score": best_score,
            "model_state_dict": model.state_dict(),
            "ema_teacher_state_dict": model.ema_teacher.teacher.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
        }

        # ۳. ذخیره آخرین وضعیت (همیشه آپدیت می‌شود برای Resume)
        torch.save(checkpoint, last_ckpt_path)

        # ۴. اگر رکورد جدیدی ثبت شد، در فایل BEST هم کپی کن
        if is_best:
            torch.save(checkpoint, best_model_path)

            # ذخیره نسخه معلم (EMA) برای تست نهایی
            checkpoint_ema = checkpoint.copy()
            checkpoint_ema["model_state_dict"] = model.ema_teacher.teacher.state_dict()
            torch.save(checkpoint_ema, best_ema_path)

            print(f"💾 ⭐ BEST MODEL UPDATED (Score: {score:.4f} at Epoch {ep})")

        # ۵. آپدیت نرخ یادگیری برای اپوک بعدی
        scheduler.step()

# =========================
# 6) EXECUTION
# =========================

"""



In [ ]:
# ==============================================================================



In [ ]:
#  بعد از تغییر لاس و اضافه کردن و بعد از ایپاک 110
import os
import copy
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Tuple # اضافه شده برای رفع خطای احتمالی تایپ
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.metrics import HausdorffDistanceMetric
from monai.transforms import KeepLargestConnectedComponent
from monai.utils import set_determinism

# =========================
# 1) CONFIGURATION
# =========================

#cfg = CFG()
set_determinism(cfg.seed)

# =========================
# 2) EMA TEACHER & HYBRID LOSS
# =========================
class EMATeacher:
    def __init__(self, model, alpha=0.99):
        self.teacher = copy.deepcopy(model).eval()
        for p in self.teacher.parameters():
            p.requires_grad = False
        self.alpha = alpha

    @torch.no_grad()
    def update(self, student):
        for t_param, s_param in zip(self.teacher.parameters(), student.parameters()):
            t_param.data.mul_(self.alpha).add_(s_param.data, alpha=1.0 - self.alpha)

from monai.losses import DiceLoss
@torch.no_grad()
def to_regions(lbl):
    wt = (lbl > 0).float()
    tc = ((lbl == 1) | (lbl == 3)).float()
    et = (lbl == 3).float()
    return torch.stack([wt, tc, et], dim=1)


class HybridLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        # استفاده خالص و پایدار از دایس روی ریجن‌ها
        self.dice_loss = DiceLoss(
            include_background=False,
            to_onehot_y=False,
            softmax=False, # چون خودمان احتمالات را می‌سازیم
            squared_pred=True # برای پایداری بیشتر در آموزش
        )

    def forward(self, logits, y):
        # ۱. تبدیل تارگت به ریجن‌های BraTS
        y_wt = (y > 0).float()
        y_tc = ((y == 1) | (y == 3)).float()
        y_et = (y == 3).float()
        y_regions = torch.cat([y_wt, y_tc, y_et], dim=1)

        # ۲. تبدیل خروجی مدل به احتمالات و سپس ترکیب برای ریجن‌ها
        probs = torch.softmax(logits, dim=1)
        p_wt = probs[:, 1:].sum(dim=1, keepdim=True)
        p_tc = probs[:, 1:2] + probs[:, 3:4]
        p_et = probs[:, 3:4]
        preds_regions = torch.cat([p_wt, p_tc, p_et], dim=1)

        # ۳. محاسبه خطا مستقیماً روی ریجن‌ها
        return self.dice_loss(preds_regions, y_regions)


# =========================
# 3) VALIDATE FUNCTION
# =========================
# خط ۱۳ یا حوالی آن را به این شکل تغییر بده:
from monai.transforms import RemoveSmallObjects  # جایگزین KeepLargestConnectedComponent

@torch.no_grad()
def validate(model, loader, loss_fn, epoch):
    net = model.ema_teacher.teacher if hasattr(model, "ema_teacher") else model
    net.eval()
    torch.cuda.empty_cache()

    compute_hd = (epoch % cfg.hd_every == 0)

    # تنظیم دقیق متریک برای تطابق با نتایج دستی
    # ما include_background را True میگذاریم و خودمان فقط ۳ کانال اصلی را میدهیم
    hd_metric = HausdorffDistanceMetric(include_background=True, percentile=95, reduction=None)

    #post_filter = KeepLargestConnectedComponent(applied_labels=[1, 2, 3], is_onehot=False)

    # هر توده زیر 32 پیکسل احتمالا نویز است و حذف آن HD95 را عالی و Dice را شفاف می‌کند.
    post_filter = RemoveSmallObjects(min_size=16, connectivity=1)

    dices, val_loss = [], 0.0

    for i, batch in enumerate(loader):
        if i >= getattr(cfg, 'val_sample_limit', 251): break

        x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)

        # --- گام جدید: دریافت ماسک مغز ---
        # این ماسک در لودر حذف نشده و حالا در دسترس است
        brain_mask = batch["brain_mask_tmp"].to(cfg.device)


        if y.ndim == 4: y = y.unsqueeze(1)
        y = y.long(); y[y == 4] = 3

        with torch.amp.autocast("cuda", enabled=cfg.use_amp):
            logits = sliding_window_inference(
                x, cfg.roi_size, cfg.sw_batch_size,
                lambda z: net(z)["logits"], overlap=cfg.sw_overlap
            )
            val_loss += loss_fn(logits, y).item()

        probs = torch.softmax(logits, dim=1)
        pred_raw = torch.argmax(probs, dim=1, keepdim=True)

        # --- گام طلایی: پاکسازی با ماسک مغز ---
        # تمام پیش‌بینی‌های خارج از فضای مغز صفر می‌شوند
        pred_raw = pred_raw * brain_mask

        # تبدیل به مناطق (WT, TC, ET) - دقیقاً مشابه تست دستی موفق
        raw_reg = to_regions(pred_raw.squeeze(1)).float()
        true_reg = to_regions(y.squeeze(1)).float()

        # محاسبه Dice (دستی و مطمئن)
        inter = (raw_reg * true_reg).sum((2,3,4))
        union = raw_reg.sum((2,3,4)) + true_reg.sum((2,3,4))
        dices.append(torch.nan_to_num(2.*inter/(union+1e-5), 0.).mean(0).cpu().numpy())

        if compute_hd:

            # ۱. اعمال فیلتر روی CPU (پیش از این هم روی CPU بود)
            preds_cpu = pred_raw.squeeze(1).cpu()
            processed = torch.stack([post_filter(p.unsqueeze(0)).squeeze(0) for p in preds_cpu])

            # ۲. تبدیل به ریجن‌ها (حالا post_reg و true_reg روی CPU هستند)
            post_reg = to_regions(processed).float() # خروجی به صورت خودکار روی CPU است
            # مطمئن شویم true_reg هم به CPU منتقل شده است
            true_reg_cpu = true_reg.cpu().float()

            # ۳. تضمین ابعاد ۵ بعدی
            if post_reg.ndim == 4: post_reg = post_reg.unsqueeze(0)
            if true_reg_cpu.ndim == 4: true_reg_cpu = true_reg_cpu.unsqueeze(0)

            # ۴. ارسال به متریک (متریک روی هر چه به آن بدهید محاسبه می‌کند)
            # چون هر دو Tensor روی CPU هستند، محاسبات روی CPU انجام می‌شود و ارور CUDA نمی‌گیرید
            hd_metric(post_reg, true_reg_cpu)

    d_cls = np.mean(dices, axis=0)

    # استخراج میانگین واقعی بدون تله‌های Inf/100
    if compute_hd:
        hd_vals = hd_metric.get_buffer()
        if isinstance(hd_vals, torch.Tensor): hd_vals = hd_vals.cpu().numpy()

        h_post = []
        for c in range(3):
            # حذف مقادیر پرت (فقط کیس‌هایی که تومور داشتند را حساب کن)
            valid_vals = hd_vals[:, c][np.isfinite(hd_vals[:, c])]
            if len(valid_vals) > 0:
                # میانگین کیس‌های معتبر (مانند 1.0 و 4.0 که در تست دستی دیدی)
                h_post.append(np.mean(valid_vals))
            else:
                h_post.append(100.0)
        hd_metric.reset()
    else:
        h_post = np.array([-1.0]*3)

    return d_cls, np.array(h_post), np.array(h_post), val_loss / (i + 1)


# =========================================================
# 4) اصلاح شده: کلاس Lookahead با شمارشگر مستقل
# =========================================================
class Lookahead(torch.optim.Optimizer):
    def __init__(self, optimizer, k=5, alpha=0.5):
        self.optimizer = optimizer
        self.k = k
        self.alpha = alpha
        self.param_groups = self.optimizer.param_groups
        self.state = self.optimizer.state
        self.defaults = self.optimizer.defaults
        self.lookahead_step = 0 # شمارشگر اختصاصی
        self.slow_weights = [[p.data.clone().detach() for p in group['params']]
                             for group in self.param_groups]

    def step(self, closure=None):
        loss = self.optimizer.step(closure)
        self.lookahead_step += 1

        if self.lookahead_step % self.k == 0:
            for group, slow_weights in zip(self.param_groups, self.slow_weights):
                for p, slow_p in zip(group['params'], slow_weights):
                    if p.grad is None: continue
                    slow_p.add_(p.data - slow_p, alpha=self.alpha)
                    p.data.copy_(slow_p)
        return loss

    def zero_grad(self, set_to_none=True):
        self.optimizer.zero_grad(set_to_none=set_to_none)

    def state_dict(self):
        sd = self.optimizer.state_dict()
        sd['lookahead_step'] = self.lookahead_step
        return sd

    def load_state_dict(self, state_dict):
        self.lookahead_step = state_dict.pop('lookahead_step', 0)
        self.optimizer.load_state_dict(state_dict)
        # 🚀 اضافه کردن این خط: همگام‌سازی وزن‌های کند با وزن‌های لود شده
        self.slow_weights = [[p.data.clone().detach() for p in group['params']]
                             for group in self.param_groups]

# =========================================================
# 5) تابع اصلی آموزش با Resume Logic هوشمند
# =========================================================
import json
def run_training(model, train_loader, val_loader):
    os.makedirs(cfg.save_dir, exist_ok=True)

    # ذخیره تنظیمات برای مراجعات بعدی
    with open(os.path.join(cfg.save_dir, 'config.json'), 'w') as f:
        json.dump(vars(cfg), f, indent=4, default=lambda x: str(x))

    '''# --- تنظیمات Loss بر اساس SOTA ---
    loss_fn = HybridLoss(cfg.device)
    # وزن‌دهی به کلاس‌ها (Background, TC, WT, ET)
    loss_fn.ce_weights = torch.tensor([0.1, 4.0, 8.0, 25.0], device=cfg.device)
    loss_fn.ce_loss = nn.CrossEntropyLoss(weight=loss_fn.ce_weights)
    '''
    # --- تنظیمات Loss بر اساس SOTA (مبتنی بر Region) ---
    loss_fn = HybridLoss(cfg.device)
    # خطوط وزن‌دهی دستی کاملاً حذف شدند تا مدل خودش ساختار تومور را یاد بگیرد


    # --- Optimizer & Lookahead ---
    base_optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    optimizer = Lookahead(base_optimizer, k=5, alpha=0.5)

    def lr_lambda(epoch):
      return 1.0 # یعنی LR همان 1e-5 ثابت بماند تا اپوک 150

    scheduler = torch.optim.lr_scheduler.LambdaLR(base_optimizer, lr_lambda=lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)

    # مقداردهی به مدل معلم (EMA)
    model.ema_teacher = EMATeacher(model, cfg.ema_alpha)

    # مسیرهای فایل
    last_ckpt_path = os.path.join(cfg.save_dir, "LAST_CHECKPOINT.pt")
    best_model_path = os.path.join(cfg.save_dir, "BEST_MODEL.pt")
    best_ema_path = os.path.join(cfg.save_dir, "BEST_MODEL_EMA.pt")
    csv_path = os.path.join(cfg.save_dir, "training_log_detailed.csv")

    start_epoch = 1
    best_score = -1.0


    # --- Resume Logic هوشمند ---
    ckpt_to_load = None
    if os.path.exists(last_ckpt_path):
        ckpt_to_load = last_ckpt_path
        print(f"🔄 Resuming from LAST checkpoint...")
    elif os.path.exists(best_model_path):
        ckpt_to_load = best_model_path
        print(f"⭐ Resuming from BEST model weights...")

    if ckpt_to_load:
        # اضافه کردن weights_only=False برای رفع ارور UnpicklingError
        ckpt = torch.load(ckpt_to_load, map_location=cfg.device, weights_only=False)
        raw_state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt




        # لود کردن وزن‌ها با متد تطبیق نام لایه‌ها
        curr_state = model.state_dict()
        new_state = {}
        matched = 0
        for k, v in raw_state.items():
            name = k.replace('module.', '').replace('model.', '')
            if name in curr_state:
                new_state[name] = v
                matched += 1
            elif f"encoder.{name}" in curr_state:
                new_state[f"encoder.{name}"] = v
                matched += 1

        model.load_state_dict(new_state, strict=False)
        print(f"✅ Loaded {matched}/{len(curr_state)} layers.")


        # لود کردن وضعیت‌های آموزش
        if isinstance(ckpt, dict):
            # اولویت اول: خواندن ایپاک از خودِ فایل (چه Best باشد چه Last)
            if "epoch" in ckpt:
                start_epoch = ckpt["epoch"] + 1
                best_score = ckpt.get("best_score", -1.0)
                print(f"📖 Found epoch in checkpoint: Resuming from {start_epoch}")
            else:
                # اگر فایل دیکشنری بود ولی عدد ایپاک نداشت (مثلاً فقط وزن بود)
                start_epoch = 1
                best_score = -1.0
                print("⚠️ Checkpoint has no epoch info. Starting from 1.")

            # لود کردن بهینه وضعیت Optimizer و Scheduler فقط در صورت وجود
            if ckpt_to_load == last_ckpt_path and "optimizer_state_dict" in ckpt:
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])
                if "scaler_state_dict" in ckpt:
                    scaler.load_state_dict(ckpt["scaler_state_dict"])


                # 🔥 --- تغییر نهایی: کاهش برای تثبیت (Annealing) --- 🔥
                new_lr = 5e-06  # کاهش از 3e-05 به 1e-05
                for param_group in base_optimizer.param_groups:
                    param_group['lr'] = new_lr

                if hasattr(scheduler, 'base_lrs'):
                    scheduler.base_lrs = [new_lr for _ in base_optimizer.param_groups]

                print(f"📉 LR Reduced to: {new_lr:.2e} for final stabilization.")

                if "scaler_state_dict" in ckpt:
                    scaler.load_state_dict(ckpt["scaler_state_dict"])


        else:
            # اگر فایل اصلاً دیکشنری نبود و فقط وزن خالص (OrderedDict) بود
            start_epoch = 1
            best_score = -1.0
            print("❗ Pure weights detected. Starting from epoch 1.")

    # همگام‌سازی‌های حیاتی
    model.ema_teacher.teacher.load_state_dict(model.state_dict())
    scheduler.last_epoch = start_epoch - 1
    # بروزرسانی LR فیزیکی در گروه‌های پارامتر
    '''
    current_lr_val = scheduler.get_last_lr()[0]
    for param_group in base_optimizer.param_groups:
        param_group['lr'] = current_lr_val


    print(f"📈 Ready! Start Epoch: {start_epoch} | Initial LR: {current_lr_val:.2e}")'''

    # همگام‌سازی‌های حیاتی
    model.ema_teacher.teacher.load_state_dict(model.state_dict())
    scheduler.last_epoch = start_epoch - 1

    # بروزرسانی LR: اگر ریست انجام شده، از آن استفاده کن، در غیر این صورت از شجولر
    current_lr_val = base_optimizer.param_groups[0]['lr']

    print(f"📈 Ready! Start Epoch: {start_epoch} | Initial LR: {current_lr_val:.2e}")



    # --- شروع لوپ آموزش ---
    print("\n" + "=" * 115)
    print(f"{'Ep':<3} | {'vLoss':<7} | {'Dice (WT/TC/ET/Mean)':<24} | {'Score':<6} | {'HD95 POST (W/T/E/M)':<20} | {'HD95 RAW (W/T/E/M)'}")
    print("-" * 115)

    for ep in range(start_epoch, cfg.epochs + 1):
        torch.cuda.empty_cache()
        model.train()
        train_loss = 0.0

        for i, batch in enumerate(train_loader):
            x, y = batch["image"].to(cfg.device), batch["label"].to(cfg.device)
            if y.ndim == 4: y = y.unsqueeze(1)
            #y = y.long(); y[y == 4] = 3
            y_cpu = y.cpu().long()
            y_cpu[y_cpu == 4] = 3
            y = y_cpu.to(cfg.device)


            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=False):
                out = model(x)
                loss = loss_fn(out["logits"], y)
                if cfg.deep_supervision and "aux1" in out:
                    loss += cfg.ds_weights[0] * loss_fn(out["aux1"], y)
                    loss += cfg.ds_weights[1] * loss_fn(out["aux2"], y)

            scaler.scale(loss/cfg.accum_iter).backward()

            if (i+1) % cfg.accum_iter == 0 or (i+1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                model.ema_teacher.update(model)

            train_loss += loss.item()
        # 🚀 اضافه کردن این بخش دقیقاً قبل از ولیدیشن 🚀
        del loss, out, x, y  # پاک کردن متغیرهای سنگین آموزش
        import gc
        gc.collect()         # پاکسازی RAM سیستم
        torch.cuda.empty_cache() # پاکسازی VRAM


        # مرحله Validation و محاسبه متریک‌ها
        avg_train_loss = train_loss / len(train_loader)
        d_cls, h_raw, h_post, v_loss = validate(model, val_loader, loss_fn, ep)
        d_mean = np.mean(d_cls)
        score = (0.2 * d_cls[0] + 0.3 * d_cls[1] + 0.5 * d_cls[2])
        current_lr = base_optimizer.param_groups[0]['lr']

        # چاپ در کنسول
        fmt_hd_print = lambda arr: f"{arr[0]:.1f}/{arr[1]:.1f}/{arr[2]:.1f}/{np.mean(arr):.1f}" if arr[0] >= 0 else f"{'-/-/-/-':<20}"
        print(f"{ep:02d} | {v_loss:.4f} | {d_cls[0]:.3f}/{d_cls[1]:.3f}/{d_cls[2]:.3f}/{d_mean:.3f} | {score:.4f} | {fmt_hd_print(h_post):<20} | {fmt_hd_print(h_raw)}")

        # =========================================================
        # لاگ‌گذاری ۱۷ ستونه (اصلاح شده برای پایداری میانگین HD95)
        # =========================================================
        log_header = not os.path.exists(csv_path)

        # اصلاح نویز میانگین: اگر مقدار منفی بود (یعنی محاسبه نشده)، میانگین را هم 1- بگذار
        h_post_mean = np.mean(h_post) if h_post[0] >= 0 else -1.0
        h_raw_mean = np.mean(h_raw) if h_raw[0] >= 0 else -1.0

        with open(csv_path, "a", newline="") as f:
            writer = csv.writer(f)
            if log_header:
                writer.writerow([
                    "epoch", "lr", "train_loss", "val_loss",
                    "dice_WT", "dice_TC", "dice_ET", "dice_mean", "score",
                    "hd_post_WT", "hd_post_TC", "hd_post_ET", "hd_post_mean",
                    "hd_raw_WT", "hd_raw_TC", "hd_raw_ET", "hd_raw_mean"
                ])

            writer.writerow([
                ep, f"{current_lr:.2e}", f"{avg_train_loss:.4f}", f"{v_loss:.4f}",
                f"{d_cls[0]:.4f}", f"{d_cls[1]:.4f}", f"{d_cls[2]:.4f}", f"{d_mean:.4f}", f"{score:.4f}",
                f"{h_post[0]:.2f}", f"{h_post[1]:.2f}", f"{h_post[2]:.2f}", f"{h_post_mean:.2f}",
                f"{h_raw[0]:.2f}", f"{h_raw[1]:.2f}", f"{h_raw[2]:.2f}", f"{h_raw_mean:.2f}"
            ])


        # ۱. مدیریت Best Score و تعیین وضعیت مدل فعلی
        # =========================================================
        # ذخیره هوشمند چک‌پوینت (جایگزین از خط ۳۶۰ به بعد)
        # =========================================================

        # ۱. مدیریت Best Score و تعیین وضعیت مدل فعلی
        is_best = False
        if score > best_score:
            best_score = score
            is_best = True

        # ۲. ساخت پکیج چک‌پوینت (یکبار برای همه مصارف)
        checkpoint = {
            "epoch": ep,
            "best_score": best_score,
            "model_state_dict": model.state_dict(),
            "ema_teacher_state_dict": model.ema_teacher.teacher.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
        }

        # ۳. ذخیره آخرین وضعیت (همیشه آپدیت می‌شود برای Resume)
        torch.save(checkpoint, last_ckpt_path)

        # ۴. اگر رکورد جدیدی ثبت شد، در فایل BEST هم کپی کن
        if is_best:
            torch.save(checkpoint, best_model_path)

            # ذخیره نسخه معلم (EMA) برای تست نهایی
            checkpoint_ema = checkpoint.copy()
            checkpoint_ema["model_state_dict"] = model.ema_teacher.teacher.state_dict()
            torch.save(checkpoint_ema, best_ema_path)

            print(f"💾 ⭐ BEST MODEL UPDATED (Score: {score:.4f} at Epoch {ep})")

        # ۵. آپدیت نرخ یادگیری برای اپوک بعدی
        scheduler.step()

# =========================
# 6) EXECUTION
# =========================


In [ ]:

model = model1.to(cfg.device)
run_training(model, train_loader, val_loader)


🔄 Resuming from LAST checkpoint...
✅ Loaded 1037/1037 layers.
📖 Found epoch in checkpoint: Resuming from 136
📉 LR Reduced to: 5.00e-06 for final stabilization.
📈 Ready! Start Epoch: 136 | Initial LR: 5.00e-06

Ep  | vLoss   | Dice (WT/TC/ET/Mean)     | Score  | HD95 POST (W/T/E/M)  | HD95 RAW (W/T/E/M)
-------------------------------------------------------------------------------------------------------------------
136 | 0.1531 | 0.912/0.877/0.771/0.853 | 0.8310 | -/-/-/-              | -/-/-/-             
137 | 0.1532 | 0.912/0.876/0.771/0.853 | 0.8310 | -/-/-/-              | -/-/-/-             


KeyboardInterrupt: 

In [ ]:

135 | 0.1541 | 0.910/0.875/0.770/0.852 | 0.8298 | 7.8/5.7/5.2/6.2      | 7.8/5.7/5.2/6.2

In [ ]:
"""

@torch.no_grad()
def to_regions(lbl):
    wt = (lbl > 0).float()
    tc = ((lbl == 1) | (lbl == 3)).float()
    et = (lbl == 3).float()
    return torch.stack([wt, tc, et], dim=1)

from monai.losses import DiceLoss, FocalLoss

class HybridDiceFocalLoss(nn.Module):
    def __init__(self, device):
        super().__init__()

        # Dice برای regions (مثل قبل)
        self.dice_loss = DiceLoss(
            include_background=False,
            to_onehot_y=False,
            softmax=False,
            squared_pred=True
        )

        # Focal برای کلاس‌های سخت (ET)
        self.focal_loss = FocalLoss(
            include_background=False,  # BG را نادیده بگیر
            to_onehot_y=True,          # target را one-hot کن
            alpha=0.25,                # وزن کلاس‌های مثبت
            gamma=2.0                  # تمرکز روی hard examples
        )

    def forward(self, logits, y):
        # ۱. Region Dice (مثل Loss v2 فعلی)
        y_wt = (y > 0).float()
        y_tc = ((y == 1) | (y == 3)).float()
        y_et = (y == 3).float()
        y_regions = torch.cat([y_wt, y_tc, y_et], dim=1)

        probs = torch.softmax(logits, dim=1)
        p_wt = probs[:, 1:].sum(dim=1, keepdim=True)
        p_tc = probs[:, 1:2] + probs[:, 3:4]
        p_et = probs[:, 3:4]
        preds_regions = torch.cat([p_wt, p_tc, p_et], dim=1)

        dice = self.dice_loss(preds_regions, y_regions)

        # ۲. Focal Loss روی کلاس‌ها
        focal = self.focal_loss(logits, y)

        # ۳. ترکیب: Dice اولویت دارد (2×)
        return 2.0 * dice + focal
"""

In [ ]:
'''
🔄 Resuming from LAST checkpoint...
🚀 Weights recovered. Starting from epoch 5

===================================================================================================================
Ep  | vLoss   | Dice (WT/TC/ET/Mean)     | Score  | HD95 POST (W/T/E/M)  | HD95 RAW (W/T/E/M)
-------------------------------------------------------------------------------------------------------------------

05 | 2.0550 | 0.041/0.015/0.008/0.022 | 0.0170 | 80.8/100.0/100.0/93.6 | 97.3/100.0/100.0/99.1
06 | 1.7104 | 0.503/0.238/0.176/0.306 | 0.2599 | -/-/-/-              | -/-/-/-
07 | 1.3557 | 0.680/0.562/0.468/0.570 | 0.5384 | -/-/-/-              | -/-/-/-
08 | 1.2117 | 0.717/0.620/0.533/0.624 | 0.5961 | -/-/-/-              | -/-/-/-
09 | 1.1232 | 0.744/0.643/0.567/0.651 | 0.6253 | -/-/-/-              | -/-/-/-
10 | 1.0553 | 0.756/0.649/0.584/0.663 | 0.6380 | 18.1/100.0/100.0/72.7 | 23.6/100.0/100.0/74.5
💾 Best Models Updated (Score: 0.6380)
11 | 0.9917 | 0.770/0.656/0.588/0.671 | 0.6446 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.6446)
12 | 0.9283 | 0.791/0.683/0.608/0.694 | 0.6673 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.6673)
13 | 0.8664 | 0.807/0.711/0.630/0.716 | 0.6896 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.6896)
14 | 0.8104 | 0.828/0.730/0.652/0.737 | 0.7105 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.7105)
15 | 0.7601 | 0.837/0.742/0.661/0.746 | 0.7203 | 13.3/100.0/100.0/71.1 | 13.8/100.0/100.0/71.3
💾 Best Models Updated (Score: 0.7203)
16 | 0.7212 | 0.842/0.751/0.669/0.754 | 0.7280 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.7280)
17 | 0.6819 | 0.850/0.762/0.678/0.764 | 0.7379 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.7379)
18 | 0.6481 | 0.857/0.775/0.689/0.774 | 0.7486 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.7486)
19 | 0.6204 | 0.856/0.779/0.693/0.776 | 0.7516 | -/-/-/-              | -/-/-/-
💾 Best Models Updated (Score: 0.7516)

22 | 0.5293 | 0.859/0.786/0.687/0.777 | 0.7510 | -/-/-/-              | -/-/-/-
23 | 0.5203 | 0.860/0.794/0.697/0.784 | 0.7586 | -/-/-/-              | -/-/-/-
24 | 0.5066 | 0.868/0.803/0.707/0.793 | 0.7680 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.7680 at Epoch 24)
25 | 0.5005 | 0.864/0.801/0.709/0.791 | 0.7678 | 12.8/100.0/100.0/70.9 | 11.2/100.0/100.0/70.4
26 | 0.4881 | 0.869/0.808/0.715/0.797 | 0.7736 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.7736 at Epoch 26)
27 | 0.4824 | 0.873/0.807/0.720/0.800 | 0.7768 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.7768 at Epoch 27)
28 | 0.4751 | 0.876/0.808/0.720/0.802 | 0.7778 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.7778 at Epoch 28)
29 | 0.4597 | 0.865/0.815/0.710/0.797 | 0.7727 | -/-/-/-              | -/-/-/-
30 | 0.4532 | 0.875/0.823/0.720/0.806 | 0.7818 | 11.2/100.0/100.0/70.4 | 8.6/100.0/100.0/69.5
💾 ⭐ BEST MODEL UPDATED (Score: 0.7818 at Epoch 30)
31 | 0.4466 | 0.883/0.828/0.727/0.813 | 0.7888 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.7888 at Epoch 31)

51 | 0.3847 | 0.905/0.856/0.745/0.835 | 0.8104 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.8104 at Epoch 51)
52 | 0.3828 | 0.907/0.857/0.748/0.837 | 0.8123 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.8123 at Epoch 52)
53 | 0.3821 | 0.907/0.857/0.748/0.837 | 0.8123 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.8123 at Epoch 53)
54 | 0.3828 | 0.905/0.856/0.749/0.837 | 0.8122 | -/-/-/-              | -/-/-/-
55 | 0.3851 | 0.905/0.852/0.746/0.835 | 0.8098 | 6.2/100.0/100.0/68.7 | 6.2/100.0/100.0/68.7
56 | 0.3846 | 0.907/0.851/0.747/0.835 | 0.8101 | -/-/-/-              | -/-/-/-
57 | 0.3851 | 0.907/0.849/0.748/0.835 | 0.8103 | -/-/-/-              | -/-/-/-
58 | 0.3852 | 0.907/0.850/0.751/0.836 | 0.8120 | -/-/-/-              | -/-/-/-
59 | 0.3809 | 0.907/0.853/0.749/0.836 | 0.8118 | -/-/-/-              | -/-/-/-
60 | 0.3791 | 0.907/0.855/0.750/0.837 | 0.8129 | 6.1/100.0/100.0/68.7 | 6.1/100.0/100.0/68.7
💾 ⭐ BEST MODEL UPDATED (Score: 0.8129 at Epoch 60)
61 | 0.3766 | 0.908/0.857/0.749/0.838 | 0.8132 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.8132 at Epoch 61)
62 | 0.3774 | 0.908/0.856/0.749/0.837 | 0.8129 | -/-/-/-              | -/-/-/-
63 | 0.3773 | 0.908/0.854/0.748/0.837 | 0.8121 | -/-/-/-              | -/-/-/-
64 | 0.3756 | 0.907/0.856/0.750/0.838 | 0.8134 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.8134 at Epoch 64)
65 | 0.3753 | 0.907/0.858/0.751/0.839 | 0.8141 | 7.4/100.0/100.0/69.1 | 7.4/100.0/100.0/69.1
💾 ⭐ BEST MODEL UPDATED (Score: 0.8141 at Epoch 65)
66 | 0.3745 | 0.908/0.857/0.750/0.839 | 0.8139 | -/-/-/-              | -/-/-/-
67 | 0.3739 | 0.909/0.857/0.750/0.839 | 0.8140 | -/-/-/-              | -/-/-/-
68 | 0.3745 | 0.910/0.856/0.750/0.838 | 0.8137 | -/-/-/-              | -/-/-/-

70 | 0.3873 | 0.912/0.858/0.752/0.841 | 0.8159 | 6.2/100.0/100.0/68.7 | 6.2/100.0/100.0/68.7
💾 ⭐ BEST MODEL UPDATED (Score: 0.8159 at Epoch 70)
71 | 0.3853 | 0.912/0.860/0.753/0.842 | 0.8169 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.8169 at Epoch 71)
72 | 0.3861 | 0.911/0.860/0.754/0.842 | 0.8173 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.8173 at Epoch 72)
73 | 0.3870 | 0.911/0.858/0.752/0.841 | 0.8158 | -/-/-/-              | -/-/-/-
74 | 0.3858 | 0.912/0.859/0.753/0.841 | 0.8167 | -/-/-/-              | -/-/-/-
75 | 0.3839 | 0.911/0.861/0.755/0.842 | 0.8178 | 6.3/100.0/100.0/68.8 | 6.3/100.0/100.0/68.8
💾 ⭐ BEST MODEL UPDATED (Score: 0.8178 at Epoch 75)
76 | 0.3831 | 0.911/0.862/0.756/0.843 | 0.8186 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.8186 at Epoch 76)
77 | 0.3825 | 0.911/0.861/0.755/0.842 | 0.8178 | -/-/-/-              | -/-/-/-
78 | 0.3827 | 0.912/0.861/0.755/0.843 | 0.8183 | -/-/-/-              | -/-/-/-
79 | 0.3808 | 0.912/0.862/0.754/0.843 | 0.8184 | -/-/-/-              | -/-/-/-
80 | 0.3802 | 0.913/0.863/0.754/0.843 | 0.8186 | 6.1/100.0/100.0/68.7 | 6.1/100.0/100.0/68.7
81 | 0.3791 | 0.912/0.863/0.755/0.843 | 0.8188 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.8188 at Epoch 81)
82 | 0.3789 | 0.912/0.864/0.755/0.843 | 0.8189 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.8189 at Epoch 82)

87 | 0.3792 | 0.914/0.862/0.754/0.843 | 0.8186 | -/-/-/-              | -/-/-/-
88 | 0.3792 | 0.914/0.862/0.755/0.844 | 0.8188 | -/-/-/-              | -/-/-/-
89 | 0.3784 | 0.914/0.863/0.755/0.844 | 0.8193 | -/-/-/-              | -/-/-/-

90 | 0.4475 | 0.914/0.873/0.769/0.852 | 0.8290 | 6.0/100.0/100.0/68.7 | 6.1/100.0/100.0/68.7
💾 ⭐ BEST MODEL UPDATED (Score: 0.8290 at Epoch 90)
91 | 0.4448 | 0.910/0.870/0.766/0.849 | 0.8260 | -/-/-/-              | -/-/-/-
92 | 0.4425 | 0.908/0.871/0.765/0.848 | 0.8253 | -/-/-/-              | -/-/-/-
93 | 0.4425 | 0.904/0.870/0.765/0.846 | 0.8243 | -/-/-/-              | -/-/-/-

101 | 0.7379 | 0.914/0.871/0.780/0.855 | 0.8345 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.8345 at Epoch 101)
102 | 0.7348 | 0.915/0.873/0.782/0.856 | 0.8357 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.8357 at Epoch 102)
103 | 0.7330 | 0.916/0.873/0.782/0.857 | 0.8363 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.8363 at Epoch 103)
104 | 0.7354 | 0.915/0.873/0.783/0.857 | 0.8364 | -/-/-/-              | -/-/-/-
💾 ⭐ BEST MODEL UPDATED (Score: 0.8364 at Epoch 104)

105 | 0.7378 | 0.916/0.872/0.782/0.857 | 0.8360 | 7.8/100.0/100.0/69.3 | 5.7/100.0/100.0/68.6
106 | 0.7378 | 0.916/0.872/0.783/0.857 | 0.8360 | -/-/-/-              | -/-/-/-

'''

In [ ]:
# Test

In [ ]:
"""# ═══════════════════════════════════════════════════════════════════════════════
# کد تست نهایی بهینه‌شده برای A100 80GB - نصف داده‌های ولیدیشن
# ═══════════════════════════════════════════════════════════════════════════════

import os
import glob
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from scipy.ndimage import label as connected_components

from monai.data import Dataset, DataLoader
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    NormalizeIntensityd, ConcatItemsd, EnsureTyped
)
from monai.inferers import sliding_window_inference
from monai.transforms import SaveImage

# ═══════════════════════════════════════════════════════════════════════════════
# تنظیمات بهینه برای A100 80GB
# ═══════════════════════════════════════════════════════════════════════════════

DATA_ROOT = "/content/data/BraTS/dataset"

EMA_PATHS = [
    "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead_v2/BEST_MODEL.pt",
    "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead_v2/LAST_CHECKPOINT.pt"
]

OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/testi/BraTS_Final_Thesis_Results_Half"

class TestCFG:
    roi_size = (160, 160, 160)
    sw_batch_size = 32
    sw_overlap = 0.70
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed = 42
    use_full_tta = True
    min_component_size = 0.01

    num_workers = 12
    prefetch_factor = 4
    persistent_workers = True

    use_half_data = True
    patch_size = (160, 160, 160)  # ✅ اصلاح: باید با roi_size یکسان باشد


tcfg = TestCFG()

# ═══════════════════════════════════════════════════════════════════════════════
# دیتا - با گزینه نصف کردن
# ═══════════════════════════════════════════════════════════════════════════════

def get_val_files(root, split=0.8, seed=42, use_half=False):
    import random
    cases = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
    random.Random(seed).shuffle(cases)
    split_idx = int(len(cases) * split)
    val_cases = cases[split_idx:]

    if use_half:
        val_cases = val_cases[:len(val_cases)//2]
        print(f"🔄 Using HALF of validation data: {len(val_cases)} cases")
    else:
        print(f"📊 Using FULL validation data: {len(val_cases)} cases")

    val_files = []
    for case in val_cases:
        case_dir = os.path.join(root, case)
        f = lambda suffix: glob.glob(os.path.join(case_dir, f"*_{suffix}.nii.gz"))[0]
        val_files.append({
            "case_id": case,
            "t1": f("t1"),
            "t1ce": f("t1ce"),
            "t2": f("t2"),
            "flair": f("flair"),
            "seg": f("seg")
        })
    return val_files

test_files = get_val_files(DATA_ROOT, split=0.8, seed=tcfg.seed, use_half=tcfg.use_half_data)

test_transforms = Compose([
    LoadImaged(keys=["t1", "t1ce", "t2", "flair", "seg"], image_only=False),  # ✅ اصلاح: image_only=False برای metadata
    EnsureChannelFirstd(keys=["t1", "t1ce", "t2", "flair", "seg"]),
    Orientationd(keys=["t1", "t1ce", "t2", "flair", "seg"], axcodes="RAS"),
    NormalizeIntensityd(keys=["t1", "t1ce", "t2", "flair"], nonzero=True, channel_wise=True),
    ConcatItemsd(keys=["t1", "t1ce", "t2", "flair"], name="image"),
    EnsureTyped(keys=["image", "seg"]),
])

test_loader = DataLoader(
    Dataset(test_files, test_transforms),
    batch_size=1,
    num_workers=tcfg.num_workers,
    pin_memory=True,
    prefetch_factor=tcfg.prefetch_factor,
    persistent_workers=tcfg.persistent_workers
)

# ═══════════════════════════════════════════════════════════════════════════════
# مدل
# ═══════════════════════════════════════════════════════════════════════════════

def get_model():
    from dataclasses import dataclass

    @dataclass
    class ModelCFG:
        root_path: str = "/content/drive/MyDrive/Colab Notebooks/testi"
        num_classes: int = 4
        base_ch: int = 64
        num_heads: int = 8
        dropout: float = 0.20
        attn_dropout: float = 0.15
        proj_dropout: float = 0.15
        ffn_dropout: float = 0.20
        dec_dropout: float = 0.25
        deep_supervision: bool = True
        device: torch.device = tcfg.device

    cfg = ModelCFG()

    model = build_model(
        cfg,
        module="MultiSeg_Model2.ipynb",
        class_name="MultiModalSegNet",
        root=cfg.root_path,
        fresh=False
    ).to(tcfg.device).eval()

    return model

def load_weights(model, path):
    print(f"📥 Loading: {Path(path).name}")
    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    # ✅ اصلاح: استخراج صحیح state_dict از ساختار checkpoint
    if "ema_teacher_state_dict" in ckpt:
        state_dict = ckpt["ema_teacher_state_dict"]
        print("   └─ Loaded from 'ema_teacher_state_dict'")
    elif "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
        print("   └─ Loaded from 'model_state_dict'")
    elif "ema_state" in ckpt:
        state_dict = ckpt["ema_state"]
        print("   └─ Loaded from 'ema_state'")
    elif "model" in ckpt:
        state_dict = ckpt["model"]
        print("   └─ Loaded from 'model'")
    elif "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
        print("   └─ Loaded from 'state_dict'")
    else:
        state_dict = ckpt
        print("   └─ Using checkpoint directly")

    # ✅ اصلاح: استفاده از strict=False برای انعطاف‌پذیری
    missing, unexpected = model.load_state_dict(state_dict, strict=False)

    if missing:
        print(f"   ⚠️  Missing keys: {len(missing)}")
    if unexpected:
        print(f"   ⚠️  Unexpected keys: {len(unexpected)}")

    return model

models = []
for p in EMA_PATHS:
    if not os.path.exists(p):
        print(f"❌ Checkpoint not found: {p}")
        continue
    model = load_weights(get_model(), p)
    model = model.half()  # FP16
    models.append(model)

if not models:
    raise ValueError("❌ No valid checkpoints loaded!")

print(f"\n✅ Loaded {len(models)} models successfully\n")

# ═══════════════════════════════════════════════════════════════════════════════
# Post-Processing
# ═══════════════════════════════════════════════════════════════════════════════

def advanced_postprocessing(pred, min_ratio=0.01):
    pred_np = pred.cpu().numpy()[0, 0]

    for class_id in [1, 2, 4]:
        mask = (pred_np == class_id)
        if mask.sum() == 0:
            continue

        labeled, num_features = connected_components(mask)

        if num_features > 1:
            sizes = [(labeled == i).sum() for i in range(1, num_features + 1)]
            max_size = max(sizes)

            for i, size in enumerate(sizes, 1):
                if size < min_ratio * max_size:
                    pred_np[labeled == i] = 0

    return torch.from_numpy(pred_np).unsqueeze(0).unsqueeze(0).to(pred.device)

# ═══════════════════════════════════════════════════════════════════════════════
# TTA + Inference
# ═══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
@torch.cuda.amp.autocast()
def inference_with_tta(models, img, use_full_tta=True):
    B, C, D, H, W = img.shape
    device = img.device

    flips = [
        [],
        [2], [3], [4],
        [2, 3], [2, 4], [3, 4],
        [2, 3, 4]
    ] if use_full_tta else [[]]

    ens_probs = []
    for model in models:
        model_probs = []
        for flip_axes in flips:
            x = img.clone()
            for ax in flip_axes:
                x = torch.flip(x, dims=[ax])

            logits = sliding_window_inference(
                inputs=x,
                roi_size=tcfg.roi_size,  # ✅ اصلاح: استفاده از roi_size به جای patch_size
                sw_batch_size=tcfg.sw_batch_size,
                predictor=model,
                overlap=tcfg.sw_overlap,  # ✅ اصلاح: استفاده از sw_overlap
                mode="gaussian"
            )

            # ✅ اصلاح: استخراج خروجی اصلی از dict (deep supervision)
            if isinstance(logits, dict):
                logits = logits.get("out", logits.get("main", list(logits.values())[0]))

            probs = torch.softmax(logits, dim=1)

            for ax in reversed(flip_axes):
                probs = torch.flip(probs, dims=[ax])

            model_probs.append(probs)

        ens_probs.append(torch.stack(model_probs).mean(0))

    return torch.stack(ens_probs).mean(0)

# ═══════════════════════════════════════════════════════════════════════════════
# محاسبه متریک‌ها
# ═══════════════════════════════════════════════════════════════════════════════

def compute_brats_regions(pred, label):
    regions_p = torch.zeros((pred.shape[0], 3, *pred.shape[2:]), device=pred.device)
    regions_l = torch.zeros((label.shape[0], 3, *label.shape[2:]), device=label.device)

    regions_p[:, 0] = (pred > 0).squeeze(1).float()
    regions_l[:, 0] = (label > 0).squeeze(1).float()

    regions_p[:, 1] = ((pred == 1) | (pred == 4)).squeeze(1).float()
    regions_l[:, 1] = ((label == 1) | (label == 4)).squeeze(1).float()

    regions_p[:, 2] = (pred == 4).squeeze(1).float()
    regions_l[:, 2] = (label == 4).squeeze(1).float()

    return regions_p, regions_l

from monai.metrics import compute_hausdorff_distance

def compute_metrics_per_case(pred_regions, label_regions, spacing):
    dice_scores = []
    hd95_scores = []

    for i in range(3):
        p = pred_regions[:, i:i+1]
        l = label_regions[:, i:i+1]

        # محاسبه Dice (که درست بود)
        inter = (p * l).sum()
        union = p.sum() + l.sum()
        dice_scores.append(((2. * inter + 1e-5) / (union + 1e-5)).item())

        # اصلاح محاسبه HD95
        if l.sum() > 0: # فقط اگر واقعا توموری در واقعیت وجود داشته باشد
            if p.sum() > 0:
                try:
                    # اضافه کردن پارامتر spacing حیاتی است
                    hd = compute_hausdorff_distance(p, l, percentile=95, spacing=spacing)
                    hd95_scores.append(hd.item())
                except:
                    hd95_scores.append(25.0) # مقدار جریمه استاندارد برای زمانی که مدل پیدا نکرده
            else:
                hd95_scores.append(373.0) # مقدار جریمه حداکثری (فاصله قطر تصویر)
        else:
            hd95_scores.append(np.nan) # اگر کلا توموری نبوده، نادیده بگیر

    return dice_scores, hd95_scores

# ═══════════════════════════════════════════════════════════════════════════════
# حلقه اصلی
# ═══════════════════════════════════════════════════════════════════════════════

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "preds"), exist_ok=True)

results = []
saver = SaveImage(
    output_dir=os.path.join(OUTPUT_DIR, "preds"),
    output_ext=".nii.gz",
    resample=False,
    print_log=False
)

print(f"\n🚀 Starting Evaluation on A100 80GB (HALF DATA)")
print(f"Cases: {len(test_loader)}")
print(f"TTA: 8-flip | Ensemble: {len(models)} models")
print(f"SW Batch Size: {tcfg.sw_batch_size} | Workers: {tcfg.num_workers}")
print(f"ROI Size: {tcfg.roi_size} | Overlap: {tcfg.sw_overlap}")
print(f"Precision: FP16 (Mixed)\n")

for batch in tqdm(test_loader, desc="Evaluating"):
    img = batch["image"].to(tcfg.device)
    label = batch["seg"].to(tcfg.device)
    case_id = batch["case_id"][0]

    probs = inference_with_tta(models, img, tcfg.use_full_tta)
    pred = torch.argmax(probs, dim=1, keepdim=True)
    pred[pred == 3] = 4
    pred = advanced_postprocessing(pred, min_ratio=tcfg.min_component_size)

    pred_regions, label_regions = compute_brats_regions(pred, label)
    dice_scores, hd95_scores = compute_metrics_per_case(pred_regions, label_regions)

    results.append({
        "Case": case_id,
        "Dice_WT": dice_scores[0],
        "Dice_TC": dice_scores[1],
        "Dice_ET": dice_scores[2],
        "HD95_WT": hd95_scores[0],
        "HD95_TC": hd95_scores[1],
        "HD95_ET": hd95_scores[2]
    })

    # ✅ اصلاح: پیدا کردن صحیح metadata key
    meta_key = next((k for k in batch.keys() if 'meta_dict' in k and k != 'image_meta_dict'), None)

    if meta_key:
        meta = {
            "affine": batch[meta_key]["affine"][0],
            "filename_or_obj": f"{case_id}.nii.gz"
        }
    else:
        # Fallback: استفاده از identity affine
        print(f"⚠️  No metadata found for {case_id}, using identity affine")
        meta = {
            "affine": torch.eye(4),
            "filename_or_obj": f"{case_id}.nii.gz"
        }

    saver(pred[0], meta)

    # ✅ اصلاح: پاکسازی cache برای جلوگیری از OOM
    torch.cuda.empty_cache()

# ═══════════════════════════════════════════════════════════════════════════════
# ذخیره نتایج
# ═══════════════════════════════════════════════════════════════════════════════

df = pd.DataFrame(results)

mean_row = {
    "Case": "MEAN",
    "Dice_WT": df["Dice_WT"].mean(),
    "Dice_TC": df["Dice_TC"].mean(),
    "Dice_ET": df["Dice_ET"].mean(),
    "HD95_WT": df["HD95_WT"].mean(),
    "HD95_TC": df["HD95_TC"].mean(),
    "HD95_ET": df["HD95_ET"].mean()
}

df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)

csv_path = os.path.join(OUTPUT_DIR, "final_metrics_half.csv")
df.to_csv(csv_path, index=False)

print("\n" + "="*70)
print("✅ Evaluation Finished (HALF DATA)!")
print("="*70)
print(f"\n📊 Mean Dice Scores:")
print(f"  WT: {mean_row['Dice_WT']:.4f}")
print(f"  TC: {mean_row['Dice_TC']:.4f}")
print(f"  ET: {mean_row['Dice_ET']:.4f}")
print(f"  Mean: {(mean_row['Dice_WT'] + mean_row['Dice_TC'] + mean_row['Dice_ET']) / 3:.4f}")

print(f"\n📊 Mean HD95:")
print(f"  WT: {mean_row['HD95_WT']:.2f} mm")
print(f"  TC: {mean_row['HD95_TC']:.2f} mm")
print(f"  ET: {mean_row['HD95_ET']:.2f} mm")

print(f"\n💾 Results saved:")
print(f"  CSV: {csv_path}")
print(f"  Predictions: {os.path.join(OUTPUT_DIR, 'preds')}")
print("="*70)"""


🔄 Using HALF of validation data: 125 cases


/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


NameError: name 'build_model' is not defined

In [ ]:
🔄 Using HALF of validation data: 125 cases
[model] MultiSeg_Model2.MultiModalSegNet kwargs={'num_classes': 4, 'base_ch': 64, 'num_heads': 8, 'deep_supervision': True, 'dropout': 0.2, 'attn_dropout': 0.15, 'proj_dropout': 0.15, 'ffn_dropout': 0.2, 'dec_dropout': 0.25}
📥 Loading: BEST_MODEL.pt
   └─ Loaded from 'ema_teacher_state_dict'
[model] MultiSeg_Model2.MultiModalSegNet kwargs={'num_classes': 4, 'base_ch': 64, 'num_heads': 8, 'deep_supervision': True, 'dropout': 0.2, 'attn_dropout': 0.15, 'proj_dropout': 0.15, 'ffn_dropout': 0.2, 'dec_dropout': 0.25}
📥 Loading: LAST_CHECKPOINT.pt
/tmp/ipykernel_140227/2394875101.py:215: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast()
   └─ Loaded from 'ema_teacher_state_dict'

✅ Loaded 2 models successfully


🚀 Starting Evaluation on A100 80GB (HALF DATA)
Cases: 125
TTA: 8-flip | Ensemble: 2 models
SW Batch Size: 8 | Workers: 8
ROI Size: (160, 160, 160) | Overlap: 0.7
Precision: FP16 (Mixed)

Evaluating: 100%|██████████| 125/125 [27:25<00:00, 13.16s/it]
======================================================================
✅ Evaluation Finished (HALF DATA)!
======================================================================

📊 Mean Dice Scores:
  WT: 0.9018
  TC: 0.8917
  ET: 0.8199
  Mean: 0.8712

📊 Mean HD95:
  WT: nan mm
  TC: nan mm
  ET: nan mm

💾 Results saved:
  CSV: /content/drive/MyDrive/Colab Notebooks/testi/BraTS_Final_Thesis_Results_Half/final_metrics_half.csv
  Predictions: /content/drive/MyDrive/Colab Notebooks/testi/BraTS_Final_Thesis_Results_Half/preds
======================================================================


SyntaxError: invalid character '🔄' (U+1F504) (4241219582.py, line 1)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# نسخه اصلاح شده برای رفع خطای UnpicklingError در PyTorch 2.6+
# ═══════════════════════════════════════════════════════════════════════════════

import os
import glob
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from scipy.ndimage import label as connected_components

from monai.data import Dataset, DataLoader
from monai.networks.nets import SegResNet
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    NormalizeIntensityd, ConcatItemsd, EnsureTyped
)
from monai.inferers import sliding_window_inference
from monai.transforms import SaveImage
from monai.metrics import compute_hausdorff_distance

# ═══════════════════════════════════════════════════════════════════════════════
# ۱. تنظیمات (Configuration)
# ═══════════════════════════════════════════════════════════════════════════════

DATA_ROOT = "/content/data/BraTS/dataset"
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/testi/BraTS_Final_Thesis_Results_Half"
EMA_PATHS = [
    "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead_v2/BEST_MODEL.pt",
    "/content/drive/MyDrive/Colab Notebooks/testi/final_train_A100_160_SOTA_Lookahead_v2/LAST_CHECKPOINT.pt"
]

class TestCFG:
    roi_size = (160, 160, 160)
    batch_size_dataloader = 1
    sw_batch_size = 64
    sw_overlap = 0.50
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_full_tta = True
    min_component_size = 0.01
    num_workers = 12
    prefetch_factor = 4
    use_half_data = True

tcfg = TestCFG()

# ═══════════════════════════════════════════════════════════════════════════════
# ۲. تعریف مدل و توابع محاسباتی (Architecture & Metrics)
# ═══════════════════════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════════════════════
# ۲. تعریف مدل و توابع محاسباتی (Architecture & Metrics)
# ═══════════════════════════════════════════════════════════════════════════════

# فرضا این تابع build_model در محیط شما وجود دارد یا از نوت‌بوک دیگری لود شده
# اگر نیست، باید کد آن را اینجا کپی کنید.

def get_model():
    # تنظیمات داخلی مدل برای مطابقت با وزن‌های SOTA
    from dataclasses import dataclass
    @dataclass
    class ModelCFG:
        root_path: str = "/content/drive/MyDrive/Colab Notebooks/testi"
        num_classes: int = 4
        base_ch: int = 64
        num_heads: int = 8
        dropout: float = 0.20
        attn_dropout: float = 0.15
        proj_dropout: float = 0.15
        ffn_dropout: float = 0.20
        dec_dropout: float = 0.25
        deep_supervision: bool = True
        device: torch.device = tcfg.device

    m_cfg = ModelCFG()

    # لود کردن مدل اصلی شما
    # نکته: تابع build_model باید قبلاً تعریف شده باشد
    model = build_model(
        m_cfg,
        module="MultiSeg_Model2.ipynb",
        class_name="MultiModalSegNet",
        root=m_cfg.root_path,
        fresh=False
    )
    return model


# ... بقیه توابع (compute_brats_regions, compute_metrics_per_case, ...) همان‌هایی که دارید درست است

def compute_brats_regions(pred, label):
    regions_p = torch.zeros((1, 3, *pred.shape[2:]), device=pred.device)
    regions_l = torch.zeros((1, 3, *label.shape[2:]), device=label.device)
    regions_p[:, 0] = (pred > 0).float(); regions_l[:, 0] = (label > 0).float()
    regions_p[:, 1] = ((pred == 1) | (pred == 4)).float(); regions_l[:, 1] = ((label == 1) | (label == 4)).float()
    regions_p[:, 2] = (pred == 4).float(); regions_l[:, 2] = (label == 4).float()
    return regions_p, regions_l

"""def compute_metrics_per_case(pred_regions, label_regions, spacing):
    dice_scores, hd95_scores = [], []
    for i in range(3):
        p, l = pred_regions[:, i:i+1], label_regions[:, i:i+1]
        inter = (p * l).sum(); union = p.sum() + l.sum()
        dice_scores.append(((2. * inter + 1e-5) / (union + 1e-5)).item())
        if l.sum() > 0:
            if p.sum() > 0:
                try:
                    hd = compute_hausdorff_distance(p, l, percentile=95, spacing=spacing)
                    hd95_scores.append(hd.item())
                except: hd95_scores.append(25.0)
            else: hd95_scores.append(373.0)
        else: hd95_scores.append(np.nan)
    return dice_scores, hd95_scores

def advanced_postprocessing(pred, min_ratio=0.01):
    pred_np = pred.cpu().numpy()[0, 0]
    for class_id in [1, 2, 4]:
        mask = (pred_np == class_id)
        if mask.sum() == 0: continue
        labeled, num = connected_components(mask)
        if num > 1:
            sizes = [(labeled == i).sum() for i in range(1, num + 1)]
            m_size = max(sizes)
            for i, s in enumerate(sizes, 1):
                if s < min_ratio * m_size: pred_np[labeled == i] = 0
    return torch.from_numpy(pred_np).unsqueeze(0).unsqueeze(0).to(pred.device)"""

def compute_metrics_per_case(pred_regions, label_regions, spacing):
    dice_scores, hd95_scores = [], []

    # اطمینان از فرمت درست spacing
    if hasattr(spacing, "tolist"):
        spacing = spacing.tolist()

    for i in range(3):
        p, l = pred_regions[:, i:i+1], label_regions[:, i:i+1]

        # ۱. محاسبه Dice روی GPU (سریع و بدون خطا)
        inter = (p * l).sum()
        union = p.sum() + l.sum()
        dice_scores.append(((2. * inter + 1e-5) / (union + 1e-5)).item())

        # ۲. محاسبه HD95
        if l.sum() > 0:
            if p.sum() > 0:
                try:
                    # ✅ ترفند اصلی: انتقال موقت به CPU برای دور زدن خطای Thrust/C++
                    p_cpu = p.detach().cpu().float()
                    l_cpu = l.detach().cpu().float()

                    hd = compute_hausdorff_distance(y_pred=p_cpu, y=l_cpu, percentile=95, spacing=spacing)
                    hd95_scores.append(hd.item())
                except Exception as e:
                    # اگر باز هم خطا داد (که بعید است)، مقدار ۲۵ بماند
                    hd95_scores.append(25.0)
            else:
                hd95_scores.append(373.0) # جریمه برای زمانی که مدل چیزی پیدا نکرده
        else:
            hd95_scores.append(np.nan) # اگر در واقعیت توموری وجود نداشته باشد

    return dice_scores, hd95_scores
# ═══════════════════════════════════════════════════════════════════════════════
# ۳. لود مدل‌ها (اصلاح شده برای فیکس خطای Unpickling)
# ═══════════════════════════════════════════════════════════════════════════════

models = []
for p in EMA_PATHS:
    if os.path.exists(p):
        print(f"📥 Loading Weights: {Path(p).name}")
        m = get_model()

        # ✅ اصلاح اصلی اینجاست: اضافه کردن weights_only=False
        try:
            ckpt = torch.load(p, map_location="cpu", weights_only=False)
        except TypeError:
            # برای ورژن‌های خیلی قدیمی torch که پارامتر weights_only ندارند
            ckpt = torch.load(p, map_location="cpu")

        state_key = next((k for k in ["ema_teacher_state_dict", "model_state_dict", "state_dict", "model"] if k in ckpt), None)
        m.load_state_dict(ckpt[state_key] if state_key else ckpt, strict=False)
        m = m.half().to(tcfg.device).eval()
        models.append(m)

# ═══════════════════════════════════════════════════════════════════════════════
# ۴. دیتالودر و آماده‌سازی داده (Data Prep)
# ═══════════════════════════════════════════════════════════════════════════════

def get_val_files(root, split=0.8, seed=42, use_half=False):
    import random
    cases = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
    random.Random(seed).shuffle(cases)
    split_idx = int(len(cases) * split)
    val_cases = cases[split_idx:]
    if use_half:
        # انتخاب نیمه دوم داده‌ها
        #mid_idx = len(val_cases) // 2
        mid_idx = 101
        val_cases = val_cases[mid_idx:]
        print(f"🔄 Using SECOND HALF of validation data: {len(val_cases)} cases")

    val_files = []
    for case in val_cases:
        case_dir = os.path.join(root, case)
        f = lambda suffix: glob.glob(os.path.join(case_dir, f"*_{suffix}.nii.gz"))[0]
        val_files.append({"case_id": case, "t1": f("t1"), "t1ce": f("t1ce"), "t2": f("t2"), "flair": f("flair"), "seg": f("seg")})
    return val_files

test_transforms = Compose([
    LoadImaged(keys=["t1", "t1ce", "t2", "flair", "seg"], image_only=False),
    EnsureChannelFirstd(keys=["t1", "t1ce", "t2", "flair", "seg"]),
    Orientationd(keys=["t1", "t1ce", "t2", "flair", "seg"], axcodes="RAS"),
    NormalizeIntensityd(keys=["t1", "t1ce", "t2", "flair"], nonzero=True, channel_wise=True),
    ConcatItemsd(keys=["t1", "t1ce", "t2", "flair"], name="image"),
    EnsureTyped(keys=["image", "seg"]),
])

test_files = get_val_files(DATA_ROOT, use_half=tcfg.use_half_data)
test_loader = DataLoader(Dataset(test_files, test_transforms), batch_size=tcfg.batch_size_dataloader, num_workers=tcfg.num_workers, pin_memory=True)

# ═══════════════════════════════════════════════════════════════════════════════
# ۵. استنتاج و حلقه اصلی (Inference Loop)
# ═══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def inference_with_tta(models, img):
    flips = [[], [2], [3], [4], [2,3], [2,4], [3,4], [2,3,4]] if tcfg.use_full_tta else [[]]
    ens_probs = []
    for model in models:
        model_probs = []
        for flip_axes in flips:
            x = img.clone()
            for ax in flip_axes: x = torch.flip(x, dims=[ax])
            with torch.cuda.amp.autocast():
                logits = sliding_window_inference(x, tcfg.roi_size, tcfg.sw_batch_size, model, overlap=tcfg.sw_overlap, mode="gaussian")
            if isinstance(logits, dict): logits = logits.get("out", list(logits.values())[0])
            probs = torch.softmax(logits, dim=1)
            for ax in reversed(flip_axes): probs = torch.flip(probs, dims=[ax])
            model_probs.append(probs)
        ens_probs.append(torch.stack(model_probs).mean(0))
    return torch.stack(ens_probs).mean(0)

os.makedirs(os.path.join(OUTPUT_DIR, "preds"), exist_ok=True)
results = []
saver = SaveImage(output_dir=os.path.join(OUTPUT_DIR, "preds"), output_ext=".nii.gz", resample=False, print_log=False)

for batch in tqdm(test_loader, desc="🚀 GPU A100 - Parallel Evaluation"):
    img, label, case_ids = batch["image"].to(tcfg.device), batch["seg"].to(tcfg.device), batch["case_id"]
    meta_key = next((k for k in batch.keys() if 'meta_dict' in k), None)

    probs = inference_with_tta(models, img)
    preds = torch.argmax(probs, dim=1, keepdim=True)
    preds[preds == 3] = 4

    for b in range(img.shape[0]):
        case_id = case_ids[b]
        p_s, l_s = preds[b:b+1], label[b:b+1]
        spacing = batch[meta_key]["pixdim"][b, 1:4].cpu().numpy() if meta_key else np.array([1.0, 1.0, 1.0])
        p_s = advanced_postprocessing(p_s, tcfg.min_component_size)
        p_reg, l_reg = compute_brats_regions(p_s, l_s)
        d_sc, h_sc = compute_metrics_per_case(p_reg, l_reg, spacing)

        results.append({
            "Case": case_id, "Dice_WT": d_sc[0], "Dice_TC": d_sc[1], "Dice_ET": d_sc[2],
            "HD95_WT": h_sc[0], "HD95_TC": h_sc[1], "HD95_ET": h_sc[2]
        })

        affine = batch[meta_key]["affine"][b] if meta_key else torch.eye(4)
        saver(p_s[0], {"affine": affine, "filename_or_obj": f"{case_id}.nii.gz"})

# ═══════════════════════════════════════════════════════════════════════════════
# ۶. خروجی نهایی (Final CSV)
# ═══════════════════════════════════════════════════════════════════════════════

df = pd.DataFrame(results)
cols = ["Dice_WT", "Dice_TC", "Dice_ET", "HD95_WT", "HD95_TC", "HD95_ET"]
mean_vals = {c: df[c].mean(skipna=True) for c in cols}
mean_vals["Case"] = "MEAN_TOTAL"
df = pd.concat([df, pd.DataFrame([mean_vals])], ignore_index=True)
df.to_csv(os.path.join(OUTPUT_DIR, "final_sota_metrics.csv"), index=False)

print(f"\n📊 Evaluation Finished Successfully!")

📥 Loading Weights: BEST_MODEL.pt
[model] MultiSeg_Model2.MultiModalSegNet kwargs={'num_classes': 4, 'base_ch': 64, 'num_heads': 8, 'deep_supervision': True, 'dropout': 0.2, 'attn_dropout': 0.15, 'proj_dropout': 0.15, 'ffn_dropout': 0.2, 'dec_dropout': 0.25}
📥 Loading Weights: LAST_CHECKPOINT.pt
[model] MultiSeg_Model2.MultiModalSegNet kwargs={'num_classes': 4, 'base_ch': 64, 'num_heads': 8, 'deep_supervision': True, 'dropout': 0.2, 'attn_dropout': 0.15, 'proj_dropout': 0.15, 'ffn_dropout': 0.2, 'dec_dropout': 0.25}
🔄 Using SECOND HALF of validation data: 150 cases


🚀 GPU A100 - Parallel Evaluation:   0%|          | 0/150 [00:00<?, ?it/s]/tmp/ipykernel_65000/3232230573.py:236: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
🚀 GPU A100 - Parallel Evaluation: 100%|██████████| 150/150 [39:54<00:00, 15.96s/it]


📊 Evaluation Finished Successfully!


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ۶. خروجی نهایی (Final CSV & Display)
# ═══════════════════════════════════════════════════════════════════════════════

df = pd.DataFrame(results)
cols = ["Dice_WT", "Dice_TC", "Dice_ET", "HD95_WT", "HD95_TC", "HD95_ET"]

# محاسبه میانگین
mean_vals = {c: df[c].mean(skipna=True) for c in cols}
mean_vals["Case"] = "MEAN_TOTAL"

# اضافه کردن ردیف میانگین به انتهای جدول
df_final = pd.concat([df, pd.DataFrame([mean_vals])], ignore_index=True)

# ذخیره در فایل
csv_path = os.path.join(OUTPUT_DIR, "final_sota_metrics.csv")
df_final.to_csv(csv_path, index=False)

print(f"\n✅ Evaluation Finished Successfully!")
print(f"📂 Results saved to: {csv_path}")

print("\n" + "="*50)
print(f"📊 FINAL MEAN METRICS (Total Cases: {len(df)})")
print("="*50)
print(f"Dice Whole Tumor (WT): {mean_vals['Dice_WT']:.4f}")
print(f"Dice Tumor Core   (TC): {mean_vals['Dice_TC']:.4f}")
print(f"Dice Enhancing    (ET): {mean_vals['Dice_ET']:.4f}")
print("-" * 30)
print(f"HD95 Whole Tumor (WT): {mean_vals['HD95_WT']:.2f} mm")
print(f"HD95 Tumor Core   (TC): {mean_vals['HD95_TC']:.2f} mm")
print(f"HD95 Enhancing    (ET): {mean_vals['HD95_ET']:.2f} mm")
print("="*50)


✅ Evaluation Finished Successfully!
📂 Results saved to: /content/drive/MyDrive/Colab Notebooks/testi/BraTS_Final_Thesis_Results_Half/final_sota_metrics.csv

📊 FINAL MEAN METRICS (Total Cases: 150)
Dice Whole Tumor (WT): 0.9257
Dice Tumor Core   (TC): 0.9035
Dice Enhancing    (ET): 0.8199
------------------------------
HD95 Whole Tumor (WT): 5.06 mm
HD95 Tumor Core   (TC): 4.98 mm
HD95 Enhancing    (ET): 5.22 mm


In [ ]:
""""

✅ Evaluation Finished Successfully!
📂 Results saved to: /content/drive/MyDrive/Colab Notebooks/testi/BraTS_Final_Thesis_Results_Half/final_sota_metrics.csv

==================================================
📊 FINAL MEAN METRICS (Total Cases: 150)
==================================================
Dice Whole Tumor (WT): 0.9257
Dice Tumor Core   (TC): 0.9035
Dice Enhancing    (ET): 0.8199
------------------------------
HD95 Whole Tumor (WT): 5.06 mm
HD95 Tumor Core   (TC): 4.98 mm
HD95 Enhancing    (ET): 5.22 mm
=================================================="""

In [ ]:
######################## test

"""
═══════════════════════════════════════════════════════════════════════════════
BraTS 2021 - Final Evaluation Script for Thesis
با متریک‌های استاندارد: Dice, HD95, Sensitivity, Specificity
═══════════════════════════════════════════════════════════════════════════════


import os
import random
import glob
from pathlib import Path
import zipfile
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import numpy as np
from monai.data import Dataset, DataLoader, decollate_batch
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    NormalizeIntensityd, ConcatItemsd, EnsureTyped, AsDiscrete
)
from monai.inferers import sliding_window_inference
from monai.transforms import SaveImage
from monai.networks.layers import Norm
from monai.networks.nets import UNETR, SwinUNETR
from monai.transforms import KeepLargestConnectedComponent
from monai.metrics import DiceMetric, HausdorffDistanceMetric, ConfusionMatrixMetric

# ═══════════════════════════════════════════════════════════════════════════════
# PART 1: DATA PREPARATION
# ═══════════════════════════════════════════════════════════════════════════════

DATA_ROOT = "/content/temp_braTS_data"
ZIP_PATH = "/content/drive/MyDrive/BraTS2021_Training_Data.zip"

# Extract data if needed
if not os.path.exists(DATA_ROOT) or len(os.listdir(DATA_ROOT)) == 0:
    print(f"📦 Extracting {ZIP_PATH}...")
    os.makedirs(DATA_ROOT, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATA_ROOT)
    print("✅ Extraction complete.")

def get_val_split_files(root, split=0.8, seed=42):
    """Get validation split (MUST match train.py seed!)"""
    cases = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
    random.Random(seed).shuffle(cases)

    split_idx = int(len(cases) * split)
    val_cases = cases[split_idx:]

    print(f"📊 Total: {len(cases)} | Validation: {len(val_cases)} cases")

    val_files = []
    for case in val_cases:
        case_dir = os.path.join(root, case)
        t1 = glob.glob(os.path.join(case_dir, "*_t1.nii.gz"))
        t1ce = glob.glob(os.path.join(case_dir, "*_t1ce.nii.gz"))
        t2 = glob.glob(os.path.join(case_dir, "*_t2.nii.gz"))
        flair = glob.glob(os.path.join(case_dir, "*_flair.nii.gz"))
        seg = glob.glob(os.path.join(case_dir, "*_seg.nii.gz"))

        if all([t1, t1ce, t2, flair, seg]):
            val_files.append({
                "case_id": case,
                "t1": t1[0], "t1ce": t1ce[0],
                "t2": t2[0], "flair": flair[0],
                "seg": seg[0]
            })

    return val_files

test_files = get_val_split_files(DATA_ROOT, split=0.8, seed=42)

# Transforms
test_transforms = Compose([
    LoadImaged(keys=["t1", "t1ce", "t2", "flair", "seg"]),
    EnsureChannelFirstd(keys=["t1", "t1ce", "t2", "flair", "seg"]),
    Orientationd(keys=["t1", "t1ce", "t2", "flair", "seg"], axcodes="RAS"),
    NormalizeIntensityd(keys=["t1", "t1ce", "t2", "flair"], nonzero=True, channel_wise=True),
    ConcatItemsd(keys=["t1", "t1ce", "t2", "flair"], name="image"),
    EnsureTyped(keys=["image", "seg"]),
])

test_dataset = Dataset(data=test_files, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4, pin_memory=True)

# ═══════════════════════════════════════════════════════════════════════════════
# PART 2: MODEL & INFERENCE SETUP
# ═══════════════════════════════════════════════════════════════════════════════

class TestCFG:
    roi_size = (160, 160, 160)
    sw_batch_size = 4
    sw_overlap = 0.60
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tcfg = TestCFG()

# ⚠️ REPLACE WITH YOUR MODEL
class ModelClass(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = SwinUNETR(
            img_size=(160, 160, 160),
            in_channels=4,
            out_channels=4,
            feature_size=48,
            use_checkpoint=True,
        )

    def forward(self, x):
        return self.model(x)

def load_ema_model(model_class, ckpt_path):
    """Load EMA checkpoint"""
    print(f"📥 Loading: {Path(ckpt_path).name}")
    model = model_class()
    ckpt = torch.load(ckpt_path, map_location="cpu")

    if isinstance(ckpt, dict):
        state_dict = ckpt.get("ema_state") or ckpt.get("model") or ckpt.get("state_dict") or ckpt
    else:
        state_dict = ckpt

    model.load_state_dict(state_dict, strict=True)
    model = model.to(tcfg.device).eval()
    return model

# Load models
ema_paths = [
    "/content/drive/MyDrive/checkpoints/ema_model_fold0.pth",
    "/content/drive/MyDrive/checkpoints/ema_model_fold1.pth",
    "/content/drive/MyDrive/checkpoints/ema_model_fold2.pth",
]

models = [load_ema_model(ModelClass, p) for p in ema_paths]
print(f"✅ Loaded {len(models)} models\n")

# Post-processing
klcc = KeepLargestConnectedComponent(applied_labels=[1, 2, 3])
post_pred = AsDiscrete(argmax=False, to_onehot=4)  # For metrics
post_label = AsDiscrete(to_onehot=4)

# ═══════════════════════════════════════════════════════════════════════════════
# PART 3: METRICS SETUP
# ═══════════════════════════════════════════════════════════════════════════════

# BraTS regions:
# - WT (Whole Tumor) = label 1 + 2 + 4
# - TC (Tumor Core) = label 1 + 4
# - ET (Enhancing Tumor) = label 4

dice_metric = DiceMetric(include_background=False, reduction="mean_batch")
hd95_metric = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean_batch")
confusion_metric = ConfusionMatrixMetric(include_background=False, metric_name=["sensitivity", "specificity"], reduction="mean_batch")

def compute_brats_regions(pred, label):
    """
    Convert class predictions to BraTS regions.
    Input: (B, H, W, D) with values {0, 1, 2, 4}
    Output: (B, 3, H, W, D) for [WT, TC, ET]
    """
    batch_size = pred.shape[0]
    regions_pred = torch.zeros((batch_size, 3, *pred.shape[1:]), device=pred.device)
    regions_label = torch.zeros((batch_size, 3, *label.shape[1:]), device=label.device)

    # WT = any tumor (1, 2, 4)
    regions_pred[:, 0] = (pred > 0).float()
    regions_label[:, 0] = (label > 0).float()

    # TC = label 1 or 4
    regions_pred[:, 1] = ((pred == 1) | (pred == 4)).float()
    regions_label[:, 1] = ((label == 1) | (label == 4)).float()

    # ET = label 4 only
    regions_pred[:, 2] = (pred == 4).float()
    regions_label[:, 2] = (label == 4).float()

    return regions_pred, regions_label

# ═══════════════════════════════════════════════════════════════════════════════
# PART 4: TTA & INFERENCE
# ═══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def tta_ensemble_predict(models, img):
    """TTA + Ensemble"""
    flips = [[], [2], [3], [4], [2, 3]]
    all_probs = []

    for flip_dims in flips:
        img_aug = torch.flip(img, dims=flip_dims) if flip_dims else img

        model_preds = []
        for model in models:
            pred = sliding_window_inference(
                inputs=img_aug,
                roi_size=tcfg.roi_size,
                sw_batch_size=tcfg.sw_batch_size,
                predictor=model,
                overlap=tcfg.sw_overlap,
                mode="gaussian"
            )
            pred = torch.softmax(pred, dim=1)
            model_preds.append(pred)

        ensemble_pred = torch.stack(model_preds).mean(0)

        if flip_dims:
            ensemble_pred = torch.flip(ensemble_pred, dims=flip_dims)

        all_probs.append(ensemble_pred.cpu())

    all_probs = torch.stack(all_probs)
    mean_probs = all_probs.mean(0).squeeze(0)
    variance_map = all_probs.var(0).squeeze(0).mean(0)
    confidence_map = mean_probs.max(0)[0]

    return mean_probs, confidence_map, variance_map

# ═══════════════════════════════════════════════════════════════════════════════
# PART 5: MAIN EVALUATION LOOP
# ═══════════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = "/content/drive/MyDrive/BraTS_Thesis_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Savers
for subdir in ["seg", "confidence", "uncertainty"]:
    os.makedirs(os.path.join(OUTPUT_DIR, subdir), exist_ok=True)

savers = {
    "seg": SaveImage(output_dir=os.path.join(OUTPUT_DIR, "seg"), output_postfix="", output_ext=".nii.gz", resample=False, separate_folder=False, print_log=False),
    "conf": SaveImage(output_dir=os.path.join(OUTPUT_DIR, "confidence"), output_postfix="", output_ext=".nii.gz", resample=False, separate_folder=False, print_log=False),
    "unc": SaveImage(output_dir=os.path.join(OUTPUT_DIR, "uncertainty"), output_postfix="", output_ext=".nii.gz", resample=False, separate_folder=False, print_log=False),
}

@torch.no_grad()
def run_evaluation():
    """Main evaluation with metrics"""
    print(f"\n{'='*80}")
    print(f"🎓 Starting Thesis Evaluation on {len(test_loader)} cases")
    print(f"{'='*80}\n")

    case_results = []

    for batch in tqdm(test_loader, desc="Evaluating"):
        img = batch["image"].to(tcfg.device)
        label = batch["seg"].to(tcfg.device)
        case_id = batch["case_id"][0]

        # Metadata
        meta = {
            "affine": batch["t1_meta_dict"]["affine"][0].cpu().numpy(),
            "original_affine": batch["t1_meta_dict"]["original_affine"][0].cpu().numpy(),
            "spatial_shape": batch["t1_meta_dict"]["spatial_shape"][0].cpu().numpy(),
            "filename_or_obj": f"{case_id}.nii.gz"
        }

        # Predict
        mean_probs, conf_map, var_map = tta_ensemble_predict(models, img)

        # Argmax
        pred = torch.argmax(mean_probs.to(tcfg.device), dim=0, keepdim=True)

        # Post-process
        pred = klcc(pred.unsqueeze(0)).squeeze(0)  # (1, H, W, D)
        pred[pred == 3] = 4  # Fix ET label

        # Save outputs
        pred_cpu = pred.cpu().unsqueeze(1)
        conf_cpu = conf_map.unsqueeze(0).unsqueeze(0)
        unc_cpu = var_map.unsqueeze(0).unsqueeze(0)

        savers["seg"](pred_cpu, meta_data=meta)
        savers["conf"](conf_cpu, meta_data=meta)
        savers["unc"](unc_cpu, meta_data=meta)

        # ─────────────────────────────────────────────────────────────────────
        # Compute Metrics
        # ─────────────────────────────────────────────────────────────────────

        # Convert to BraTS regions
        pred_regions, label_regions = compute_brats_regions(pred, label)

        # Dice
        dice_metric(y_pred=pred_regions, y=label_regions)
        dice_scores = dice_metric.aggregate().cpu().numpy()  # [WT, TC, ET]

        # HD95
        hd95_metric(y_pred=pred_regions, y=label_regions)
        hd95_scores = hd95_metric.aggregate().cpu().numpy()  # [WT, TC, ET]

        # Sensitivity & Specificity
        confusion_metric(y_pred=pred_regions, y=label_regions)
        sens_spec = confusion_metric.aggregate()  # (2, 3) -> [sensitivity, specificity] × [WT, TC, ET]
        sensitivity = sens_spec[0].cpu().numpy()
        specificity = sens_spec[1].cpu().numpy()

        # Reset metrics for next case
        dice_metric.reset()
        hd95_metric.reset()
        confusion_metric.reset()

        # Store results
        case_results.append({
            "Case": case_id,
            "Dice_WT": dice_scores[0],
            "Dice_TC": dice_scores[1],
            "Dice_ET": dice_scores[2],
            "HD95_WT": hd95_scores[0],
            "HD95_TC": hd95_scores[1],
            "HD95_ET": hd95_scores[2],
            "Sens_WT": sensitivity[0],
            "Sens_TC": sensitivity[1],
            "Sens_ET": sensitivity[2],
            "Spec_WT": specificity[0],
            "Spec_TC": specificity[1],
            "Spec_ET": specificity[2],
        })

    # ─────────────────────────────────────────────────────────────────────────
    # Save Results
    # ─────────────────────────────────────────────────────────────────────────
    df = pd.DataFrame(case_results)

    # Add mean row
    mean_row = df.select_dtypes(include=[np.number]).mean().to_dict()
    mean_row["Case"] = "MEAN"
    df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)

    # Add std row
    std_row = df[:-1].select_dtypes(include=[np.number]).std().to_dict()
    std_row["Case"] = "STD"
    df = pd.concat([df, pd.DataFrame([std_row])], ignore_index=True)

    # Save CSV
    csv_path = os.path.join(OUTPUT_DIR, "evaluation_results.csv")
    df.to_csv(csv_path, index=False, float_format="%.4f")

    # ─────────────────────────────────────────────────────────────────────────
    # Print Summary
    # ─────────────────────────────────────────────────────────────────────────
    print(f"\n{'='*80}")
    print(f"📊 EVALUATION RESULTS (Mean ± Std)")
    print(f"{'='*80}")
    print(f"\n{'Metric':<15} {'WT':<20} {'TC':<20} {'ET':<20}")
    print(f"{'-'*80}")

    mean_vals = df[df["Case"] == "MEAN"].iloc[0]
    std_vals = df[df["Case"] == "STD"].iloc[0]

    for metric in ["Dice", "HD95", "Sens", "Spec"]:
        wt = f"{mean_vals[f'{metric}_WT']:.4f} ± {std_vals[f'{metric}_WT']:.4f}"
        tc = f"{mean_vals[f'{metric}_TC']:.4f} ± {std_vals[f'{metric}_TC']:.4f}"
        et = f"{mean_vals[f'{metric}_ET']:.4f} ± {std_vals[f'{metric}_ET']:.4f}"
        print(f"{metric:<15} {wt:<20} {tc:<20} {et:<20}")

    print(f"\n{'='*80}")
    print(f"✅ Results saved to: {csv_path}")
    print(f"📁 Predictions saved to: {OUTPUT_DIR}")
    print(f"{'='*80}\n")

# ═══════════════════════════════════════════════════════════════════════════════
# RUN
# ═══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    run_evaluation()
"""

In [ ]:
'''
این نقشه شامل ۳ فاز اصلی است که تضمین می‌کند مدل نه تنها یاد بگیرد، بلکه به اصطلاح «تعمیم‌پذیر» (Generalizable) شود.
فاز ۱: نهایی‌سازی روی ۲۰ عکس (تست سلامت)

قبل از سوئیچ به ۱۰۰۰ عکس، باید مطمئن شویم «موتور» مدل سالم است.

    هدف: رسیدن به Dice میانگین بالای ۰.۶ روی همین ۲۰ عکس.

    نکته: چون تعداد عکس‌ها کم است، دایس خیلی بالا نمی‌رود، اما اگر به ۰.۶ رسید یعنی مدل آماده است.

    زمان سوئیچ: به محض اینکه منحنی Loss صاف شد (Plateau).

فاز ۲: سوئیچ به ۱۰۰۰ عکس (فاز قدرت)

وقتی دیتای اصلی را لود کردی، باید پارامترهای CFG را به این شکل تنظیم کنی:
پارامتر	مقدار برای ۱۰۰۰ عکس	چرا این تغییر لازم است؟
batch_size	2 to 4	در A100، اگر پچ‌سایز ۱۲۸ است، همین مقدار عالیست.
epochs	300 - 500	دیتای بزرگ نیاز به زمان بیشتری برای «هضم شدن» دارد.
lr (آموزش از صفر)	2e-4	کمی کمتر از الان، تا مدل با دقت بیشتری وزن‌ها را آپدیت کند.
ema_alpha	0.999	در دیتای زیاد، مدل معلم باید «کندتر» و «باتجربه‌تر» باشد.
num_samples	2 to 4	برای اینکه در هر اپوک، تنوع پچ‌های بیشتری از مغز دیده شود.
فاز ۳: عملیات «رکوردزنی» (Refinement)

اینجاست که مدل تو از یک مدل معمولی به یک مدل به‌شدت بهتر تبدیل می‌شود.

    استراتژی LR Decay: وقتی به اپوک ۲۰۰ رسیدی و دایس مثلاً روی ۸۵٪ قفل کرد، دستی یا با Scheduler نرخ یادگیری را به 1e-5 کاهش بده. این کار باعث می‌شود مدل سگمنتیشن را در لبه‌های ظریف تومور (که HD95 را می‌سازند) اصلاح کند.

    استفاده از نسخه EMA: همیشه برای گزارش نهایی از BEST_MODEL_EMA.pt استفاده کن. این مدل در مواجهه با بیمارانی که تومورهای نامتعارف دارند، بسیار هوشمندتر عمل می‌کند.

    Hard Example Mining: در اپوک‌های آخر، اگر دیدی مدل هنوز در ET مشکل دارد، وزن ET را در Loss از ۵ به ۷ افزایش بده تا مدل در آخرین تلاش‌ها، تمرکزش را فقط روی سخت‌ترین بخش بگذارد.

چک‌لیست نهایی قبل از شروع ۱۰۰۰ عکس:

    [ ] مطمئن شو که فضای Google Drive حداقل ۱۰ تا ۲۰ گیگابایت خالی دارد (برای ذخیره پیاپی چک‌پوینت‌ها).

    [ ] چک کن که accum_iter را بر اساس حافظه تنظیم کرده باشی (اگر Batch Size را ۱ گذاشتی، accum_iter را روی ۴ بگذار تا Batch معادل ۴ شود).

    [ ] دیتای ۱۰۰۰ تایی را حتماً به صورت Pre-processed (یعنی از قبل ری‌سایز یا کراپ شده) دم دست داشته باش تا سرعت لود بالا برود.

نکته آخر: مدلی که الان داری (با Deep Supervision و EMA) پتانسیل این را دارد که در دیتای ۱۰۰۰ تایی، به دایس بالای ۹۰٪ برسد، مشروط بر اینکه در فاز آخر آموزش (اپوک ۳۰۰ به بعد) با نرخ یادگیری بسیار پایین (Fine-tuning) به آن فرصت بدهی مرزها را صیقل بزند.

آیا می‌خواهی کد مربوط به TTA (Test Time Augmentation) را هم آماده داشته باشیم تا بعد از پایان آموزش، برای گرفتن اسکور نهایی از آن استفاده کنی؟ (این همان فوت کوزه‌گری برای چند درصد دایس بالاتر است).
'''

In [ ]:

'''

# =========================================================
# 9️⃣ تعریف Transform و DataLoader (مثال)
# =========================================================
train_transforms = Compose([
    LoadImaged(keys=["image","label"]),
    EnsureChannelFirstd(keys=["image","label"]),
    Orientationd(keys=["image","label"], axcodes="RAS"),
    Spacingd(keys=["image","label"], pixdim=(1.0,1.0,1.0), mode=("bilinear","nearest")),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    CropForegroundd(keys=["image","label"], source_key="image"),
    RandCropByPosNegLabeld(
        keys=["image","label"], label_key="label", spatial_size=cfg.roi_size,
        pos=2, neg=1, num_samples=1, image_key="image", image_threshold=0
    ),
    RandBiasFieldd(keys=["image"], prob=0.3, coeff_range=(0.2,0.3)),
    RandGaussianNoised(keys=["image"], prob=0.15, std=0.01),
    RandGaussianSmoothd(keys=["image"], prob=0.2, sigma_x=(0.5,1.0), sigma_y=(0.5,1.0), sigma_z=(0.5,1.0)),
    RandScaleIntensityd(keys=["image"], factors=0.1, prob=0.5),
    RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.5),
    RandFlipd(keys=["image","label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image","label"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image","label"], prob=0.5, spatial_axis=2),
    RandRotate90d(keys=["image","label"], prob=0.5, max_k=3),
    EnsureTyped(keys=["image","label"])
])
train_ds = CacheDataset(data=train_files, transform=train_transforms, cache_rate=0.0, num_workers=cfg.num_workers)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)

# =========================================================
# 10️⃣ اجرای نهایی
# =========================================================
model = model1.to(cfg.device)
run_training(model, train_loader, val_loader)

'''


In [ ]:
# test the model after training

In [ ]:
'''
import os
import glob
import random
from monai.data import Dataset, DataLoader
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    NormalizeIntensityd, ConcatItemsd, EnsureTyped
)

# =========================================================
# 1. مسیر دیتاست (همان جایی که دیتا الآن هست)
# =========================================================
DATA_ROOT = "/content/temp_braTS_data"

# اگر پوشه خالی است، آنزیپ کن (فقط برای اطمینان)
if not os.path.exists(DATA_ROOT) or not os.listdir(DATA_ROOT):
    print("📦 Unzipping data...")
    os.makedirs(DATA_ROOT, exist_ok=True)
    ZIP_PATH = "/content/drive/MyDrive/BraTS2021_Training_Data.zip"
    !unzip -q "$ZIP_PATH" -d "$DATA_ROOT"

# =========================================================
# 2. جدا کردن ۲۰٪ آخر (Validation Set)
# =========================================================
def get_val_split_files(root, split=0.8, seed=42):
    # لیست تمام کیس‌ها
    cases = sorted(os.listdir(root))

    # ⚠️ بسیار مهم: دقیقاً همان Seed که در آموزش استفاده کردید
    random.Random(seed).shuffle(cases)

    # نقطه برش
    split_idx = int(len(cases) * split)

    # انتخاب ۲۰٪ آخر لیست (داده‌هایی که مدل ندیده)
    val_cases = cases[split_idx:]

    print(f"📊 Total Cases: {len(cases)}")
    print(f"✂️ Split Point: {split_idx}")
    print(f"🧪 Test Set (Validation): {len(val_cases)} cases")

    data = []
    for c in val_cases:
        case_dir = os.path.join(root, c)
        t1 = glob.glob(os.path.join(case_dir, "*t1.nii.gz"))
        t1ce = glob.glob(os.path.join(case_dir, "*t1ce.nii.gz"))
        t2 = glob.glob(os.path.join(case_dir, "*t2.nii.gz"))
        flair = glob.glob(os.path.join(case_dir, "*flair.nii.gz"))

        if t1 and t1ce and t2 and flair:
            data.append({
                "t1": t1[0], "t1ce": t1ce[0], "t2": t2[0], "flair": flair[0],
                "case_id": c,
                # مسیر لیبل برای محاسبه دایس احتمالی در آینده (اختیاری)
                "seg": glob.glob(os.path.join(case_dir, "*seg.nii.gz"))[0]
            })
    return data

test_files = get_val_split_files(DATA_ROOT, split=0.8, seed=42)

# =========================================================
# 3. دیتالودر تست (بدون Spacingd ❌)
# =========================================================
test_transforms = Compose([
    LoadImaged(keys=["t1", "t1ce", "t2", "flair"]),
    EnsureChannelFirstd(keys=["t1", "t1ce", "t2", "flair"]),
    Orientationd(keys=["t1", "t1ce", "t2", "flair"], axcodes="RAS"),
    # ❌ Spacingd حذف شد تا سایز اصلی حفظ شود
    NormalizeIntensityd(keys=["t1", "t1ce", "t2", "flair"], nonzero=True, channel_wise=True),
    ConcatItemsd(keys=["t1", "t1ce", "t2", "flair"], name="image"),
    EnsureTyped(keys=["image"])
])

test_ds = Dataset(data=test_files, transform=test_transforms)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=4)

print("✅ Test Loader Ready (Using Validation Split).")
'''

In [ ]:
'''
# =========================================================
# Part 1: Data Preparation & Loader 🛠️
# =========================================================
import os
import glob
import shutil
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    NormalizeIntensityd, ConcatItemsd, EnsureTyped
)
from monai.data import Dataset, DataLoader

# 1. تنظیم مسیرها (اگر دیتا روی درایو است، آنزیپ می‌کنیم تا سرعت بالا برود)
ZIP_PATH = "/content/drive/MyDrive/BraTS2023_TestingData.zip" # مسیر فایل زیپ شما
TEST_ROOT = "/content/temp_braTS_test"                        # مسیر موقت پرسرعت

if not os.path.exists(TEST_ROOT):
    print(f"⏳ Unzipping data to {TEST_ROOT}...")
    os.makedirs(TEST_ROOT, exist_ok=True)
    # اگر فایل زیپ دارید خط زیر را از کامنت خارج کنید:
    # !unzip -q "$ZIP_PATH" -d "$TEST_ROOT"
    # اگر فایل‌ها همین الان در دسترس هستند، مسیر TEST_ROOT را به مسیر فعلی تغییر دهید.
    print("✅ Unzip complete.")
else:
    print("✅ Data directory exists.")

# 2. ساخت لیست فایل‌ها
def make_test_list(root):
    cases = sorted(os.listdir(root))
    data = []
    for c in cases:
        case_dir = os.path.join(root, c)
        # استفاده از glob برای پیدا کردن فایل‌ها با هر پسوندی
        t1 = glob.glob(os.path.join(case_dir, "*t1.nii.gz"))
        t1ce = glob.glob(os.path.join(case_dir, "*t1ce.nii.gz"))
        t2 = glob.glob(os.path.join(case_dir, "*t2.nii.gz"))
        flair = glob.glob(os.path.join(case_dir, "*flair.nii.gz"))

        if t1 and t1ce and t2 and flair:
            data.append({
                "t1": t1[0], "t1ce": t1ce[0], "t2": t2[0], "flair": flair[0],
                "case_id": c
            })
    return data

test_files = make_test_list(TEST_ROOT)
print(f"🧪 Number of test cases: {len(test_files)}")

# 3. ترنسفرم‌های تست (Spacingd حذف شد ❌)
# نکته: Spacingd حذف شد تا سایز خروجی دقیقاً مثل ورودی باشد (استاندارد سابمیشن)
test_transforms = Compose([
    LoadImaged(keys=["t1", "t1ce", "t2", "flair"]),
    EnsureChannelFirstd(keys=["t1", "t1ce", "t2", "flair"]),
    Orientationd(keys=["t1", "t1ce", "t2", "flair"], axcodes="RAS"),
    # نرمال‌سازی (دقیقاً مثل آموزش)
    NormalizeIntensityd(keys=["t1", "t1ce", "t2", "flair"], nonzero=True, channel_wise=True),
    ConcatItemsd(keys=["t1", "t1ce", "t2", "flair"], name="image"),
    EnsureTyped(keys=["image"])
])

# 4. ساخت لودر
test_ds = Dataset(data=test_files, transform=test_transforms)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=4)

print("✅ Part 1 Done: Test Loader is Ready.")
'''

In [ ]:
'''
# =========================================================
# Part 2: Inference, TTA & Saving 🚀
# =========================================================
import torch
import numpy as np
import os
from monai.inferers import sliding_window_inference
from monai.transforms import KeepLargestConnectedComponent, SaveImage

# 1. تنظیمات
class TestCFG:
    roi_size = (160, 160, 160)
    sw_batch_size = 4     # 🚀 سرعت بالا روی A100
    sw_overlap = 0.60     # همپوشانی استاندارد
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tcfg = TestCFG()

# 2. تابع لود مدل
def load_ema_model(model_class, ckpt_path):
    print(f"🔄 Loading model: {os.path.basename(ckpt_path)}")
    model = model_class().to(tcfg.device)
    ckpt = torch.load(ckpt_path, map_location=tcfg.device)

    # هندل کردن ساختارهای مختلف فایل ذخیره شده
    if isinstance(ckpt, dict):
        if "ema_state" in ckpt: state_dict = ckpt["ema_state"]
        elif "model" in ckpt: state_dict = ckpt["model"]
        elif "state_dict" in ckpt: state_dict = ckpt["state_dict"]
        else: state_dict = ckpt
    else:
        state_dict = ckpt

    model.load_state_dict(state_dict)
    model.eval()
    return model

# ⚠️ مسیر چک‌پوینت‌های خود را اینجا وارد کنید
ema_paths = [
    "/content/drive/MyDrive/experiment_final/BEST_EMA_MODEL.pt",
    # "/content/drive/MyDrive/experiment_final/BEST_EMA_MODEL_fold2.pt",
]

# ⚠️ مهم: به جای YourModelClass نام کلاسی که مدل را با آن ساختید بگذارید (مثلا SegResNet)
# from monai.networks.nets import SegResNet
# models = [load_ema_model(SegResNet, p) for p in ema_paths]
# فرض موقت: مدل model1 هنوز در حافظه است:
models = [load_ema_model(type(model1), p) for p in ema_paths]

# 3. ابزارها (KLCC)
klcc = KeepLargestConnectedComponent(applied_labels=[1,2,3])

# 4. تابع TTA + Ensemble
def tta_ensemble_predict(models, x):
    flips = [[], [2], [3], [4], [2,3]]
    all_probs = []

    for dims in flips:
        x_aug = torch.flip(x, dims) if dims else x

        # Ensemble Loop
        prob_sum = 0
        for model in models:
            logits = sliding_window_inference(
                x_aug, tcfg.roi_size, tcfg.sw_batch_size,
                lambda z: model(z)["logits"],
                overlap=tcfg.sw_overlap,
                mode="gaussian"
            )
            prob_sum += torch.softmax(logits, dim=1)

        prob_mean = prob_sum / len(models)

        # Flip Back
        if dims:
            prob_mean = torch.flip(prob_mean, dims)

        # ✅ انتقال به CPU برای جلوگیری از پر شدن حافظه GPU
        all_probs.append(prob_mean.cpu())

    all_probs = torch.stack(all_probs)

    # میانگین‌گیری و بازگرداندن به GPU برای ادامه محاسبات
    mean_probs = all_probs.mean(0).to(tcfg.device)
    variance_map = all_probs.var(0).mean(1).to(tcfg.device)
    confidence_map = mean_probs.max(1)[0].to(tcfg.device)

    return mean_probs, confidence_map, variance_map

# 5. مسیرهای ذخیره‌سازی
OUTPUT_DIR = "/content/drive/MyDrive/BraTS_Test_Predictions"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# تنظیم Saver ها (بدون پسوند اضافی)
savers = {
    "seg": SaveImage(output_dir=os.path.join(OUTPUT_DIR, "seg"), output_ext=".nii.gz", output_postfix="", separate_folder=False, print_log=False),
    "conf": SaveImage(output_dir=os.path.join(OUTPUT_DIR, "confidence"), output_ext=".nii.gz", output_postfix="", separate_folder=False, print_log=False),
    "unc": SaveImage(output_dir=os.path.join(OUTPUT_DIR, "uncertainty"), output_ext=".nii.gz", output_postfix="", separate_folder=False, print_log=False),
}

# 6. اجرای نهایی
@torch.no_grad()
def run_inference(test_loader):
    print(f"🚀 Starting Inference on {len(test_loader)} cases...")

    for i, batch in enumerate(test_loader):
        img = batch["image"].to(tcfg.device)

        # ✅ اصلاح متادیتا: استفاده از t1_meta_dict برای هدر صحیح
        meta = batch["t1_meta_dict"]
        case_id = batch["case_id"][0]

        # Prediction
        mean_probs, conf_map, var_map = tta_ensemble_predict(models, img)
        pred = torch.argmax(mean_probs, dim=1)

        # Post-Processing
        pred = klcc(pred.unsqueeze(1)).squeeze(1)

        # 🚨 اصلاح لیبل: 3 (ET آموزش) -> 4 (ET استاندارد)
        pred[pred == 3] = 4

        # Save
        savers["seg"](pred, meta_data=meta)
        savers["conf"](conf_map.unsqueeze(1), meta_data=meta)
        savers["unc"](var_map.unsqueeze(1), meta_data=meta)

        print(f"[{i+1}/{len(test_loader)}] Saved: {case_id}")

    print("✅ All outputs saved successfully.")

# اجرا
run_inference(test_loader)
'''